In [ ]:
import os
import time
import math
import numpy as np
import pandas as pd
import requests
import yfinance as yf
from pathlib import Path


# =============================================================================
# Feature Construction — STUDENT FEATURE PANEL CONSTRUCTION
# =============================================================================

BASE_DIR = Path(
    r"<local project path>"
    r"\Bayesian-Macro-Regime-Mapping\Asset allocation with asset specific regime forecasts"
    r"\two_asset_allocation_sjm\ensembles"
)

MKT_PATH = BASE_DIR / "mkt_struct_panel.csv"
CMDTY_PATH = BASE_DIR / "Commodities for the Long Run Index Level Data Monthly.xlsx"
MODEL_PANEL_PATH = BASE_DIR / "model_panel_start_1970_01_31.parquet"

OUT_WITH_CREDIT = BASE_DIR / "mkt_struct_panel_merged.csv"
OUT_WO_CREDIT = BASE_DIR / "mkt_struct_panel_merged_wo_credit.csv"

STUDENT_WITH_CREDIT_PATH = BASE_DIR / "student_ensemble_feature_panel.csv"
STUDENT_WO_CREDIT_PATH = BASE_DIR / "student_ensemble_feature_panel_wo_credit.csv"

FRED_API_KEY = "afb30f46b3f62d2ff330f43bf9d2321f"
BASE_OBS = "https://api.stlouisfed.org/fred/series/observations"

EPS = 1e-12

WINSOR_WINDOW = 120
WINSOR_MIN_PERIODS = 60
WINSOR_Q_LOW = 0.01
WINSOR_Q_HIGH = 0.99

EXPECTED_FINAL_FEATURE_COUNT = 134

Z_WINDOW = 60
Z_MIN_PERIODS = 36
Z_EPS = 1e-12
Z_SMOOTH_HALFLIFE = 12

RUN_GRID_PLOTS = True

PLOT_START_DATE = None
PLOT_END_DATE = None

N_COLS = 4
N_ROWS = 5
FIGSIZE = (24, 16)

USE_TWIN_AXIS = True

RAW_COLOR = "black"
Z_COLOR = "tab:blue"
SMOOTH_COLOR = "tab:red"

RAW_LINEWIDTH = 1.0
Z_LINEWIDTH = 0.9
SMOOTH_LINEWIDTH = 1.8

RAW_ALPHA = 0.90
Z_ALPHA = 0.75
SMOOTH_ALPHA = 0.95

SHOW_LEGEND = True

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 300)
pd.set_option("display.max_rows", 500)


# =============================================================================
# 1. Input checks
# =============================================================================

for p in [MKT_PATH, CMDTY_PATH, MODEL_PANEL_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required input file: {p}")


# =============================================================================
# 2. Helper functions
# =============================================================================

def force_month_end(x):
    return pd.to_datetime(x, errors="coerce").dt.to_period("M").dt.to_timestamp("M")


def coverage_report(df, cols):
    rows = []

    for c in cols:
        s = pd.to_numeric(df[c], errors="coerce")
        valid = s.notna()

        if not valid.any():
            rows.append({
                "variable": c,
                "n_obs": 0,
                "nonmissing_pct": 0.0,
                "first_valid_date": pd.NaT,
                "last_valid_date": pd.NaT,
                "min": np.nan,
                "median": np.nan,
                "max": np.nan,
            })
            continue

        rows.append({
            "variable": c,
            "n_obs": int(valid.sum()),
            "nonmissing_pct": float(valid.mean()),
            "first_valid_date": df.loc[valid, "month_end"].min().date(),
            "last_valid_date": df.loc[valid, "month_end"].max().date(),
            "min": float(s.min(skipna=True)),
            "median": float(s.median(skipna=True)),
            "max": float(s.max(skipna=True)),
        })

    return (
        pd.DataFrame(rows)
        .sort_values(["first_valid_date", "variable"], na_position="last")
        .reset_index(drop=True)
    )


def fred_obs(series_id, clean_name):
    params = {
        "series_id": series_id,
        "api_key": FRED_API_KEY,
        "file_type": "json",
        "observation_start": "1900-01-01",
    }

    r = requests.get(BASE_OBS, params=params, timeout=30)
    r.raise_for_status()

    out = pd.DataFrame(r.json().get("observations", []))

    if out.empty:
        raise ValueError(f"No FRED observations returned for {series_id}")

    out = out[["date", "value"]].copy()
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out[clean_name] = pd.to_numeric(out["value"].replace(".", pd.NA), errors="coerce")

    return out[["date", clean_name]].dropna(subset=["date"])


def to_month_end_last_available(df, value_col):
    tmp = df.copy()
    tmp["month_end"] = force_month_end(tmp["date"])

    return (
        tmp.sort_values(["month_end", "date"])
           .groupby("month_end", as_index=False)
           .tail(1)
           [["month_end", value_col]]
           .sort_values("month_end")
           .reset_index(drop=True)
    )


def yahoo_download_robust(ticker, start="1970-01-01", max_tries=3):
    errors = []

    for i in range(max_tries):
        try:
            raw = yf.download(
                ticker,
                start=start,
                interval="1d",
                auto_adjust=False,
                progress=False,
                threads=False,
            )

            if raw is not None and not raw.empty:
                return raw

            errors.append(f"download start={start} returned empty, try={i + 1}")
            time.sleep(2)

        except Exception as e:
            errors.append(f"download start={start} failed, try={i + 1}: {repr(e)}")
            time.sleep(2)

    for i in range(max_tries):
        try:
            raw = yf.download(
                ticker,
                period="max",
                interval="1d",
                auto_adjust=False,
                progress=False,
                threads=False,
            )

            if raw is not None and not raw.empty:
                raw = raw.loc[pd.to_datetime(raw.index) >= pd.to_datetime(start)].copy()
                if not raw.empty:
                    return raw

            errors.append(f"download period=max returned empty after start filter, try={i + 1}")
            time.sleep(2)

        except Exception as e:
            errors.append(f"download period=max failed, try={i + 1}: {repr(e)}")
            time.sleep(2)

    try:
        raw = yf.Ticker(ticker).history(
            start=start,
            interval="1d",
            auto_adjust=False,
            actions=False,
        )

        if raw is not None and not raw.empty:
            return raw

        errors.append("Ticker.history returned empty")

    except Exception as e:
        errors.append(f"Ticker.history failed: {repr(e)}")

    raise ValueError(f"No Yahoo data downloaded for {ticker}. Details: {' | '.join(errors)}")


def yahoo_monthly_adjusted_return(ticker, ret_col, start="1970-01-01"):
    raw = yahoo_download_robust(ticker=ticker, start=start)

    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)

    if "Adj Close" not in raw.columns:
        raise ValueError(f"Adjusted close missing for {ticker}; do not use raw Close.")

    d = raw[["Adj Close"]].copy()
    d = d.rename(columns={"Adj Close": "adj_close"})
    d = d.reset_index()

    date_col = "Date" if "Date" in d.columns else d.columns[0]
    d[date_col] = pd.to_datetime(d[date_col], errors="coerce")
    d["month_end"] = force_month_end(d[date_col])

    monthly_level = (
        d.dropna(subset=["month_end", "adj_close"])
         .sort_values(date_col)
         .groupby("month_end", as_index=False)
         .last()[["month_end", "adj_close"]]
         .sort_values("month_end")
         .reset_index(drop=True)
    )

    monthly_level[ret_col] = monthly_level["adj_close"].pct_change()

    return monthly_level[["month_end", ret_col]]


def _row_dispersion(df, cols):
    return df[cols].std(axis=1, ddof=0)


def _row_best_minus_worst(df, cols):
    return df[cols].max(axis=1) - df[cols].min(axis=1)


def _roll_sum(s, w):
    return pd.to_numeric(s, errors="coerce").rolling(w, min_periods=w).sum()


def _roll_mean(s, w):
    return pd.to_numeric(s, errors="coerce").rolling(w, min_periods=w).mean()


def _roll_std(s, w):
    return pd.to_numeric(s, errors="coerce").rolling(w, min_periods=w).std(ddof=0)


def _diff1(s):
    return pd.to_numeric(s, errors="coerce").diff()


def _log_diff1(s):
    s = pd.to_numeric(s, errors="coerce")
    return np.log(s.where(s > 0)).diff()


def _safe_fisher_corr(rho):
    rho = pd.to_numeric(rho, errors="coerce").clip(-0.999, 0.999)
    return 0.5 * np.log((1.0 + rho) / (1.0 - rho))


def _rolling_corr_fisher_diff(df, x_col, y_col, window=12):
    rho = df[x_col].rolling(window, min_periods=window).corr(df[y_col])
    fisher = _safe_fisher_corr(rho)
    return fisher.diff()


def _rolling_pairwise_corr_stats(df, cols, window=12, prefix=None):
    avg_corr = pd.Series(np.nan, index=df.index)
    absavg_corr = pd.Series(np.nan, index=df.index)
    pca1_share = pd.Series(np.nan, index=df.index)
    rotation_speed = pd.Series(np.nan, index=df.index)

    prev_corr = None
    X = df[cols].copy()

    for i in range(len(df)):
        if i + 1 < window:
            continue

        block = X.iloc[i + 1 - window:i + 1].dropna()

        if len(block) < window:
            prev_corr = None
            continue

        corr = block.corr().to_numpy(dtype=float)

        if not np.isfinite(corr).all():
            prev_corr = None
            continue

        n = corr.shape[0]

        if n < 2:
            prev_corr = None
            continue

        upper = corr[np.triu_indices(n, k=1)]

        avg_corr.iloc[i] = np.nanmean(upper)
        absavg_corr.iloc[i] = np.nanmean(np.abs(upper))

        try:
            eigvals = np.linalg.eigvalsh(corr)
            eigvals = np.maximum(eigvals, 0.0)
            denom = eigvals.sum()
            pca1_share.iloc[i] = eigvals[-1] / denom if denom > EPS else np.nan
        except Exception:
            pca1_share.iloc[i] = np.nan

        if prev_corr is not None and prev_corr.shape == corr.shape:
            rotation_speed.iloc[i] = np.linalg.norm(corr - prev_corr, ord="fro")

        prev_corr = corr.copy()

    return {
        f"{prefix}_corr_avg_12m": avg_corr,
        f"{prefix}_corr_absavg_12m": absavg_corr,
        f"{prefix}_pca1_share_12m": pca1_share,
        f"{prefix}_pca1_share_diff": pca1_share.diff(),
        f"{prefix}_rotation_speed_12m": rotation_speed,
    }


def _lambda_f_series(df, cols, window=12, prefix=None, include_corr=True):
    lam_cov = pd.Series(np.nan, index=df.index)
    lam_corr = pd.Series(np.nan, index=df.index)

    X = df[cols].copy()

    prev_cov = None
    prev_corr = None

    for i in range(len(df)):
        if i + 1 < window:
            continue

        block = X.iloc[i + 1 - window:i + 1].dropna()

        if len(block) < window:
            prev_cov = None
            prev_corr = None
            continue

        cov = block.cov().to_numpy(dtype=float)
        corr = block.corr().to_numpy(dtype=float)

        if not np.isfinite(cov).all():
            prev_cov = None
            prev_corr = None
            continue

        if include_corr and not np.isfinite(corr).all():
            prev_cov = None
            prev_corr = None
            continue

        if prev_cov is not None and prev_cov.shape == cov.shape:
            dF = cov - prev_cov
            comm = cov @ dF - dF @ cov
            denom = np.linalg.norm(cov, ord="fro") * np.linalg.norm(dF, ord="fro")

            if denom > EPS:
                lam_cov.iloc[i] = np.log1p(
                    100.0 * np.linalg.norm(comm, ord="fro") / denom
                )

        if include_corr:
            if prev_corr is not None and prev_corr.shape == corr.shape:
                dF = corr - prev_corr
                comm = corr @ dF - dF @ corr
                denom = np.linalg.norm(corr, ord="fro") * np.linalg.norm(dF, ord="fro")

                if denom > EPS:
                    lam_corr.iloc[i] = np.log1p(
                        100.0 * np.linalg.norm(comm, ord="fro") / denom
                    )

        prev_cov = cov.copy()
        prev_corr = corr.copy() if include_corr else None

    out = {
        f"{prefix}_lambda_cov_12m": lam_cov,
        f"{prefix}_lambda_cov_diff": lam_cov.diff(),
    }

    if include_corr:
        out[f"{prefix}_lambda_corr_12m"] = lam_corr
        out[f"{prefix}_lambda_corr_diff"] = lam_corr.diff()

    return out


def _require_cols(df, cols, block_name):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"{block_name} missing columns: {missing}")


def rolling_winsorize_series(
    s,
    window=120,
    min_periods=60,
    q_low=0.01,
    q_high=0.99,
):
    x = pd.to_numeric(s, errors="coerce")

    lo = (
        x.shift(1)
         .rolling(window=window, min_periods=min_periods)
         .quantile(q_low)
    )

    hi = (
        x.shift(1)
         .rolling(window=window, min_periods=min_periods)
         .quantile(q_high)
    )

    out = x.copy()
    valid_band = lo.notna() & hi.notna() & x.notna()

    out.loc[valid_band] = np.minimum(
        np.maximum(x.loc[valid_band], lo.loc[valid_band]),
        hi.loc[valid_band],
    )

    return out


def causal_rolling_zscore(
    s,
    window=60,
    min_periods=36,
    eps=1e-12,
):
    x = pd.to_numeric(s, errors="coerce")

    mu = (
        x.shift(1)
         .rolling(window=window, min_periods=min_periods)
         .mean()
    )

    sig = (
        x.shift(1)
         .rolling(window=window, min_periods=min_periods)
         .std(ddof=0)
    )

    return (x - mu) / (sig + eps)


def add_z_and_smooth(panel, output_date_col="date"):
    out = panel.copy()

    if "month_end" not in out.columns:
        raise KeyError("Expected column month_end not found.")

    out["month_end"] = pd.to_datetime(out["month_end"]) + pd.offsets.MonthEnd(0)

    out = (
        out.sort_values("month_end")
           .drop_duplicates("month_end", keep="last")
           .reset_index(drop=True)
    )

    raw_feature_cols = [c for c in out.columns if c != "month_end"]

    for c in raw_feature_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    z_cols = []

    for c in raw_feature_cols:
        z_col = f"{c}_z"
        out[z_col] = causal_rolling_zscore(
            out[c],
            window=Z_WINDOW,
            min_periods=Z_MIN_PERIODS,
            eps=Z_EPS,
        )
        z_cols.append(z_col)

    smooth_cols = []

    for z_col in z_cols:
        smooth_col = f"{z_col}_smooth"
        out[smooth_col] = (
            out[z_col]
            .ewm(
                halflife=Z_SMOOTH_HALFLIFE,
                adjust=False,
                min_periods=1,
            )
            .mean()
        )
        smooth_cols.append(smooth_col)

    final_cols = ["month_end"] + raw_feature_cols + z_cols + smooth_cols
    out = out[final_cols].copy()

    complete_cols = raw_feature_cols + z_cols + smooth_cols
    complete_mask = out[complete_cols].notna().all(axis=1)

    out = (
        out.loc[complete_mask]
           .rename(columns={"month_end": output_date_col})
           .reset_index(drop=True)
    )

    return out, raw_feature_cols, z_cols, smooth_cols


def student_panel_summary(name, df, raw_cols, z_cols, smooth_cols, path):
    print("=" * 120)
    print(name)
    print("=" * 120)
    print("Saved path:", path)
    print("Shape:", df.shape)
    print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
    print("Raw/base feature columns:", len(raw_cols))
    print("Causal z-score columns:", len(z_cols))
    print("Smoothed diagnostic columns:", len(smooth_cols))
    print("Total columns including date:", df.shape[1])
    print("Complete rows:", int(df.notna().all(axis=1).sum()))

    expected_total = 1 + len(raw_cols) + len(z_cols) + len(smooth_cols)

    if df.shape[1] != expected_total:
        raise ValueError(f"{name}: column count check failed.")

    if len(raw_cols) != len(z_cols) or len(raw_cols) != len(smooth_cols):
        raise ValueError(f"{name}: raw/z/smooth count mismatch.")

    z_missing = [f"{c}_z" for c in raw_cols if f"{c}_z" not in df.columns]
    smooth_missing = [f"{c}_z_smooth" for c in raw_cols if f"{c}_z_smooth" not in df.columns]

    if z_missing:
        raise ValueError(f"{name}: missing z columns: {z_missing[:20]}")

    if smooth_missing:
        raise ValueError(f"{name}: missing smooth columns: {smooth_missing[:20]}")

    print("Column check: passed")
    print()


def plot_raw_z_smooth_grid(
    input_df,
    panel_name,
    plot_start_date=None,
    plot_end_date=None,
    n_cols=4,
    n_rows=5,
    figsize=(24, 16),
    use_twin_axis=True,
):
    plot_df = input_df.copy()

    if "date" not in plot_df.columns:
        raise KeyError(f"{panel_name}: expected column 'date' not found.")

    plot_df["date"] = pd.to_datetime(plot_df["date"]) + pd.offsets.MonthEnd(0)

    plot_df = (
        plot_df.sort_values("date")
               .drop_duplicates("date", keep="last")
               .reset_index(drop=True)
    )

    if plot_start_date is not None:
        plot_df = plot_df.loc[plot_df["date"] >= pd.Timestamp(plot_start_date)].copy()

    if plot_end_date is not None:
        plot_df = plot_df.loc[plot_df["date"] <= pd.Timestamp(plot_end_date)].copy()

    if plot_df.empty:
        raise ValueError(f"{panel_name}: plot_df is empty after date filtering.")

    all_cols = [c for c in plot_df.columns if c != "date"]

    raw_feature_cols = [
        c for c in all_cols
        if not c.endswith("_z")
        and not c.endswith("_z_smooth")
        and f"{c}_z" in plot_df.columns
        and f"{c}_z_smooth" in plot_df.columns
    ]

    if not raw_feature_cols:
        raise ValueError(f"{panel_name}: no raw/z/smooth feature triplets found.")

    plots_per_fig = n_cols * n_rows
    n_figs = math.ceil(len(raw_feature_cols) / plots_per_fig)

    print("\n" + "=" * 120)
    print("GRID PLOTS: RAW + Z + Z_SMOOTH")
    print("=" * 120)
    print("Panel:", panel_name)
    print("Rows:", len(plot_df))
    print("Date range:", plot_df["date"].min().date(), "to", plot_df["date"].max().date())
    print("Base features plotted:", len(raw_feature_cols))
    print("Plots per figure:", plots_per_fig)
    print("Figures:", n_figs)
    print("Twin axis:", use_twin_axis)

    for fig_id in range(n_figs):
        chunk = raw_feature_cols[
            fig_id * plots_per_fig : (fig_id + 1) * plots_per_fig
        ]

        fig, axes = plt.subplots(
            n_rows,
            n_cols,
            figsize=figsize,
            sharex=False,
        )

        axes = np.asarray(axes).reshape(-1)

        for ax, base_col in zip(axes, chunk):
            z_col = f"{base_col}_z"
            smooth_col = f"{base_col}_z_smooth"

            temp = plot_df[["date", base_col, z_col, smooth_col]].copy()

            temp[base_col] = pd.to_numeric(temp[base_col], errors="coerce")
            temp[z_col] = pd.to_numeric(temp[z_col], errors="coerce")
            temp[smooth_col] = pd.to_numeric(temp[smooth_col], errors="coerce")

            raw_temp = temp[["date", base_col]].dropna()
            z_temp = temp[["date", z_col]].dropna()
            smooth_temp = temp[["date", smooth_col]].dropna()

            if raw_temp.empty and z_temp.empty and smooth_temp.empty:
                ax.set_title(f"{base_col}\nno data", fontsize=8.5, fontweight="bold")
                ax.axis("off")
                continue

            if use_twin_axis:
                ax_z = ax.twinx()

                if not raw_temp.empty:
                    ax.plot(
                        raw_temp["date"],
                        raw_temp[base_col],
                        color=RAW_COLOR,
                        lw=RAW_LINEWIDTH,
                        alpha=RAW_ALPHA,
                        label="raw",
                    )

                if not z_temp.empty:
                    ax_z.plot(
                        z_temp["date"],
                        z_temp[z_col],
                        color=Z_COLOR,
                        lw=Z_LINEWIDTH,
                        alpha=Z_ALPHA,
                        label="z",
                    )

                if not smooth_temp.empty:
                    ax_z.plot(
                        smooth_temp["date"],
                        smooth_temp[smooth_col],
                        color=SMOOTH_COLOR,
                        lw=SMOOTH_LINEWIDTH,
                        alpha=SMOOTH_ALPHA,
                        label="z_smooth",
                    )

                ax.axhline(0.0, color="gray", lw=0.7, ls="--", alpha=0.45)
                ax_z.axhline(0.0, color="gray", lw=0.7, ls=":", alpha=0.35)

                ax.tick_params(axis="y", labelsize=7, colors=RAW_COLOR)
                ax_z.tick_params(axis="y", labelsize=7, colors=Z_COLOR)

                ax.set_ylabel("raw", fontsize=7, color=RAW_COLOR)
                ax_z.set_ylabel("z / smooth", fontsize=7, color=Z_COLOR)

                if SHOW_LEGEND:
                    lines_1, labels_1 = ax.get_legend_handles_labels()
                    lines_2, labels_2 = ax_z.get_legend_handles_labels()

                    ax.legend(
                        lines_1 + lines_2,
                        labels_1 + labels_2,
                        fontsize=7,
                        loc="best",
                        frameon=True,
                    )

            else:
                if not raw_temp.empty:
                    ax.plot(
                        raw_temp["date"],
                        raw_temp[base_col],
                        color=RAW_COLOR,
                        lw=RAW_LINEWIDTH,
                        alpha=RAW_ALPHA,
                        label="raw",
                    )

                if not z_temp.empty:
                    ax.plot(
                        z_temp["date"],
                        z_temp[z_col],
                        color=Z_COLOR,
                        lw=Z_LINEWIDTH,
                        alpha=Z_ALPHA,
                        label="z",
                    )

                if not smooth_temp.empty:
                    ax.plot(
                        smooth_temp["date"],
                        smooth_temp[smooth_col],
                        color=SMOOTH_COLOR,
                        lw=SMOOTH_LINEWIDTH,
                        alpha=SMOOTH_ALPHA,
                        label="z_smooth",
                    )

                ax.axhline(0.0, color="gray", lw=0.7, ls="--", alpha=0.45)

                if SHOW_LEGEND:
                    ax.legend(fontsize=7, loc="best", frameon=True)

            valid_any = temp[[base_col, z_col, smooth_col]].notna().any(axis=1)

            if valid_any.any():
                first_valid = temp.loc[valid_any, "date"].min().date()
                last_valid = temp.loc[valid_any, "date"].max().date()
                subtitle = f"{first_valid} to {last_valid}"
            else:
                subtitle = "no data"

            ax.set_title(
                f"{base_col}\n{subtitle}",
                fontsize=8.5,
                fontweight="bold",
            )

            ax.grid(True, alpha=0.25)

        for ax in axes[len(chunk):]:
            ax.axis("off")

        fig.suptitle(
            f"{panel_name} | raw + z + z_smooth | Figure {fig_id + 1}/{n_figs}",
            fontsize=15,
            fontweight="bold",
            y=0.995,
        )

        plt.tight_layout(rect=[0, 0, 1, 0.97])
        plt.show()


# =============================================================================
# 3. Raw input panel
# =============================================================================

mkt = pd.read_csv(MKT_PATH)
mkt["month_end"] = force_month_end(mkt["month_end"])

mkt_cols = [
    "ff_mktrf", "ff_smb", "ff_hml", "ff_rmw", "ff_cma", "ff_mom",
    "smlo_vwret", "smme_vwret", "smhi_vwret",
    "bilo_vwret", "bime_vwret", "bihi_vwret",
    "crsp_ind12_buseq", "crsp_ind12_chems", "crsp_ind12_durbl",
    "crsp_ind12_enrgy", "crsp_ind12_hlth", "crsp_ind12_manuf",
    "crsp_ind12_money", "crsp_ind12_nodur", "crsp_ind12_other",
    "crsp_ind12_shops", "crsp_ind12_telcm", "crsp_ind12_utils",
    "crsp_tsy_1_3y", "crsp_tsy_3_7y", "crsp_tsy_7_10y",
    "crsp_tsy_10_20y", "crsp_tsy_20y_plus",
]

_require_cols(mkt, mkt_cols, "mkt_struct_panel.csv")

mkt = mkt[["month_end"] + mkt_cols].copy()

for c in mkt_cols:
    mkt[c] = pd.to_numeric(mkt[c], errors="coerce")

mkt = (
    mkt.sort_values("month_end")
       .drop_duplicates("month_end", keep="last")
       .reset_index(drop=True)
)

cmdty = pd.read_excel(
    CMDTY_PATH,
    sheet_name="Commodities for the Long Run",
    header=10,
)

cmdty = cmdty.dropna(axis=0, how="all").dropna(axis=1, how="all")
cmdty = cmdty.rename(columns={cmdty.columns[0]: "month_end"})
cmdty = cmdty.rename(columns={
    "Excess return of equal-weight commodities portfolio": "cmdty_broad_aqr_ret",
})

if "cmdty_broad_aqr_ret" not in cmdty.columns:
    raise ValueError("Missing AQR commodity excess return column.")

cmdty["month_end"] = force_month_end(cmdty["month_end"])
cmdty = cmdty[cmdty["month_end"].notna()].copy()
cmdty["cmdty_broad_aqr_ret"] = pd.to_numeric(cmdty["cmdty_broad_aqr_ret"], errors="coerce")

cmdty = (
    cmdty[["month_end", "cmdty_broad_aqr_ret"]]
    .sort_values("month_end")
    .drop_duplicates("month_end", keep="last")
    .reset_index(drop=True)
)

x = cmdty["cmdty_broad_aqr_ret"]

cmdty["cmdty_ret_1m"] = x
cmdty["cmdty_ret_6m"] = x.rolling(6, min_periods=6).sum()
cmdty["cmdty_ret_12m"] = x.rolling(12, min_periods=12).sum()
cmdty["cmdty_vol_12m"] = x.rolling(12, min_periods=12).std() * np.sqrt(12)

cmdty["cmdty_sharpe_12m"] = (
    x.rolling(12, min_periods=12).mean()
    / (x.rolling(12, min_periods=12).std() + EPS)
    * np.sqrt(12)
)

cmdty_cols = [
    "cmdty_broad_aqr_ret",
    "cmdty_ret_1m",
    "cmdty_ret_6m",
    "cmdty_ret_12m",
    "cmdty_vol_12m",
    "cmdty_sharpe_12m",
]

cmdty = cmdty[["month_end"] + cmdty_cols].copy()

raw_aligned = (
    mkt.merge(cmdty, on="month_end", how="outer")
       .sort_values("month_end")
       .reset_index(drop=True)
)

FRED_SERIES = {
    "fred_nfci": "NFCI",
    "fred_anfci": "ANFCI",
}

fred_monthly = None

for clean_name, series_id in FRED_SERIES.items():
    raw = fred_obs(series_id, clean_name)
    monthly = to_month_end_last_available(raw, clean_name)

    if fred_monthly is None:
        fred_monthly = monthly
    else:
        fred_monthly = fred_monthly.merge(monthly, on="month_end", how="outer")

fred_monthly = (
    fred_monthly.sort_values("month_end")
                .drop_duplicates("month_end", keep="last")
                .reset_index(drop=True)
)

CREDIT_TICKERS = {
    "credit_ig_proxy_ret": "VWESX",
    "credit_hy_proxy_ret": "VWEHX",
}

credit_parts = []

for ret_col, ticker in CREDIT_TICKERS.items():
    credit_parts.append(
        yahoo_monthly_adjusted_return(
            ticker=ticker,
            ret_col=ret_col,
            start="1970-01-01",
        )
    )

credit_monthly = credit_parts[0]

for part in credit_parts[1:]:
    credit_monthly = credit_monthly.merge(part, on="month_end", how="outer")

credit_monthly = (
    credit_monthly.sort_values("month_end")
                  .drop_duplicates("month_end", keep="last")
                  .reset_index(drop=True)
)

model_panel = pd.read_parquet(MODEL_PANEL_PATH)

if "date" not in model_panel.columns:
    raise ValueError("Expected date column 'date' not found in model panel.")

model_panel = model_panel.copy()
model_panel["month_end"] = force_month_end(model_panel["date"])

model_raw_cols = [
    "raw_DGS1",
    "derived_slope_10y_2y_wrds",
    "derived_belly_slope",
    "credit_spread_baa_aaa",
    "CORESTICKM159SFRBATL",
    "VIXCLSx",
    "rv_60d",
    "vwretd",
    "sprtrn_sp500",
    "rf",
    "agg_ret",
    "CFNAI",
    "UMCSENTx",
    "bond_10y_ret",
    "bond_10y_mom_12m",
]

_require_cols(model_panel, model_raw_cols, "model_panel_start_1970_01_31.parquet")

model_raw = model_panel[["month_end"] + model_raw_cols].copy()

for c in model_raw_cols:
    model_raw[c] = pd.to_numeric(model_raw[c], errors="coerce")

model_raw = (
    model_raw.sort_values("month_end")
             .drop_duplicates("month_end", keep="last")
             .reset_index(drop=True)
)

external_cols = list(CREDIT_TICKERS.keys()) + list(FRED_SERIES.keys())

raw_aligned_final = (
    raw_aligned
    .drop(columns=[c for c in external_cols + model_raw_cols if c in raw_aligned.columns], errors="ignore")
    .merge(credit_monthly, on="month_end", how="left")
    .merge(fred_monthly, on="month_end", how="left")
    .merge(model_raw, on="month_end", how="left")
    .sort_values("month_end")
    .drop_duplicates("month_end", keep="last")
    .reset_index(drop=True)
)

for c in raw_aligned_final.columns:
    if c != "month_end":
        raw_aligned_final[c] = pd.to_numeric(raw_aligned_final[c], errors="coerce")


# =============================================================================
# 4. Market-structure features
# =============================================================================

mkt_raw = raw_aligned_final.copy()

factor_cols = [
    "ff_mktrf", "ff_smb", "ff_hml", "ff_rmw", "ff_cma", "ff_mom",
]

size_value_cols = [
    "smlo_vwret", "smme_vwret", "smhi_vwret",
    "bilo_vwret", "bime_vwret", "bihi_vwret",
]

industry_cols = sorted([c for c in mkt_raw.columns if c.startswith("crsp_ind12_")])

treasury_cols = [
    "crsp_tsy_1_3y",
    "crsp_tsy_3_7y",
    "crsp_tsy_7_10y",
    "crsp_tsy_10_20y",
    "crsp_tsy_20y_plus",
]

credit_cols = [
    "credit_ig_proxy_ret",
    "credit_hy_proxy_ret",
]

commodity_cols = [
    "cmdty_broad_aqr_ret",
]

financial_conditions_cols = [
    "fred_nfci",
    "fred_anfci",
]

legacy_source_cols = [
    "sprtrn_sp500",
    "agg_ret",
    "rf",
    "raw_DGS1",
    "derived_slope_10y_2y_wrds",
    "derived_belly_slope",
    "credit_spread_baa_aaa",
    "CORESTICKM159SFRBATL",
    "VIXCLSx",
    "rv_60d",
    "vwretd",
    "CFNAI",
    "UMCSENTx",
    "bond_10y_ret",
    "bond_10y_mom_12m",
]

for block_name, cols in [
    ("factor_cols", factor_cols),
    ("size_value_cols", size_value_cols),
    ("industry_cols", industry_cols),
    ("treasury_cols", treasury_cols),
    ("credit_cols", credit_cols),
    ("commodity_cols", commodity_cols),
    ("financial_conditions_cols", financial_conditions_cols),
    ("legacy_source_cols", legacy_source_cols),
]:
    _require_cols(mkt_raw, cols, block_name)

d = {}

d["factor_ret_dispersion_1m"] = _row_dispersion(mkt_raw, factor_cols)
d["factor_ret_dispersion_6m"] = _roll_mean(d["factor_ret_dispersion_1m"], 6)
d["ff_mom_minus_hml"] = mkt_raw["ff_mom"] - mkt_raw["ff_hml"]
d["ff_rmw_minus_cma"] = mkt_raw["ff_rmw"] - mkt_raw["ff_cma"]
d.update(_rolling_pairwise_corr_stats(mkt_raw, factor_cols, window=12, prefix="factor"))
d.update(_lambda_f_series(mkt_raw, factor_cols, window=12, prefix="factor", include_corr=True))

d["small_avg_ret"] = mkt_raw[["smlo_vwret", "smme_vwret", "smhi_vwret"]].mean(axis=1)
d["big_avg_ret"] = mkt_raw[["bilo_vwret", "bime_vwret", "bihi_vwret"]].mean(axis=1)
d["growth_avg_ret"] = mkt_raw[["smlo_vwret", "bilo_vwret"]].mean(axis=1)
d["value_avg_ret"] = mkt_raw[["smhi_vwret", "bihi_vwret"]].mean(axis=1)
d["small_minus_big"] = d["small_avg_ret"] - d["big_avg_ret"]
d["value_minus_growth"] = d["value_avg_ret"] - d["growth_avg_ret"]
d["small_value_minus_big_growth"] = mkt_raw["smhi_vwret"] - mkt_raw["bilo_vwret"]
d["size_value_ret_dispersion_1m"] = _row_dispersion(mkt_raw, size_value_cols)
d["size_value_ret_dispersion_6m"] = _roll_mean(d["size_value_ret_dispersion_1m"], 6)
d.update(_rolling_pairwise_corr_stats(mkt_raw, size_value_cols, window=12, prefix="size_value"))
d.update(_lambda_f_series(mkt_raw, size_value_cols, window=12, prefix="size_value", include_corr=True))

d["industry_ret_dispersion_1m"] = _row_dispersion(mkt_raw, industry_cols)
d["industry_ret_dispersion_6m"] = _roll_mean(d["industry_ret_dispersion_1m"], 6)
d["industry_best_minus_worst_1m"] = _row_best_minus_worst(mkt_raw, industry_cols)
d["industry_best_minus_worst_6m"] = _roll_mean(d["industry_best_minus_worst_1m"], 6)
d.update(_rolling_pairwise_corr_stats(mkt_raw, industry_cols, window=12, prefix="industry"))
d.update(_lambda_f_series(mkt_raw, industry_cols, window=12, prefix="industry", include_corr=True))

d["tsy_short_ret"] = mkt_raw["crsp_tsy_1_3y"]
d["tsy_intermediate_ret"] = mkt_raw[["crsp_tsy_3_7y", "crsp_tsy_7_10y"]].mean(axis=1)
d["tsy_long_ret"] = mkt_raw[["crsp_tsy_10_20y", "crsp_tsy_20y_plus"]].mean(axis=1)
d["tsy_long_minus_short"] = d["tsy_long_ret"] - d["tsy_short_ret"]
d["tsy_20y_minus_1_3y"] = mkt_raw["crsp_tsy_20y_plus"] - mkt_raw["crsp_tsy_1_3y"]
d["tsy_belly_minus_barbell"] = (
    mkt_raw[["crsp_tsy_3_7y", "crsp_tsy_7_10y"]].mean(axis=1)
    - mkt_raw[["crsp_tsy_1_3y", "crsp_tsy_20y_plus"]].mean(axis=1)
)
d["tsy_ret_dispersion_1m"] = _row_dispersion(mkt_raw, treasury_cols)
d["tsy_ret_dispersion_6m"] = _roll_mean(d["tsy_ret_dispersion_1m"], 6)
d.update(_rolling_pairwise_corr_stats(mkt_raw, treasury_cols, window=12, prefix="tsy"))
d.update(_lambda_f_series(mkt_raw, treasury_cols, window=12, prefix="tsy", include_corr=True))

d["credit_hy_minus_ig"] = mkt_raw["credit_hy_proxy_ret"] - mkt_raw["credit_ig_proxy_ret"]
d["credit_ig_ret_6m"] = _roll_sum(mkt_raw["credit_ig_proxy_ret"], 6)
d["credit_hy_ret_6m"] = _roll_sum(mkt_raw["credit_hy_proxy_ret"], 6)
d["credit_hy_ig_corr_12m_fisher"] = _safe_fisher_corr(
    mkt_raw["credit_hy_proxy_ret"].rolling(12, min_periods=12).corr(mkt_raw["credit_ig_proxy_ret"])
)
d["credit_hy_ig_corr_12m_fisher_diff"] = d["credit_hy_ig_corr_12m_fisher"].diff()
d["credit_ret_dispersion_1m"] = _row_dispersion(mkt_raw, credit_cols)
d["credit_ret_dispersion_6m"] = _roll_mean(d["credit_ret_dispersion_1m"], 6)
d.update(_lambda_f_series(mkt_raw, credit_cols, window=12, prefix="credit", include_corr=False))

d["cmdty_ret_1m"] = mkt_raw["cmdty_broad_aqr_ret"]
d["cmdty_ret_6m"] = _roll_sum(mkt_raw["cmdty_broad_aqr_ret"], 6)
d["cmdty_ret_12m"] = _roll_sum(mkt_raw["cmdty_broad_aqr_ret"], 12)
d["cmdty_vol_12m"] = _roll_std(mkt_raw["cmdty_broad_aqr_ret"], 12) * np.sqrt(12.0)

cmdty_roll_mean_12 = _roll_mean(mkt_raw["cmdty_broad_aqr_ret"], 12)
cmdty_roll_std_12 = _roll_std(mkt_raw["cmdty_broad_aqr_ret"], 12)

d["cmdty_sharpe_12m"] = (
    cmdty_roll_mean_12
    / (cmdty_roll_std_12 + EPS)
    * np.sqrt(12.0)
)

d["nfci_level"] = mkt_raw["fred_nfci"]
d["nfci_diff"] = mkt_raw["fred_nfci"].diff()
d["anfci_level"] = mkt_raw["fred_anfci"]
d["anfci_diff"] = mkt_raw["fred_anfci"].diff()
d["nfci_minus_anfci"] = mkt_raw["fred_nfci"] - mkt_raw["fred_anfci"]

d["cross_bond_proxy_ret"] = mkt_raw[treasury_cols].mean(axis=1)

tmp_cross = pd.concat(
    [
        mkt_raw,
        pd.DataFrame({"cross_bond_proxy_ret": d["cross_bond_proxy_ret"]}, index=mkt_raw.index),
    ],
    axis=1,
)

d["cross_eq_bond_corr_12m_fisher_diff"] = _rolling_corr_fisher_diff(
    tmp_cross, "ff_mktrf", "cross_bond_proxy_ret", window=12
)
d["cross_eq_tsy20_corr_12m_fisher_diff"] = _rolling_corr_fisher_diff(
    tmp_cross, "ff_mktrf", "crsp_tsy_20y_plus", window=12
)
d["cross_eq_credit_hy_corr_12m_fisher_diff"] = _rolling_corr_fisher_diff(
    tmp_cross, "ff_mktrf", "credit_hy_proxy_ret", window=12
)
d["cross_bond_credit_ig_corr_12m_fisher_diff"] = _rolling_corr_fisher_diff(
    tmp_cross, "cross_bond_proxy_ret", "credit_ig_proxy_ret", window=12
)
d["cross_eq_cmdty_corr_12m_fisher_diff"] = _rolling_corr_fisher_diff(
    tmp_cross, "ff_mktrf", "cmdty_broad_aqr_ret", window=12
)
d["cross_bond_cmdty_corr_12m_fisher_diff"] = _rolling_corr_fisher_diff(
    tmp_cross, "cross_bond_proxy_ret", "cmdty_broad_aqr_ret", window=12
)

equity_excess = mkt_raw["sprtrn_sp500"] - mkt_raw["rf"]
bond_excess = mkt_raw["agg_ret"] - mkt_raw["rf"]

rho_raw = equity_excess.rolling(12, min_periods=12).corr(bond_excess)
rho_fisher = _safe_fisher_corr(rho_raw)

d["policy_rate_diff"] = _diff1(mkt_raw["raw_DGS1"])
d["curve_10y_2y_diff"] = _diff1(mkt_raw["derived_slope_10y_2y_wrds"])
d["curve_belly_diff"] = _diff1(mkt_raw["derived_belly_slope"])
d["credit_spread_diff"] = _diff1(mkt_raw["credit_spread_baa_aaa"])
d["core_inflation_diff"] = _diff1(mkt_raw["CORESTICKM159SFRBATL"])
d["vix_logdiff"] = _log_diff1(mkt_raw["VIXCLSx"])
d["rv_60d_logdiff"] = _log_diff1(mkt_raw["rv_60d"])
d["equity_market_level"] = mkt_raw["vwretd"]
d["rho_12m_eq_fi_fisher_diff"] = rho_fisher.diff()
d["cfnai_level"] = mkt_raw["CFNAI"]
d["sentiment_logdiff"] = _log_diff1(mkt_raw["UMCSENTx"])
d["bond_10y_ret_level"] = mkt_raw["bond_10y_ret"]
d["bond_10y_mom_diff"] = _diff1(mkt_raw["bond_10y_mom_12m"])

derived_df = pd.DataFrame(d, index=mkt_raw.index)

raw_cols_to_drop = [c for c in derived_df.columns if c in mkt_raw.columns and c != "month_end"]

mkt_struct_panel_intermediate = pd.concat(
    [
        mkt_raw.drop(columns=raw_cols_to_drop, errors="ignore"),
        derived_df,
    ],
    axis=1,
).copy()

mkt_struct_panel_intermediate = (
    mkt_struct_panel_intermediate
    .sort_values("month_end")
    .drop_duplicates("month_end", keep="last")
    .reset_index(drop=True)
)


# =============================================================================
# 5. Final base-feature panels
# =============================================================================

SOURCE_ONLY_COLS = [
    "CFNAI",
    "CORESTICKM159SFRBATL",
    "UMCSENTx",
    "VIXCLSx",
    "agg_ret",
    "bond_10y_mom_12m",
    "bond_10y_ret",
    "credit_spread_baa_aaa",
    "derived_belly_slope",
    "derived_slope_10y_2y_wrds",
    "raw_DGS1",
    "rf",
    "rv_60d",
    "sprtrn_sp500",
    "vwretd",
]

missing_source_only = [c for c in SOURCE_ONLY_COLS if c not in mkt_struct_panel_intermediate.columns]
if missing_source_only:
    raise KeyError(f"Expected source-only columns missing before finalization: {missing_source_only}")

mkt_struct_panel_pre_winsor = (
    mkt_struct_panel_intermediate
    .drop(columns=SOURCE_ONLY_COLS)
    .sort_values("month_end")
    .drop_duplicates("month_end", keep="last")
    .reset_index(drop=True)
)

feature_cols = [c for c in mkt_struct_panel_pre_winsor.columns if c != "month_end"]

if len(feature_cols) != EXPECTED_FINAL_FEATURE_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_FINAL_FEATURE_COUNT} final features, got {len(feature_cols)}."
    )

mkt_struct_panel_merged_full_history = mkt_struct_panel_pre_winsor.copy()

winsor_rows = []

for c in feature_cols:
    before = pd.to_numeric(mkt_struct_panel_merged_full_history[c], errors="coerce")

    after = rolling_winsorize_series(
        before,
        window=WINSOR_WINDOW,
        min_periods=WINSOR_MIN_PERIODS,
        q_low=WINSOR_Q_LOW,
        q_high=WINSOR_Q_HIGH,
    )

    changed = before.notna() & after.notna() & (before != after)

    winsor_rows.append({
        "variable": c,
        "n_obs": int(before.notna().sum()),
        "n_winsorized": int(changed.sum()),
        "winsorized_pct": float(changed.mean()),
        "min_before": float(before.min(skipna=True)) if before.notna().any() else np.nan,
        "max_before": float(before.max(skipna=True)) if before.notna().any() else np.nan,
        "min_after": float(after.min(skipna=True)) if after.notna().any() else np.nan,
        "max_after": float(after.max(skipna=True)) if after.notna().any() else np.nan,
    })

    mkt_struct_panel_merged_full_history[c] = after

winsor_report = (
    pd.DataFrame(winsor_rows)
    .sort_values(["n_winsorized", "variable"], ascending=[False, True])
    .reset_index(drop=True)
)

still_present = [c for c in SOURCE_ONLY_COLS if c in mkt_struct_panel_merged_full_history.columns]
if still_present:
    raise ValueError(f"Source-only columns still present after finalization: {still_present}")

with_credit_feature_cols = [c for c in mkt_struct_panel_merged_full_history.columns if c != "month_end"]
complete_with_credit = mkt_struct_panel_merged_full_history[with_credit_feature_cols].notna().all(axis=1)

if not complete_with_credit.any():
    raise ValueError("No complete-case rows found for with-credit panel.")

first_with_credit = mkt_struct_panel_merged_full_history.loc[complete_with_credit, "month_end"].iloc[0]
last_with_credit = mkt_struct_panel_merged_full_history.loc[complete_with_credit, "month_end"].iloc[-1]

mkt_struct_panel_merged = (
    mkt_struct_panel_merged_full_history
    .loc[
        (mkt_struct_panel_merged_full_history["month_end"] >= first_with_credit)
        & (mkt_struct_panel_merged_full_history["month_end"] <= last_with_credit)
    ]
    .copy()
    .reset_index(drop=True)
)

final_complete_check = mkt_struct_panel_merged[with_credit_feature_cols].notna().all(axis=1)

if not final_complete_check.all():
    bad_dates = mkt_struct_panel_merged.loc[~final_complete_check, "month_end"].dt.date.tolist()
    raise ValueError(f"With-credit saved range contains non-complete rows: {bad_dates[:20]}")

mkt_struct_panel_merged.to_csv(OUT_WITH_CREDIT, index=False)

CREDIT_FEATURE_COLS_TO_DROP = [
    "credit_ig_proxy_ret",
    "credit_hy_proxy_ret",
    "credit_hy_minus_ig",
    "credit_ig_ret_6m",
    "credit_hy_ret_6m",
    "credit_hy_ig_corr_12m_fisher",
    "credit_hy_ig_corr_12m_fisher_diff",
    "credit_ret_dispersion_1m",
    "credit_ret_dispersion_6m",
    "credit_lambda_cov_12m",
    "credit_lambda_cov_diff",
    "cross_eq_credit_hy_corr_12m_fisher_diff",
    "cross_bond_credit_ig_corr_12m_fisher_diff",
]

missing_credit_drop = [c for c in CREDIT_FEATURE_COLS_TO_DROP if c not in mkt_struct_panel_merged_full_history.columns]
if missing_credit_drop:
    raise KeyError(f"Credit columns expected for wo-credit drop are missing: {missing_credit_drop}")

wo_credit_full_history = (
    mkt_struct_panel_merged_full_history
    .drop(columns=CREDIT_FEATURE_COLS_TO_DROP)
    .copy()
)

wo_credit_feature_cols = [c for c in wo_credit_full_history.columns if c != "month_end"]
complete_wo_credit = wo_credit_full_history[wo_credit_feature_cols].notna().all(axis=1)

if not complete_wo_credit.any():
    raise ValueError("No complete-case rows found for wo-credit panel.")

first_wo_credit = wo_credit_full_history.loc[complete_wo_credit, "month_end"].iloc[0]
last_wo_credit = wo_credit_full_history.loc[complete_wo_credit, "month_end"].iloc[-1]

mkt_struct_panel_merged_wo_credit = (
    wo_credit_full_history
    .loc[
        (wo_credit_full_history["month_end"] >= first_wo_credit)
        & (wo_credit_full_history["month_end"] <= last_wo_credit)
    ]
    .copy()
    .reset_index(drop=True)
)

wo_credit_complete_check = mkt_struct_panel_merged_wo_credit[wo_credit_feature_cols].notna().all(axis=1)

if not wo_credit_complete_check.all():
    bad_dates = mkt_struct_panel_merged_wo_credit.loc[~wo_credit_complete_check, "month_end"].dt.date.tolist()
    raise ValueError(f"Wo-credit saved range contains non-complete rows: {bad_dates[:20]}")

mkt_struct_panel_merged_wo_credit.to_csv(OUT_WO_CREDIT, index=False)


# =============================================================================
# 6. Student panels
# =============================================================================

student_ensemble_feature_panel, raw_cols_credit, z_cols_credit, smooth_cols_credit = add_z_and_smooth(
    mkt_struct_panel_merged,
    output_date_col="date",
)

expected_credit_cols = 1 + 3 * len(raw_cols_credit)

if student_ensemble_feature_panel.shape[1] != expected_credit_cols:
    raise ValueError(
        f"With-credit student panel column count mismatch. "
        f"Expected {expected_credit_cols}, got {student_ensemble_feature_panel.shape[1]}."
    )

student_ensemble_feature_panel.to_csv(STUDENT_WITH_CREDIT_PATH, index=False)

student_ensemble_feature_panel_wo_credit, raw_cols_wo_credit, z_cols_wo_credit, smooth_cols_wo_credit = add_z_and_smooth(
    mkt_struct_panel_merged_wo_credit,
    output_date_col="date",
)

expected_wo_credit_cols = 1 + 3 * len(raw_cols_wo_credit)

if student_ensemble_feature_panel_wo_credit.shape[1] != expected_wo_credit_cols:
    raise ValueError(
        f"Without-credit student panel column count mismatch. "
        f"Expected {expected_wo_credit_cols}, got {student_ensemble_feature_panel_wo_credit.shape[1]}."
    )

student_ensemble_feature_panel_wo_credit.to_csv(STUDENT_WO_CREDIT_PATH, index=False)


# =============================================================================
# 7. Diagnostics
# =============================================================================

availability_with_credit = coverage_report(mkt_struct_panel_merged, with_credit_feature_cols)
availability_wo_credit = coverage_report(mkt_struct_panel_merged_wo_credit, wo_credit_feature_cols)

print("=" * 120)
print("STUDENT FEATURE PANEL CONSTRUCTION COMPLETE")
print("=" * 120)

print("\nInput files")
print("MKT_PATH:", MKT_PATH)
print("CMDTY_PATH:", CMDTY_PATH)
print("MODEL_PANEL_PATH:", MODEL_PANEL_PATH)

print("\nRaw panel")
print("Object: raw_aligned_final")
print("Shape:", raw_aligned_final.shape)
print(
    "Date range:",
    raw_aligned_final["month_end"].min().date(),
    "to",
    raw_aligned_final["month_end"].max().date(),
)
print("Raw variables excluding month_end:", raw_aligned_final.shape[1] - 1)

print("\nFull-history base-feature panel")
print("Object: mkt_struct_panel_merged_full_history")
print("Shape:", mkt_struct_panel_merged_full_history.shape)
print(
    "Date range:",
    mkt_struct_panel_merged_full_history["month_end"].min().date(),
    "to",
    mkt_struct_panel_merged_full_history["month_end"].max().date(),
)
print("Feature variables excluding month_end:", len(with_credit_feature_cols))

print("\nWinsorization")
print("Window:", WINSOR_WINDOW)
print("Min periods:", WINSOR_MIN_PERIODS)
print("Lower quantile:", WINSOR_Q_LOW)
print("Upper quantile:", WINSOR_Q_HIGH)
print("Total winsorized observations:", int(winsor_report["n_winsorized"].sum()))
print("Variables winsorized at least once:", int((winsor_report["n_winsorized"] > 0).sum()))

print("\nWith-credit base-feature panel")
print("Object: mkt_struct_panel_merged")
print("Path:", OUT_WITH_CREDIT)
print("Shape:", mkt_struct_panel_merged.shape)
print(
    "Date range:",
    mkt_struct_panel_merged["month_end"].min().date(),
    "to",
    mkt_struct_panel_merged["month_end"].max().date(),
)
print("Feature variables excluding month_end:", len(with_credit_feature_cols))
print("Complete rows:", int(mkt_struct_panel_merged[with_credit_feature_cols].notna().all(axis=1).sum()))

print("\nWithout-credit base-feature panel")
print("Object: mkt_struct_panel_merged_wo_credit")
print("Path:", OUT_WO_CREDIT)
print("Shape:", mkt_struct_panel_merged_wo_credit.shape)
print(
    "Date range:",
    mkt_struct_panel_merged_wo_credit["month_end"].min().date(),
    "to",
    mkt_struct_panel_merged_wo_credit["month_end"].max().date(),
)
print("Feature variables excluding month_end:", len(wo_credit_feature_cols))
print("Dropped credit-related features:", len(CREDIT_FEATURE_COLS_TO_DROP))
print("Complete rows:", int(mkt_struct_panel_merged_wo_credit[wo_credit_feature_cols].notna().all(axis=1).sum()))

print("\nSource-only columns")
print(SOURCE_ONLY_COLS)

print("\nCredit-related features dropped in the without-credit version")
print(CREDIT_FEATURE_COLS_TO_DROP)

print("\nTop winsorized features")
print(winsor_report.head(80).to_string(index=False))

print("\nWith-credit availability after complete-case cut")
print(availability_with_credit.to_string(index=False))

print("\nWithout-credit availability after complete-case cut")
print(availability_wo_credit.to_string(index=False))

student_panel_summary(
    "WITH-CREDIT STUDENT ENSEMBLE FEATURE PANEL",
    student_ensemble_feature_panel,
    raw_cols_credit,
    z_cols_credit,
    smooth_cols_credit,
    STUDENT_WITH_CREDIT_PATH,
)

student_panel_summary(
    "WITHOUT-CREDIT STUDENT ENSEMBLE FEATURE PANEL",
    student_ensemble_feature_panel_wo_credit,
    raw_cols_wo_credit,
    z_cols_wo_credit,
    smooth_cols_wo_credit,
    STUDENT_WO_CREDIT_PATH,
)


# =============================================================================
# 8. Plots
# =============================================================================

if RUN_GRID_PLOTS:
    plot_raw_z_smooth_grid(
        input_df=student_ensemble_feature_panel,
        panel_name="WITH-CREDIT STUDENT FEATURE PANEL",
        plot_start_date=PLOT_START_DATE,
        plot_end_date=PLOT_END_DATE,
        n_cols=N_COLS,
        n_rows=N_ROWS,
        figsize=FIGSIZE,
        use_twin_axis=USE_TWIN_AXIS,
    )

else:
    print("\nGRID PLOTS SKIPPED: RUN_GRID_PLOTS = False")

In [ ]:
# =============================================================================
# Teacher Regime Labeling — SJM TEACHER CONSTRUCTION AND MAJORITY-VOTE TARGETS
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import time
from pathlib import Path
from matplotlib.patches import Patch
from matplotlib.gridspec import GridSpec

from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist

warnings.filterwarnings("ignore")

try:
    from numba import njit
    NUMBA_AVAILABLE = True
except Exception:
    NUMBA_AVAILABLE = False

EPS = 1e-12
RANDOM_SEED = 123

pd.set_option("display.max_columns", 320)
pd.set_option("display.width", 340)
pd.set_option("display.float_format", lambda x: f"{x:0.6f}")


# =============================================================================
# 1. Paths and configuration
# =============================================================================

BASE_PANEL_PATH = BASE_DIR / "base_panel_monthly_trunc_2024_11.parquet"

if not BASE_PANEL_PATH.exists():
    raise FileNotFoundError(f"Missing base panel: {BASE_PANEL_PATH}")

if "student_ensemble_feature_panel" not in globals():
    STUDENT_WITH_CREDIT_PATH = BASE_DIR / "student_ensemble_feature_panel.csv"
    if not STUDENT_WITH_CREDIT_PATH.exists():
        raise FileNotFoundError(f"Missing student feature panel: {STUDENT_WITH_CREDIT_PATH}")
    student_ensemble_feature_panel = pd.read_csv(STUDENT_WITH_CREDIT_PATH)

student_ensemble_feature_panel = student_ensemble_feature_panel.copy()
student_ensemble_feature_panel["date"] = pd.to_datetime(student_ensemble_feature_panel["date"])
student_ensemble_feature_panel = (
    student_ensemble_feature_panel
    .sort_values("date")
    .drop_duplicates("date", keep="last")
    .reset_index(drop=True)
)

HALFLIFE_MAP = {
    "h1": 1,
    "h2": 2,
    "h4": 4,
}

Q_LO = 0.01
Q_HI = 0.99
EPS_FLOOR_VALUE = np.log(EPS)

FINAL_TEACHER_FEATURE_GROUPS = [
    "SHORT_TAIL_VOL",
    "EWM_RETURN_RISK_DOWNSIDE",
    "EWM_RETURN_RISK_DRAWDOWN",
]

FINAL_TEACHER_MODEL_FAMILIES = [
    "SJM_L2",
    "SJM_L1_MEDOIDS",
]

ASSETS = ["equity", "bond"]

MIN_TRAIN_OBS_GRID = [96, 120]
ACTIVE_MIN_TRAIN_OBS = 120

MIN_TRAIN_OBS = ACTIVE_MIN_TRAIN_OBS
REFIT_EVERY = 1

TC_BPS = 5.0
TC = TC_BPS / 10000.0

LAMBDA_GRID = [
    0.0, 0.05, 0.10, 0.25, 0.50,
    1.00, 2.00, 4.00, 8.00, 16.00,
]

MAX_CD_ITER = 50
N_INIT = 10

STATE_NAMING_RULES = {
    "SHORT_TAIL_VOL": {
        "ordering_metric": "realized_ann_vol",
        "higher_is_state1": True,
        "state1_name": "VOL_HIGH",
        "state0_name": "VOL_LOW",
    },
    "EWM_RETURN_RISK_DOWNSIDE": {
        "ordering_metric": "realized_sharpe",
        "higher_is_state1": True,
        "state1_name": "RETURN_RISK_HIGH_SHARPE",
        "state0_name": "RETURN_RISK_LOW_SHARPE",
    },
    "EWM_RETURN_RISK_DRAWDOWN": {
        "ordering_metric": "realized_sharpe",
        "higher_is_state1": True,
        "state1_name": "PATH_HIGH_SHARPE",
        "state0_name": "PATH_LOW_SHARPE",
    },
}

SAVE_TEACHER_FIGURES = False
TEACHER_PLOT_DIR = BASE_DIR / "teacher_label_plots"


# =============================================================================
# 2. Helper functions
# =============================================================================

def _title(x):
    print("\n" + "=" * 160)
    print(x)
    print("=" * 160)


def _date(x):
    if pd.isna(x):
        return ""
    return pd.to_datetime(x).strftime("%Y-%m-%d")


def _to_series(x, name):
    if isinstance(x, pd.DataFrame):
        if x.shape[1] != 1:
            raise ValueError(f"{name} must be a Series or single-column DataFrame.")
        x = x.iloc[:, 0]

    if not isinstance(x, pd.Series):
        raise TypeError(f"{name} must be a pandas Series or single-column DataFrame.")

    out = x.copy()
    out.name = name
    out.index = pd.to_datetime(out.index)

    return out.sort_index().astype(float)


def _safe_div(a, b):
    if isinstance(b, pd.Series):
        return a / b.replace(0.0, np.nan)

    return a / np.where(b == 0, np.nan, b)


def _rolling_sum(s, w):
    return s.rolling(w, min_periods=w).sum()


def _rolling_mean(s, w):
    return s.rolling(w, min_periods=w).mean()


def _rolling_std(s, w):
    if w == 1:
        return s.abs()

    return s.rolling(w, min_periods=w).std(ddof=1)


def _rolling_quantile(s, w, q):
    return s.rolling(w, min_periods=w).quantile(q)


def _rolling_min(s, w):
    return s.rolling(w, min_periods=w).min()


def _downside_dev(s, w, annualize=True):
    neg_sq = np.minimum(s, 0.0) ** 2
    out = np.sqrt(neg_sq.rolling(w, min_periods=w).mean())

    if annualize:
        out = np.sqrt(12.0) * out

    return out


def _sortino(s, w):
    avg = _rolling_mean(s, w)
    down = _downside_dev(s, w, annualize=True)

    return np.sqrt(12.0) * _safe_div(avg, down)


def _sharpe(s, w):
    avg = _rolling_mean(s, w)
    vol = np.sqrt(12.0) * _rolling_std(s, w)

    return np.sqrt(12.0) * _safe_div(avg, vol)


def _wealth_from_returns(r):
    return (1.0 + r.fillna(0.0)).cumprod()


def _rolling_drawdown_from_wealth(P, w):
    peak = P.rolling(w, min_periods=w).max()

    return _safe_div(P, peak) - 1.0


def _expanding_drawdown_from_wealth(P):
    peak = P.expanding(min_periods=1).max()

    return _safe_div(P, peak) - 1.0


def _recovery_from_trough(P, w):
    trough = P.rolling(w, min_periods=w).min()

    return _safe_div(P, trough) - 1.0


def _ewm_mean(s, halflife):
    return s.ewm(halflife=halflife, adjust=False, min_periods=halflife).mean()


def _ewm_std(s, halflife):
    return s.ewm(halflife=halflife, adjust=False, min_periods=halflife).std()


def _ewm_downside(s, halflife, annualize=False):
    neg_sq = np.minimum(s, 0.0) ** 2
    out = np.sqrt(
        neg_sq.ewm(
            halflife=halflife,
            adjust=False,
            min_periods=halflife,
        ).mean()
    )

    if annualize:
        out = np.sqrt(12.0) * out

    return out


def _ewm_drawdown(dd_exp, halflife):
    return dd_exp.ewm(halflife=halflife, adjust=False, min_periods=halflife).mean()


def _ewm_recovery(recovery_source, halflife):
    return recovery_source.ewm(halflife=halflife, adjust=False, min_periods=halflife).mean()


def _minimal_local_vol_feature(x, prefix):
    out = pd.DataFrame(index=x.index)
    out[f"{prefix}_z_std_6"] = x.rolling(6, min_periods=6).std(ddof=1)

    return out


def _count_switches(states):
    s = pd.Series(states).dropna()

    if len(s) <= 1:
        return 0

    return int((s != s.shift(1)).sum() - 1)


def _mean_spell_length(states):
    s = pd.Series(states).dropna()

    if len(s) == 0:
        return np.nan

    return float(len(s) / (1 + _count_switches(s)))


def _annualized_avg_ret(x):
    x = pd.Series(x).dropna()

    return float(12.0 * x.mean()) if len(x) else np.nan


def _annualized_vol(x):
    x = pd.Series(x).dropna()

    if len(x) < 3:
        return np.nan

    sd = x.std(ddof=1)

    return float(np.sqrt(12.0) * sd) if np.isfinite(sd) else np.nan


def _annualized_sharpe(x):
    x = pd.Series(x).dropna()

    if len(x) < 3:
        return np.nan

    sd = x.std(ddof=1)

    if not np.isfinite(sd) or sd < EPS:
        return np.nan

    return float(np.sqrt(12.0) * x.mean() / sd)


def _downside_dev_ann(x):
    x = pd.Series(x).dropna().astype(float)

    if len(x) < 3:
        return np.nan

    neg = np.minimum(x, 0.0)

    return float(np.sqrt(12.0) * np.sqrt(np.mean(neg ** 2)))


def _realized_drawdown_from_returns(r):
    r = pd.Series(r).astype(float).fillna(0.0)
    wealth = (1.0 + r).cumprod()
    peak = wealth.cummax()

    return wealth / peak - 1.0


def _avg_drawdown_from_dd(dd):
    dd = pd.Series(dd).dropna()

    return float(dd.mean()) if len(dd) else np.nan


def _avg_drawdown_from_returns(ret):
    dd = _realized_drawdown_from_returns(ret)

    return float(dd.mean()) if len(dd) else np.nan


def _max_drawdown(ret):
    dd = _realized_drawdown_from_returns(ret)

    return float(dd.min()) if len(dd) else np.nan


def _teacher_only_strategy_metrics(q_state1, asset_ret, tc=TC):
    q = pd.Series(q_state1).astype(float)
    r = pd.Series(asset_ret).astype(float)
    w = q.shift(1).fillna(0.0)
    turnover = w.diff().abs().fillna(0.0)
    net = w * r - tc * turnover

    return {
        "teacher_only_net_sharpe": _annualized_sharpe(net),
        "teacher_only_max_drawdown": _max_drawdown(net),
        "teacher_only_turnover": float(turnover.mean()),
    }


def _asset_ret_col(asset):
    return "equity_excess" if asset == "equity" else "bond_excess"


def _asset_dd_col(asset):
    return "equity_dd_realized" if asset == "equity" else "bond_dd_realized"


def make_spec_id(df):
    return (
        df["model_family"].astype(str)
        + "|"
        + df["feature_source"].astype(str)
        + "|"
        + df["spec_name"].astype(str)
    )


# =============================================================================
# 3. Teacher feature construction
# =============================================================================

def _asset_feature_block(r, prefix):
    out = pd.DataFrame(index=r.index)
    P = _wealth_from_returns(r)

    out[f"{prefix}_ret_1m"] = r
    out[f"{prefix}_ret_3m"] = _rolling_sum(r, 3)

    out[f"{prefix}_mom1m_lag"] = r.shift(1)
    out[f"{prefix}_mom3m_lag"] = r.shift(1).rolling(3, min_periods=3).sum()
    out[f"{prefix}_chmom_1_3"] = out[f"{prefix}_mom1m_lag"] - _safe_div(out[f"{prefix}_mom3m_lag"], 3.0)

    out[f"{prefix}_vol_1m"] = np.sqrt(12.0) * r.abs()
    out[f"{prefix}_vol_3m"] = np.sqrt(12.0) * _rolling_std(r, 3)
    out[f"{prefix}_dvol_3m"] = out[f"{prefix}_vol_3m"] - out[f"{prefix}_vol_3m"].shift(3)
    out[f"{prefix}_volratio_1_3"] = _safe_div(out[f"{prefix}_vol_1m"], out[f"{prefix}_vol_3m"])

    out[f"{prefix}_sharpe_3m"] = _sharpe(r, 3)
    out[f"{prefix}_sortino_3m"] = _sortino(r, 3)

    out[f"{prefix}_downside_1m"] = _downside_dev(r, 1, annualize=True)
    out[f"{prefix}_downside_3m"] = _downside_dev(r, 3, annualize=True)

    out[f"{prefix}_logdownside_1m"] = np.log1p(out[f"{prefix}_downside_1m"].clip(lower=0.0))
    out[f"{prefix}_logdownside_3m"] = np.log1p(out[f"{prefix}_downside_3m"].clip(lower=0.0))

    out[f"{prefix}_worst_3m"] = _rolling_min(r, 3)
    out[f"{prefix}_tail10_6m"] = _rolling_quantile(r, 6, 0.10)

    out[f"{prefix}_dd_exp"] = _expanding_drawdown_from_wealth(P)

    for w in [3, 6, 12]:
        out[f"{prefix}_dd_{w}m"] = _rolling_drawdown_from_wealth(P, w)
        out[f"{prefix}_recovery_{w}m"] = _recovery_from_trough(P, w)

    for w in [1, 3, 6]:
        out[f"{prefix}_ddspeed_{w}m"] = out[f"{prefix}_dd_exp"] - out[f"{prefix}_dd_exp"].shift(w)

    for label, hl in HALFLIFE_MAP.items():
        out[f"{prefix}_mean_{label}"] = _ewm_mean(r, hl)
        out[f"{prefix}_std_{label}"] = _ewm_std(r, hl)
        out[f"{prefix}_ewm_downside_{label}"] = _ewm_downside(r, hl, annualize=False)

        out[f"{prefix}_sortino_{label}"] = _safe_div(
            out[f"{prefix}_mean_{label}"],
            out[f"{prefix}_ewm_downside_{label}"] + EPS,
        )

        out[f"{prefix}_sharpe_{label}"] = _safe_div(
            out[f"{prefix}_mean_{label}"],
            out[f"{prefix}_std_{label}"] + EPS,
        )

        out[f"{prefix}_logdd_{label}"] = np.log(out[f"{prefix}_ewm_downside_{label}"] + EPS)
        out[f"{prefix}_dd_{label}"] = _ewm_drawdown(out[f"{prefix}_dd_exp"], hl)
        out[f"{prefix}_recovery_{label}"] = _ewm_recovery(out[f"{prefix}_recovery_3m"], hl)

    out[f"{prefix}_ddspeed_h2"] = out[f"{prefix}_dd_h2"] - out[f"{prefix}_dd_h2"].shift(1)

    out = out.join(_minimal_local_vol_feature(r, prefix))

    for label in HALFLIFE_MAP:
        c = f"{prefix}_logdd_{label}"
        floor_mask = np.isclose(out[c], EPS_FLOOR_VALUE, atol=1e-10)
        out.loc[floor_mask, c] = np.nan

    return out


def _causal_winsorize_iqr(df, cols, k=5.0, min_periods=60):
    out = df.copy()

    for c in cols:
        if c not in out.columns:
            continue

        s = out[c].astype(float)

        med = s.expanding(min_periods=min_periods).median().shift(1)
        q75 = s.expanding(min_periods=min_periods).quantile(0.75).shift(1)
        q25 = s.expanding(min_periods=min_periods).quantile(0.25).shift(1)

        iqr = (q75 - q25).replace(0.0, np.nan)

        lo = med - k * iqr
        hi = med + k * iqr

        out[c] = s.clip(lower=lo, upper=hi)

    return out


def _static_quantile_winsorize(df, cols, q_lo=0.01, q_hi=0.99):
    out = df.copy()

    for c in cols:
        if c not in out.columns:
            continue

        s = out[c].astype(float)
        valid = s.dropna()

        if valid.empty:
            continue

        lo = valid.quantile(q_lo)
        hi = valid.quantile(q_hi)

        out[c] = s.clip(lower=lo, upper=hi)

    return out


def _causal_expanding_zscore(df, cols, min_periods=60):
    out = df.copy()

    for c in cols:
        if c not in out.columns:
            continue

        past_mean = out[c].expanding(min_periods=min_periods).mean().shift(1)
        past_std = out[c].expanding(min_periods=min_periods).std(ddof=1).shift(1)

        out[c + "_z"] = (out[c] - past_mean) / past_std.replace(0.0, np.nan)

    return out


def _causal_expanding_robust_zscore(df, cols, min_periods=60):
    out = df.copy()

    for c in cols:
        if c not in out.columns:
            continue

        past_median = out[c].expanding(min_periods=min_periods).median().shift(1)
        past_q75 = out[c].expanding(min_periods=min_periods).quantile(0.75).shift(1)
        past_q25 = out[c].expanding(min_periods=min_periods).quantile(0.25).shift(1)
        past_iqr = past_q75 - past_q25

        out[c + "_rz"] = (out[c] - past_median) / past_iqr.replace(0.0, np.nan)

    return out


def _z(cols):
    return [c + "_z" for c in cols]


def _rz(cols):
    return [c + "_rz" for c in cols]


_src = pd.read_parquet(BASE_PANEL_PATH).copy()
_src["date"] = pd.to_datetime(_src["date"])
_src = _src.sort_values("date").set_index("date")

required_base_cols = ["sprtrn_sp500", "agg_ret", "rf"]
missing = [c for c in required_base_cols if c not in _src.columns]

if missing:
    raise KeyError(f"Base panel missing required columns: {missing}")

eq_ret = _to_series(_src["sprtrn_sp500"], "equity_ret")
bd_ret = _to_series(_src["agg_ret"], "bond_ret")
rf_ret = _to_series(_src["rf"], "rf")

base = pd.concat([eq_ret, bd_ret, rf_ret], axis=1).sort_index()
base = base.dropna(subset=["equity_ret", "bond_ret", "rf"]).copy()

base["equity_excess"] = base["equity_ret"] - base["rf"]
base["bond_excess"] = base["bond_ret"] - base["rf"]

eq_features = _asset_feature_block(base["equity_excess"], "eq")
bd_features = _asset_feature_block(base["bond_excess"], "bd")

feature_panel_raw = pd.concat(
    [
        base[["equity_ret", "bond_ret", "rf", "equity_excess", "bond_excess"]],
        eq_features,
        bd_features,
    ],
    axis=1,
)

feature_panel_raw = (
    feature_panel_raw
    .reset_index()
    .rename(columns={"index": "date"})
    .sort_values("date")
    .reset_index(drop=True)
    .replace([np.inf, -np.inf], np.nan)
)

feature_panel_raw["date"] = pd.to_datetime(feature_panel_raw["date"])

EWM_RETURN_RISK_DOWNSIDE_EQ = [
    "eq_mean_h1", "eq_mean_h2", "eq_mean_h4",
    "eq_sortino_h1", "eq_sortino_h2", "eq_sortino_h4",
    "eq_logdd_h1", "eq_logdd_h2", "eq_logdd_h4",
]

EWM_RETURN_RISK_DOWNSIDE_BD = [
    "bd_mean_h1", "bd_mean_h2", "bd_mean_h4",
    "bd_sharpe_h1", "bd_sharpe_h2", "bd_sharpe_h4",
    "bd_logdd_h1", "bd_logdd_h2", "bd_logdd_h4",
]

EWM_RETURN_RISK_DRAWDOWN_EQ = [
    "eq_mean_h1", "eq_mean_h2", "eq_mean_h4",
    "eq_sortino_h1", "eq_sortino_h2", "eq_sortino_h4",
    "eq_logdd_h1", "eq_logdd_h2", "eq_logdd_h4",
    "eq_dd_h1", "eq_dd_h4", "eq_ddspeed_h2", "eq_recovery_h4",
]

EWM_RETURN_RISK_DRAWDOWN_BD = [
    "bd_mean_h1", "bd_mean_h2", "bd_mean_h4",
    "bd_sharpe_h1", "bd_sharpe_h2", "bd_sharpe_h4",
    "bd_logdd_h1", "bd_logdd_h2", "bd_logdd_h4",
    "bd_dd_h1", "bd_dd_h4", "bd_ddspeed_h2", "bd_recovery_h4",
]

SHORT_TAIL_VOL_EQ = [
    "eq_vol_1m", "eq_vol_3m", "eq_dvol_3m", "eq_volratio_1_3",
    "eq_downside_1m", "eq_downside_3m",
    "eq_logdownside_1m", "eq_logdownside_3m",
    "eq_worst_3m", "eq_tail10_6m", "eq_z_std_6",
]

SHORT_TAIL_VOL_BD = [
    "bd_vol_1m", "bd_vol_3m", "bd_dvol_3m", "bd_volratio_1_3",
    "bd_downside_1m", "bd_downside_3m",
    "bd_logdownside_1m", "bd_logdownside_3m",
    "bd_worst_3m", "bd_tail10_6m", "bd_z_std_6",
]

FEATURE_GROUPS_RAW = {
    "SHORT_TAIL_VOL": {
        "equity": SHORT_TAIL_VOL_EQ,
        "bond": SHORT_TAIL_VOL_BD,
    },
    "EWM_RETURN_RISK_DOWNSIDE": {
        "equity": EWM_RETURN_RISK_DOWNSIDE_EQ,
        "bond": EWM_RETURN_RISK_DOWNSIDE_BD,
    },
    "EWM_RETURN_RISK_DRAWDOWN": {
        "equity": EWM_RETURN_RISK_DRAWDOWN_EQ,
        "bond": EWM_RETURN_RISK_DRAWDOWN_BD,
    },
}

all_raw_feature_cols = sorted(
    set(
        col
        for group_dict in FEATURE_GROUPS_RAW.values()
        for asset_cols in group_dict.values()
        for col in asset_cols
    )
)

missing = [c for c in all_raw_feature_cols if c not in feature_panel_raw.columns]

if missing:
    raise KeyError(f"Missing expected teacher feature columns: {missing}")

STATIC_P1_P99_WINSOR_FEATURES = [
    "bd_volratio_1_3",
    "bd_sharpe_3m",
]

COLS_TO_WINSORIZE = sorted(
    set(
        c for c in all_raw_feature_cols
        if (
            "sortino" in c
            or "sharpe" in c
            or "logdd" in c
            or "ddspeed" in c
            or "volratio" in c
            or "tail10" in c
        )
    )
    - set(STATIC_P1_P99_WINSOR_FEATURES)
)

feature_panel_raw = _causal_winsorize_iqr(
    feature_panel_raw,
    cols=COLS_TO_WINSORIZE,
    k=5.0,
    min_periods=60,
)

feature_panel_raw = _static_quantile_winsorize(
    feature_panel_raw,
    cols=STATIC_P1_P99_WINSOR_FEATURES,
    q_lo=Q_LO,
    q_hi=Q_HI,
)

feature_panel_raw = feature_panel_raw.replace([np.inf, -np.inf], np.nan)

feature_panel_std = _causal_expanding_zscore(
    feature_panel_raw,
    cols=all_raw_feature_cols,
    min_periods=60,
)

feature_panel_std = _causal_expanding_robust_zscore(
    feature_panel_std,
    cols=all_raw_feature_cols,
    min_periods=60,
)

feature_panel_std = feature_panel_std.replace([np.inf, -np.inf], np.nan)

FEATURE_GROUPS_STD = {
    group_name: {
        "equity": _z(group_dict["equity"]),
        "bond": _z(group_dict["bond"]),
    }
    for group_name, group_dict in FEATURE_GROUPS_RAW.items()
}

FEATURE_GROUPS_ROBUST_STD = {
    group_name: {
        "equity": _rz(group_dict["equity"]),
        "bond": _rz(group_dict["bond"]),
    }
    for group_name, group_dict in FEATURE_GROUPS_RAW.items()
}

feature_panel_for_teacher = feature_panel_std.copy()

work = feature_panel_for_teacher.copy()
work["date"] = pd.to_datetime(work["date"])
work = work.replace([np.inf, -np.inf], np.nan).sort_values("date").reset_index(drop=True)

work["equity_dd_realized"] = _realized_drawdown_from_returns(work["equity_excess"])
work["bond_dd_realized"] = _realized_drawdown_from_returns(work["bond_excess"])

_title("TEACHER FEATURE PANEL BUILT")
print("Base panel rows:", len(_src))
print("Student panel rows:", len(student_ensemble_feature_panel))
print("Teacher panel rows:", len(feature_panel_for_teacher))
print("Teacher date range:", feature_panel_for_teacher["date"].min(), "to", feature_panel_for_teacher["date"].max())
print("Teacher feature groups:", list(FEATURE_GROUPS_RAW.keys()))
print("Raw teacher features:", len(all_raw_feature_cols))


# =============================================================================
# 4. SJM fitters
# =============================================================================

if NUMBA_AVAILABLE:
    @njit(cache=False)
    def _dp_path_numba(loss, jump_lambda):
        n = loss.shape[0]
        V = np.empty((n, 2), dtype=np.float64)
        back = np.empty((n, 2), dtype=np.int64)

        V[0, 0] = loss[0, 0]
        V[0, 1] = loss[0, 1]
        back[0, 0] = 0
        back[0, 1] = 1

        for t in range(1, n):
            for s in range(2):
                stay = V[t - 1, s]
                switch = V[t - 1, 1 - s] + jump_lambda

                if stay <= switch:
                    V[t, s] = loss[t, s] + stay
                    back[t, s] = s
                else:
                    V[t, s] = loss[t, s] + switch
                    back[t, s] = 1 - s

        path = np.empty(n, dtype=np.int64)
        path[n - 1] = 0 if V[n - 1, 0] <= V[n - 1, 1] else 1

        for t in range(n - 2, -1, -1):
            path[t] = back[t + 1, path[t + 1]]

        obj = V[n - 1, path[n - 1]]

        return path, obj


def _dp_path(loss, jump_lambda):
    loss = np.asarray(loss, dtype=np.float64)

    if loss.ndim != 2 or loss.shape[1] != 2:
        raise ValueError("Only K=2 supported.")

    if NUMBA_AVAILABLE:
        return _dp_path_numba(loss, float(jump_lambda))

    n = loss.shape[0]
    V = np.zeros((n, 2), dtype=float)
    back = np.zeros((n, 2), dtype=int)
    V[0] = loss[0]

    for t in range(1, n):
        for s in [0, 1]:
            stay = V[t - 1, s]
            switch = V[t - 1, 1 - s] + jump_lambda

            if stay <= switch:
                V[t, s] = loss[t, s] + stay
                back[t, s] = s
            else:
                V[t, s] = loss[t, s] + switch
                back[t, s] = 1 - s

    path = np.zeros(n, dtype=int)
    path[-1] = int(np.argmin(V[-1]))

    for t in range(n - 2, -1, -1):
        path[t] = back[t + 1, path[t + 1]]

    return path, float(np.min(V[-1]))


def _init_labels(X, method, rng, asset_ret=None):
    if method == "kmeans":
        km = KMeans(
            n_clusters=2,
            n_init=10,
            random_state=int(rng.integers(1, 10_000_000)),
        )
        return km.fit_predict(X)

    if method == "quantile_ret" and asset_ret is not None:
        rr = pd.Series(asset_ret).fillna(0.0).to_numpy()
        med = np.nanmedian(rr)
        lab = (rr > med).astype(int)

        if len(np.unique(lab)) == 2:
            return lab

    if method == "random":
        lab = rng.integers(0, 2, size=X.shape[0])

        if len(np.unique(lab)) == 2:
            return lab

    km = KMeans(
        n_clusters=2,
        n_init=10,
        random_state=int(rng.integers(1, 10_000_000)),
    )

    return km.fit_predict(X)


def _centers_from_labels_mean(X, labels):
    centers = np.zeros((2, X.shape[1]), dtype=float)
    global_center = np.nanmean(X, axis=0)

    for k in [0, 1]:
        centers[k] = global_center if np.sum(labels == k) == 0 else np.nanmean(X[labels == k], axis=0)

    return centers


def _center_distance(centers):
    return float(np.linalg.norm(centers[0] - centers[1])) if centers is not None else np.nan


def _l2_loss_matrix(X, centers):
    return np.column_stack([
        np.sum((X - centers[0]) ** 2, axis=1),
        np.sum((X - centers[1]) ** 2, axis=1),
    ])


def _l1_loss_matrix(X, medoids):
    return np.column_stack([
        np.sum(np.abs(X - medoids[0]), axis=1),
        np.sum(np.abs(X - medoids[1]), axis=1),
    ])


def _update_medoids(X, labels, rng):
    medoids = np.zeros((2, X.shape[1]), dtype=float)

    for k in [0, 1]:
        Xk = X[labels == k]

        if len(Xk) == 0:
            medoids[k] = X[int(rng.integers(0, len(X)))]
            continue

        dist_sum = cdist(Xk, Xk, metric="cityblock").sum(axis=1)
        medoids[k] = Xk[int(np.argmin(dist_sum))]

    return medoids


def fit_sjm_l2_window(
    X,
    jump_lambda,
    asset_ret=None,
    max_iter=MAX_CD_ITER,
    n_init=N_INIT,
    seed=RANDOM_SEED,
):
    rng = np.random.default_rng(seed)
    best = None
    init_methods = ["kmeans", "quantile_ret", "random"]

    for init_id in range(n_init):
        labels = _init_labels(X, init_methods[init_id % len(init_methods)], rng, asset_ret=asset_ret)
        centers = _centers_from_labels_mean(X, labels)
        last_path = None

        for _ in range(max_iter):
            loss = _l2_loss_matrix(X, centers)
            path, _ = _dp_path(loss, jump_lambda)

            if last_path is not None and np.array_equal(path, last_path):
                break

            last_path = path.copy()
            centers = _centers_from_labels_mean(X, path)

        loss = _l2_loss_matrix(X, centers)
        path, _ = _dp_path(loss, jump_lambda)

        data_loss = float(np.sum(loss[np.arange(len(path)), path]))
        jump_loss = float(jump_lambda * np.sum(path[1:] != path[:-1]))

        candidate = {
            "labels": path,
            "obj": data_loss + jump_loss,
            "data_loss": data_loss,
            "jump_loss": jump_loss,
            "centers": centers,
            "center_dist": _center_distance(centers),
        }

        if best is None or candidate["obj"] < best["obj"]:
            best = candidate

    return best


def fit_sjm_l1_medoids_window(
    X,
    jump_lambda,
    asset_ret=None,
    max_iter=MAX_CD_ITER,
    n_init=N_INIT,
    seed=RANDOM_SEED,
):
    rng = np.random.default_rng(seed)
    best = None
    init_methods = ["kmeans", "quantile_ret", "random"]

    for init_id in range(n_init):
        labels = _init_labels(X, init_methods[init_id % len(init_methods)], rng, asset_ret=asset_ret)
        medoids = _update_medoids(X, labels, rng)
        last_path = None

        for _ in range(max_iter):
            loss = _l1_loss_matrix(X, medoids)
            path, _ = _dp_path(loss, jump_lambda)

            if last_path is not None and np.array_equal(path, last_path):
                break

            last_path = path.copy()
            medoids = _update_medoids(X, path, rng)

        loss = _l1_loss_matrix(X, medoids)
        path, _ = _dp_path(loss, jump_lambda)

        data_loss = float(np.sum(loss[np.arange(len(path)), path]))
        jump_loss = float(jump_lambda * np.sum(path[1:] != path[:-1]))

        candidate = {
            "labels": path,
            "obj": data_loss + jump_loss,
            "data_loss": data_loss,
            "jump_loss": jump_loss,
            "centers": medoids,
            "center_dist": _center_distance(medoids),
        }

        if best is None or candidate["obj"] < best["obj"]:
            best = candidate

    return best


# =============================================================================
# 5. Teacher evaluation
# =============================================================================

def _state_order_from_training_labels(labels, train_positions, asset_ret_arr, asset_dd_arr, feature_group):
    labels = np.asarray(labels, dtype=int)
    train_positions = np.asarray(train_positions, dtype=int)

    r = np.asarray(asset_ret_arr, dtype=float)[train_positions]
    dd = np.asarray(asset_dd_arr, dtype=float)[train_positions]

    rule = STATE_NAMING_RULES[feature_group]
    ordering_metric = rule["ordering_metric"]
    higher_is_state1 = bool(rule["higher_is_state1"])

    metric_by_raw = {}
    aux_stats = {}

    for k in [0, 1]:
        mask = labels == k
        rk = pd.Series(r[mask]).dropna()
        ddk = pd.Series(dd[mask]).dropna()

        aux_stats[f"raw_state{k}_n_order"] = int(mask.sum())
        aux_stats[f"raw_state{k}_avg_ret_ann_order"] = _annualized_avg_ret(rk)
        aux_stats[f"raw_state{k}_sharpe_order"] = _annualized_sharpe(rk)
        aux_stats[f"raw_state{k}_vol_ann_order"] = _annualized_vol(rk)
        aux_stats[f"raw_state{k}_avg_drawdown_order"] = _avg_drawdown_from_dd(ddk)

        if ordering_metric == "realized_sharpe":
            metric_by_raw[k] = aux_stats[f"raw_state{k}_sharpe_order"]
        elif ordering_metric == "realized_ann_vol":
            metric_by_raw[k] = aux_stats[f"raw_state{k}_vol_ann_order"]
        elif ordering_metric == "realized_avg_drawdown":
            metric_by_raw[k] = aux_stats[f"raw_state{k}_avg_drawdown_order"]
        else:
            raise ValueError(f"Unknown ordering_metric: {ordering_metric}")

    m0 = metric_by_raw[0]
    m1 = metric_by_raw[1]

    if np.isfinite(m0) and np.isfinite(m1):
        raw_state1 = int(m1 > m0) if higher_is_state1 else int(m1 < m0)
    else:
        fallback0 = aux_stats["raw_state0_avg_ret_ann_order"]
        fallback1 = aux_stats["raw_state1_avg_ret_ann_order"]
        raw_state1 = int(fallback1 > fallback0) if np.isfinite(fallback0) and np.isfinite(fallback1) else 1

    raw_state0 = 1 - raw_state1

    aux_stats["ordering_metric"] = ordering_metric
    aux_stats["raw_state_mapped_to_state1"] = raw_state1
    aux_stats["raw_state_mapped_to_state0"] = raw_state0
    aux_stats["state0_name"] = rule["state0_name"]
    aux_stats["state1_name"] = rule["state1_name"]

    return raw_state1, aux_stats


def _run_causal_hard_model(df, asset, feature_cols, fit_func, spec, feature_source_name):
    asset_ret_col = _asset_ret_col(asset)
    asset_dd_col = _asset_dd_col(asset)

    n = len(df)

    state_raw = np.full(n, np.nan)
    state_id = np.full(n, np.nan)
    q_state1 = np.full(n, np.nan)
    state_name = np.full(n, None, dtype=object)
    raw_state_mapped_to_state1_arr = np.full(n, np.nan)

    last_meta = {}

    X_all = df[feature_cols].replace([np.inf, -np.inf], np.nan).to_numpy(dtype=float)
    asset_ret_arr = df[asset_ret_col].to_numpy(dtype=float)
    asset_dd_arr = df[asset_dd_col].to_numpy(dtype=float)

    valid_features = np.isfinite(X_all).all(axis=1)
    train_positions_list = []

    for end_pos in range(n):
        if valid_features[end_pos]:
            train_positions_list.append(end_pos)

        if end_pos % REFIT_EVERY != 0:
            continue

        if not valid_features[end_pos]:
            continue

        if len(train_positions_list) < MIN_TRAIN_OBS:
            continue

        train_positions = np.asarray(train_positions_list, dtype=int)
        X_train = X_all[train_positions]
        r_train = asset_ret_arr[train_positions]

        try:
            fit = fit_func(X_train, r_train)
        except Exception as e:
            last_meta = {"fit_error": str(e)}
            continue

        labels = np.asarray(fit["labels"], dtype=int)
        raw_last = int(labels[-1])

        raw_state1, order_stats = _state_order_from_training_labels(
            labels=labels,
            train_positions=train_positions,
            asset_ret_arr=asset_ret_arr,
            asset_dd_arr=asset_dd_arr,
            feature_group=spec["feature_group"],
        )

        mapped_state = int(raw_last == raw_state1)

        state_raw[end_pos] = raw_last
        state_id[end_pos] = mapped_state
        q_state1[end_pos] = float(mapped_state)
        raw_state_mapped_to_state1_arr[end_pos] = raw_state1
        state_name[end_pos] = (
            STATE_NAMING_RULES[spec["feature_group"]]["state1_name"]
            if mapped_state == 1
            else STATE_NAMING_RULES[spec["feature_group"]]["state0_name"]
        )

        last_meta = dict(fit)
        last_meta.update(order_stats)

    panel = pd.DataFrame({
        "date": df["date"].to_numpy(),
        "asset": asset,
        "model_family": spec["model_family"],
        "feature_group": spec["feature_group"],
        "feature_source": feature_source_name,
        "spec_name": spec["spec_name"],
        "state_raw": state_raw,
        "state_id": state_id,
        "state_name": state_name,
        "q_state1": q_state1,
        "q_good": q_state1,
        "raw_state_mapped_to_state1": raw_state_mapped_to_state1_arr,
        "asset_excess": df[asset_ret_col].to_numpy(dtype=float),
        "asset_drawdown": df[asset_dd_col].to_numpy(dtype=float),
    })

    for k, v in spec.items():
        if k not in panel.columns and isinstance(v, (int, float, str, bool, np.integer, np.floating)):
            panel[k] = v

    return panel, last_meta


def _evaluate_teacher_panel(panel, spec, last_meta, feature_cols):
    p_valid = panel.dropna(subset=["q_state1", "state_id"]).copy()
    rule = STATE_NAMING_RULES[spec["feature_group"]]

    metrics = {
        "asset": spec["asset"],
        "model_family": spec["model_family"],
        "feature_group": spec["feature_group"],
        "feature_source": spec["feature_source"],
        "spec_name": spec["spec_name"],
        "n_obs_total": len(panel),
        "n_obs_signal": len(p_valid),
        "ordering_metric": rule["ordering_metric"],
        "state0_name": rule["state0_name"],
        "state1_name": rule["state1_name"],
    }

    for k, v in spec.items():
        if isinstance(v, (int, float, np.integer, np.floating, bool, str)) or v is None:
            metrics[k] = v

    if len(p_valid) == 0:
        metrics.update({
            "state0_share": np.nan,
            "state1_share": np.nan,
            "n_switches": np.nan,
            "mean_spell_length": np.nan,
            "teacher_only_net_sharpe": np.nan,
            "teacher_only_max_drawdown": np.nan,
            "teacher_only_turnover": np.nan,
        })
        return metrics

    state = p_valid["state_id"].astype(int)

    metrics["state0_share"] = float((state == 0).mean())
    metrics["state1_share"] = float((state == 1).mean())
    metrics["q_state1_mean"] = float(p_valid["q_state1"].mean())
    metrics["q_state1_std"] = float(p_valid["q_state1"].std(ddof=1))
    metrics["n_switches"] = _count_switches(state)
    metrics["mean_spell_length"] = _mean_spell_length(state)

    for k in [0, 1]:
        sub = p_valid[state == k]
        metrics[f"state{k}_avg_ret_ann"] = _annualized_avg_ret(sub["asset_excess"])
        metrics[f"state{k}_sharpe"] = _annualized_sharpe(sub["asset_excess"])
        metrics[f"state{k}_vol_ann"] = _annualized_vol(sub["asset_excess"])
        metrics[f"state{k}_avg_drawdown"] = _avg_drawdown_from_dd(sub["asset_drawdown"])

    metrics["state1_minus_state0_avg_ret_ann"] = metrics["state1_avg_ret_ann"] - metrics["state0_avg_ret_ann"]
    metrics["state1_minus_state0_sharpe"] = metrics["state1_sharpe"] - metrics["state0_sharpe"]
    metrics["state1_minus_state0_vol_ann"] = metrics["state1_vol_ann"] - metrics["state0_vol_ann"]
    metrics["state1_minus_state0_avg_drawdown"] = metrics["state1_avg_drawdown"] - metrics["state0_avg_drawdown"]

    metrics.update(_teacher_only_strategy_metrics(p_valid["q_state1"], p_valid["asset_excess"], tc=TC))

    for k, v in last_meta.items():
        if k in {"labels", "probs", "prob_last", "centers"}:
            continue

        if isinstance(v, (int, float, np.integer, np.floating, bool, str)) or v is None:
            metrics[k] = v

    return metrics


# =============================================================================
# 6. Majority-vote target construction
# =============================================================================

selected_six_specs_template = pd.DataFrame([
    {
        "asset": "equity",
        "feature_group": "SHORT_TAIL_VOL",
        "state_col": "eq_risk_state",
        "prob_col": "eq_risk_prob",
        "good_state": 0,
        "role": "Equity risk state",
    },
    {
        "asset": "equity",
        "feature_group": "EWM_RETURN_RISK_DOWNSIDE",
        "state_col": "eq_downside_state",
        "prob_col": "eq_downside_prob",
        "good_state": 1,
        "role": "Equity return-risk state",
    },
    {
        "asset": "equity",
        "feature_group": "EWM_RETURN_RISK_DRAWDOWN",
        "state_col": "eq_drawdown_state",
        "prob_col": "eq_drawdown_prob",
        "good_state": 1,
        "role": "Equity drawdown-aware return-risk state",
    },
    {
        "asset": "bond",
        "feature_group": "SHORT_TAIL_VOL",
        "state_col": "bd_risk_state",
        "prob_col": "bd_risk_prob",
        "good_state": 0,
        "role": "Bond risk state",
    },
    {
        "asset": "bond",
        "feature_group": "EWM_RETURN_RISK_DOWNSIDE",
        "state_col": "bd_downside_state",
        "prob_col": "bd_downside_prob",
        "good_state": 1,
        "role": "Bond return-risk state",
    },
    {
        "asset": "bond",
        "feature_group": "EWM_RETURN_RISK_DRAWDOWN",
        "state_col": "bd_drawdown_state",
        "prob_col": "bd_drawdown_prob",
        "good_state": 1,
        "role": "Bond drawdown-aware return-risk state",
    },
])

level_df = work[["date", "equity_excess", "bond_excess"]].copy()
level_df["date"] = pd.to_datetime(level_df["date"])
level_df = level_df.sort_values("date").reset_index(drop=True)

level_df["equity_level"] = 100.0 * (1.0 + level_df["equity_excess"].fillna(0.0)).cumprod()
level_df["bond_level"] = 100.0 * (1.0 + level_df["bond_excess"].fillna(0.0)).cumprod()
level_df["equity_drawdown_full_path"] = _realized_drawdown_from_returns(level_df["equity_excess"])
level_df["bond_drawdown_full_path"] = _realized_drawdown_from_returns(level_df["bond_excess"])


def _available_cols(cols):
    return [c for c in cols if c in work.columns]


def _state_conditional_metrics(df, state_col, ret_col, target_name, asset, feature_group, timing):
    x = df[["date", state_col, ret_col]].dropna().copy()
    x[state_col] = x[state_col].astype(int)

    bad = x.loc[x[state_col].eq(0), ret_col]
    good = x.loc[x[state_col].eq(1), ret_col]

    out = {
        "target": target_name,
        "timing": timing,
        "asset": asset,
        "feature_group": feature_group,
        "n_obs": int(len(x)),
        "n_bad": int(len(bad)),
        "n_good": int(len(good)),
        "share_bad": float((x[state_col] == 0).mean()) if len(x) else np.nan,
        "share_good": float((x[state_col] == 1).mean()) if len(x) else np.nan,
        "switches": _count_switches(x[state_col]),
        "spell": _mean_spell_length(x[state_col]),
        "ann_mean_bad": _annualized_avg_ret(bad),
        "ann_mean_good": _annualized_avg_ret(good),
        "ann_vol_bad": _annualized_vol(bad),
        "ann_vol_good": _annualized_vol(good),
        "ann_downside_bad": _downside_dev_ann(bad),
        "ann_downside_good": _downside_dev_ann(good),
        "avg_drawdown_bad": _avg_drawdown_from_returns(bad),
        "avg_drawdown_good": _avg_drawdown_from_returns(good),
        "max_drawdown_bad": _max_drawdown(bad),
        "max_drawdown_good": _max_drawdown(good),
        "ann_sharpe_bad": _annualized_sharpe(bad),
        "ann_sharpe_good": _annualized_sharpe(good),
    }

    out["good_minus_bad_ann_mean"] = out["ann_mean_good"] - out["ann_mean_bad"]
    out["good_minus_bad_ann_vol"] = out["ann_vol_good"] - out["ann_vol_bad"]
    out["good_minus_bad_downside"] = out["ann_downside_good"] - out["ann_downside_bad"]
    out["good_minus_bad_avg_drawdown"] = out["avg_drawdown_good"] - out["avg_drawdown_bad"]
    out["good_minus_bad_sharpe"] = out["ann_sharpe_good"] - out["ann_sharpe_bad"]

    return out


def _build_one_teacher_version(min_train_obs):
    global MIN_TRAIN_OBS

    MIN_TRAIN_OBS = int(min_train_obs)
    version_tag = f"minobs{MIN_TRAIN_OBS}"

    teacher_store_local = {}
    teacher_metrics_local = []
    teacher_panels_local = []

    def _register_result_local(asset, model_family, feature_group, spec_name, panel, metrics, spec, feature_cols):
        key = (asset, model_family, feature_group, spec_name)

        teacher_store_local[key] = {
            "panel": panel,
            "metrics": metrics,
            "spec": spec,
            "feature_cols": feature_cols,
        }

        teacher_metrics_local.append(metrics)
        teacher_panels_local.append(panel)

    def _run_and_store_hard_local(asset, model_family, feature_group, feature_source, spec_name, spec_extra, feature_cols, fit_func):
        spec = {
            "asset": asset,
            "model_family": model_family,
            "feature_group": feature_group,
            "feature_source": feature_source,
            "spec_name": spec_name,
        }

        spec.update(spec_extra)

        panel, meta = _run_causal_hard_model(
            work,
            asset,
            feature_cols,
            fit_func,
            spec,
            feature_source_name=feature_source,
        )

        metrics = _evaluate_teacher_panel(panel, spec, meta, feature_cols)
        metrics["min_train_obs"] = MIN_TRAIN_OBS
        panel["min_train_obs"] = MIN_TRAIN_OBS

        _register_result_local(asset, model_family, feature_group, spec_name, panel, metrics, spec, feature_cols)

    _title(f"RUNNING SJM TEACHER GRID — MIN_TRAIN_OBS = {MIN_TRAIN_OBS}")
    print("Feature groups:", FINAL_TEACHER_FEATURE_GROUPS)
    print("Model families:", FINAL_TEACHER_MODEL_FAMILIES)
    print("Lambda grid:", LAMBDA_GRID)
    print("MIN_TRAIN_OBS:", MIN_TRAIN_OBS)
    print("Numba available:", NUMBA_AVAILABLE)

    t0 = time.time()

    for asset in ASSETS:
        for feature_group in FINAL_TEACHER_FEATURE_GROUPS:
            feature_cols = _available_cols(FEATURE_GROUPS_STD[feature_group][asset])

            for lam in LAMBDA_GRID:
                spec_name = f"SJM_L2_lam{lam:g}"

                def fit_func(X_train, r_train, lam=lam):
                    return fit_sjm_l2_window(
                        X_train,
                        jump_lambda=lam,
                        asset_ret=r_train,
                        max_iter=MAX_CD_ITER,
                        n_init=N_INIT,
                        seed=RANDOM_SEED,
                    )

                _run_and_store_hard_local(
                    asset=asset,
                    model_family="SJM_L2",
                    feature_group=feature_group,
                    feature_source="STD_Z",
                    spec_name=spec_name,
                    spec_extra={"lambda": lam},
                    feature_cols=feature_cols,
                    fit_func=fit_func,
                )

    for asset in ASSETS:
        for feature_group in FINAL_TEACHER_FEATURE_GROUPS:
            feature_cols = _available_cols(FEATURE_GROUPS_ROBUST_STD[feature_group][asset])

            for lam in LAMBDA_GRID:
                spec_name = f"SJM_L1_MEDOIDS_lam{lam:g}"

                def fit_func(X_train, r_train, lam=lam):
                    return fit_sjm_l1_medoids_window(
                        X_train,
                        jump_lambda=lam,
                        asset_ret=r_train,
                        max_iter=MAX_CD_ITER,
                        n_init=N_INIT,
                        seed=RANDOM_SEED,
                    )

                _run_and_store_hard_local(
                    asset=asset,
                    model_family="SJM_L1_MEDOIDS",
                    feature_group=feature_group,
                    feature_source="ROBUST_RZ",
                    spec_name=spec_name,
                    spec_extra={"lambda": lam},
                    feature_cols=feature_cols,
                    fit_func=fit_func,
                )

    teacher_results_local = pd.DataFrame(teacher_metrics_local)
    teacher_panel_all_local = pd.concat(teacher_panels_local, axis=0, ignore_index=True)

    _title(f"SJM GRID COMPLETE — MIN_TRAIN_OBS = {MIN_TRAIN_OBS}")
    print("Elapsed seconds:", round(time.time() - t0, 2))
    print("teacher_results shape:", teacher_results_local.shape)
    print("teacher_panel_all shape:", teacher_panel_all_local.shape)

    selected_specs_local = teacher_results_local.copy()

    valid = (
        selected_specs_local["n_obs_signal"].notna()
        & selected_specs_local["n_obs_signal"].gt(0)
        & selected_specs_local["state1_share"].notna()
        & selected_specs_local["state1_share"].gt(0.0)
        & selected_specs_local["state1_share"].lt(1.0)
        & selected_specs_local["n_switches"].notna()
        & selected_specs_local["mean_spell_length"].notna()
    )

    selected_specs_local = (
        selected_specs_local.loc[valid]
        .drop_duplicates(["asset", "model_family", "feature_group", "feature_source", "spec_name"])
        .reset_index(drop=True)
    )

    selected_specs_local["spec_id"] = make_spec_id(selected_specs_local)

    _title(f"VALID SJM TEACHER PATHS — MIN_TRAIN_OBS = {MIN_TRAIN_OBS}")

    valid_summary = (
        selected_specs_local
        .groupby(["asset", "feature_group", "model_family", "feature_source"], dropna=False)
        .agg(
            n_paths=("spec_name", "count"),
            min_state1_share=("state1_share", "min"),
            max_state1_share=("state1_share", "max"),
            median_switches=("n_switches", "median"),
            median_spell=("mean_spell_length", "median"),
            median_teacher_sharpe=("teacher_only_net_sharpe", "median"),
        )
        .reset_index()
    )

    print(valid_summary.to_string(index=False))

    majority_vote_specs_local = selected_six_specs_template.copy()
    majority_vote_specs_local["target_design"] = f"all_valid_sjm_majority_vote_{version_tag}"
    majority_vote_specs_local["consensus_rule"] = "all_valid_sjm_majority_vote"
    majority_vote_specs_local["consensus_threshold"] = 0.50
    majority_vote_specs_local["label_similarity_threshold"] = np.nan
    majority_vote_specs_local["label_cluster_id"] = np.nan
    majority_vote_specs_local["min_train_obs"] = MIN_TRAIN_OBS

    tp_local = teacher_panel_all_local.copy()
    tp_local["date"] = pd.to_datetime(tp_local["date"])

    tp_local = tp_local.merge(
        selected_specs_local[
            ["asset", "model_family", "feature_group", "feature_source", "spec_name", "spec_id"]
        ],
        on=["asset", "model_family", "feature_group", "feature_source", "spec_name"],
        how="inner",
    )

    all_majority_parts = []
    majority_diag_rows = []
    majority_panels_local = {}
    wide_parts = []

    for _, spec in majority_vote_specs_local.iterrows():
        asset = spec["asset"]
        feature_group = spec["feature_group"]
        state_col = spec["state_col"]
        prob_col = spec["prob_col"]
        good_state = int(spec["good_state"])

        sub = tp_local[
            tp_local["asset"].eq(asset)
            & tp_local["feature_group"].eq(feature_group)
        ].copy()

        if sub.empty:
            raise ValueError(
                f"No valid SJM teachers found for asset={asset}, "
                f"feature_group={feature_group}, min_train_obs={MIN_TRAIN_OBS}"
            )

        state_wide = (
            sub.pivot_table(index="date", columns="spec_id", values="state_id", aggfunc="last")
            .sort_index()
        )

        q_wide = (
            sub.pivot_table(index="date", columns="spec_id", values="q_state1", aggfunc="last")
            .reindex(state_wide.index)
            .sort_index()
        )

        n_teachers = state_wide.notna().sum(axis=1)
        avg_state1 = state_wide.mean(axis=1, skipna=True)
        avg_qstate1 = q_wide.mean(axis=1, skipna=True)

        majority_state1 = pd.Series(np.nan, index=avg_state1.index, dtype=float)
        valid_vote = n_teachers.gt(0)
        majority_state1.loc[valid_vote] = (avg_state1.loc[valid_vote] >= 0.5).astype(float)

        out = pd.DataFrame({
            "date": avg_state1.index,
            "asset": asset,
            "feature_group": feature_group,
            "n_teachers_available": n_teachers.to_numpy(dtype=float),
            "all_teacher_vote_state1": avg_state1.to_numpy(dtype=float),
            "all_teacher_avg_q_state1": avg_qstate1.to_numpy(dtype=float),
            "all_teacher_majority_state1": majority_state1.to_numpy(dtype=float),
        }).reset_index(drop=True)

        if good_state == 1:
            out[state_col] = out["all_teacher_majority_state1"]
            out[prob_col] = out["all_teacher_avg_q_state1"]
        else:
            out[state_col] = 1.0 - out["all_teacher_majority_state1"]
            out[prob_col] = 1.0 - out["all_teacher_avg_q_state1"]

        out.loc[out["all_teacher_majority_state1"].isna(), state_col] = np.nan
        out.loc[out["all_teacher_avg_q_state1"].isna(), prob_col] = np.nan

        out[state_col + "_h1"] = out[state_col].shift(-1)
        out[prob_col + "_h1"] = out[prob_col].shift(-1)
        out["min_train_obs"] = MIN_TRAIN_OBS

        majority_panels_local[(asset, feature_group)] = out.copy()

        all_majority_parts.append(
            out[[
                "date",
                "asset",
                "feature_group",
                state_col,
                prob_col,
                state_col + "_h1",
                prob_col + "_h1",
                "n_teachers_available",
                "all_teacher_vote_state1",
                "all_teacher_avg_q_state1",
                "all_teacher_majority_state1",
                "min_train_obs",
            ]]
        )

        wide_parts.append(
            out[[
                "date",
                state_col,
                prob_col,
                state_col + "_h1",
                prob_col + "_h1",
            ]].copy()
        )

        majority_diag_rows.append({
            "target": state_col,
            "target_h1": state_col + "_h1",
            "asset": asset,
            "feature_group": feature_group,
            "min_train_obs": MIN_TRAIN_OBS,
            "n_obs_current": int(out[state_col].notna().sum()),
            "n_obs_h1": int(out[state_col + "_h1"].notna().sum()),
            "first_valid_current": _date(out.loc[out[state_col].notna(), "date"].min()) if out[state_col].notna().any() else "",
            "last_valid_current": _date(out.loc[out[state_col].notna(), "date"].max()) if out[state_col].notna().any() else "",
            "first_valid_h1": _date(out.loc[out[state_col + "_h1"].notna(), "date"].min()) if out[state_col + "_h1"].notna().any() else "",
            "last_valid_h1": _date(out.loc[out[state_col + "_h1"].notna(), "date"].max()) if out[state_col + "_h1"].notna().any() else "",
            "good_share_current": float(out[state_col].mean(skipna=True)),
            "good_share_h1": float(out[state_col + "_h1"].mean(skipna=True)),
            "switches_current": _count_switches(out[state_col]),
            "spell_current": _mean_spell_length(out[state_col]),
            "teachers_min": float(out["n_teachers_available"].min(skipna=True)),
            "teachers_median": float(out["n_teachers_available"].median(skipna=True)),
            "teachers_max": float(out["n_teachers_available"].max(skipna=True)),
            "mean_vote_strength": float(np.maximum(out[prob_col], 1.0 - out[prob_col]).mean(skipna=True)),
        })

    all_teacher_majority_long_local = pd.concat(all_majority_parts, axis=0, ignore_index=True)
    all_teacher_majority_diag_local = pd.DataFrame(majority_diag_rows)

    majority_vote_target_wide_local = wide_parts[0].copy()

    for part in wide_parts[1:]:
        majority_vote_target_wide_local = majority_vote_target_wide_local.merge(part, on="date", how="outer")

    majority_vote_target_wide_local = (
        majority_vote_target_wide_local
        .sort_values("date")
        .reset_index(drop=True)
    )

    majority_vote_target_wide_local["min_train_obs"] = MIN_TRAIN_OBS

    majority_vote_target_diagnostics_local = []

    for _, spec in majority_vote_specs_local.iterrows():
        state_col = spec["state_col"]
        prob_col = spec["prob_col"]
        h1_col = state_col + "_h1"
        h1_prob_col = prob_col + "_h1"

        majority_vote_target_diagnostics_local.append({
            "target": h1_col,
            "asset": spec["asset"],
            "feature_group": spec["feature_group"],
            "target_design": f"all_valid_sjm_majority_vote_{version_tag}",
            "min_train_obs": MIN_TRAIN_OBS,
            "n_obs": int(majority_vote_target_wide_local[h1_col].notna().sum()),
            "positive_share_good_state": float(majority_vote_target_wide_local[h1_col].mean(skipna=True)),
            "prob_mean": float(majority_vote_target_wide_local[h1_prob_col].mean(skipna=True)),
            "prob_std": float(majority_vote_target_wide_local[h1_prob_col].std(skipna=True)),
            "first_valid": majority_vote_target_wide_local.loc[
                majority_vote_target_wide_local[h1_col].notna(), "date"
            ].min() if majority_vote_target_wide_local[h1_col].notna().any() else pd.NaT,
            "last_valid": majority_vote_target_wide_local.loc[
                majority_vote_target_wide_local[h1_col].notna(), "date"
            ].max() if majority_vote_target_wide_local[h1_col].notna().any() else pd.NaT,
        })

    majority_vote_target_diagnostics_local = pd.DataFrame(majority_vote_target_diagnostics_local)

    metric_rows_current = []
    metric_rows_h1 = []

    metric_base = majority_vote_target_wide_local.merge(
        level_df[["date", "equity_excess", "bond_excess"]],
        on="date",
        how="left",
    ).sort_values("date").reset_index(drop=True)

    for _, spec in majority_vote_specs_local.iterrows():
        asset = spec["asset"]
        feature_group = spec["feature_group"]
        state_col = spec["state_col"]
        h1_col = state_col + "_h1"
        ret_col = "equity_excess" if asset == "equity" else "bond_excess"

        metric_rows_current.append(
            _state_conditional_metrics(
                metric_base,
                state_col,
                ret_col,
                state_col,
                asset,
                feature_group,
                timing="current",
            )
        )

        metric_rows_h1.append(
            _state_conditional_metrics(
                metric_base,
                h1_col,
                ret_col,
                h1_col,
                asset,
                feature_group,
                timing="h1_student_target",
            )
        )

    majority_vote_current_metrics_local = pd.DataFrame(metric_rows_current)
    majority_vote_h1_metrics_local = pd.DataFrame(metric_rows_h1)

    majority_vote_current_metrics_local["min_train_obs"] = MIN_TRAIN_OBS
    majority_vote_h1_metrics_local["min_train_obs"] = MIN_TRAIN_OBS

    _title(f"MAJORITY-VOTE TARGET DIAGNOSTICS — MIN_TRAIN_OBS = {MIN_TRAIN_OBS}")
    print(majority_vote_target_diagnostics_local.to_string(index=False))

    _title(f"ALL-VALID-SJM MAJORITY DESIGN TABLE — MIN_TRAIN_OBS = {MIN_TRAIN_OBS}")
    print(all_teacher_majority_diag_local.to_string(index=False))

    metric_cols = [
        "target",
        "timing",
        "asset",
        "feature_group",
        "n_obs",
        "share_good",
        "switches",
        "spell",
        "ann_mean_good",
        "ann_mean_bad",
        "ann_vol_good",
        "ann_vol_bad",
        "ann_downside_good",
        "ann_downside_bad",
        "avg_drawdown_good",
        "avg_drawdown_bad",
        "ann_sharpe_good",
        "ann_sharpe_bad",
        "good_minus_bad_ann_mean",
        "good_minus_bad_ann_vol",
        "good_minus_bad_downside",
        "good_minus_bad_avg_drawdown",
        "good_minus_bad_sharpe",
    ]

    _title(f"TABLE 1 — MAJORITY-VOTE CURRENT-MONTH TARGET METRICS — MIN_TRAIN_OBS = {MIN_TRAIN_OBS}")
    print(majority_vote_current_metrics_local[metric_cols].to_string(index=False))

    _title(f"TABLE 2 — MAJORITY-VOTE H1 STUDENT TARGET METRICS — MIN_TRAIN_OBS = {MIN_TRAIN_OBS}")
    print(majority_vote_h1_metrics_local[metric_cols].to_string(index=False))

    return {
        "version_tag": version_tag,
        "min_train_obs": MIN_TRAIN_OBS,
        "teacher_store": teacher_store_local,
        "teacher_results": teacher_results_local,
        "teacher_panel_all": teacher_panel_all_local,
        "selected_teacher_specs_for_label_clustering": selected_specs_local,
        "majority_vote_specs": majority_vote_specs_local,
        "majority_vote_target_wide": majority_vote_target_wide_local,
        "majority_vote_target_diagnostics": majority_vote_target_diagnostics_local,
        "majority_vote_current_metrics": majority_vote_current_metrics_local,
        "majority_vote_h1_metrics": majority_vote_h1_metrics_local,
        "all_teacher_majority_long": all_teacher_majority_long_local,
        "all_teacher_majority_diag": all_teacher_majority_diag_local,
        "majority_panels": majority_panels_local,
        "valid_summary": valid_summary,
    }


teacher_versions = {}

for min_obs in MIN_TRAIN_OBS_GRID:
    teacher_versions[int(min_obs)] = _build_one_teacher_version(int(min_obs))


# =============================================================================
# 7. Versioned object aliases
# =============================================================================

teacher_results_by_minobs = {
    k: v["teacher_results"] for k, v in teacher_versions.items()
}

teacher_panel_all_by_minobs = {
    k: v["teacher_panel_all"] for k, v in teacher_versions.items()
}

selected_teacher_specs_for_label_clustering_by_minobs = {
    k: v["selected_teacher_specs_for_label_clustering"] for k, v in teacher_versions.items()
}

majority_vote_specs_by_minobs = {
    k: v["majority_vote_specs"] for k, v in teacher_versions.items()
}

majority_vote_target_wide_by_minobs = {
    k: v["majority_vote_target_wide"] for k, v in teacher_versions.items()
}

majority_vote_target_diagnostics_by_minobs = {
    k: v["majority_vote_target_diagnostics"] for k, v in teacher_versions.items()
}

majority_vote_current_metrics_by_minobs = {
    k: v["majority_vote_current_metrics"] for k, v in teacher_versions.items()
}

majority_vote_h1_metrics_by_minobs = {
    k: v["majority_vote_h1_metrics"] for k, v in teacher_versions.items()
}

all_teacher_majority_long_by_minobs = {
    k: v["all_teacher_majority_long"] for k, v in teacher_versions.items()
}

all_teacher_majority_diag_by_minobs = {
    k: v["all_teacher_majority_diag"] for k, v in teacher_versions.items()
}

majority_panels_by_minobs = {
    k: v["majority_panels"] for k, v in teacher_versions.items()
}

teacher_results_96 = teacher_versions[96]["teacher_results"]
teacher_panel_all_96 = teacher_versions[96]["teacher_panel_all"]
selected_teacher_specs_for_label_clustering_96 = teacher_versions[96]["selected_teacher_specs_for_label_clustering"]
majority_vote_specs_96 = teacher_versions[96]["majority_vote_specs"]
majority_vote_target_wide_96 = teacher_versions[96]["majority_vote_target_wide"]
majority_vote_target_diagnostics_96 = teacher_versions[96]["majority_vote_target_diagnostics"]
majority_vote_current_metrics_96 = teacher_versions[96]["majority_vote_current_metrics"]
majority_vote_h1_metrics_96 = teacher_versions[96]["majority_vote_h1_metrics"]
all_teacher_majority_long_96 = teacher_versions[96]["all_teacher_majority_long"]
all_teacher_majority_diag_96 = teacher_versions[96]["all_teacher_majority_diag"]
majority_panels_96 = teacher_versions[96]["majority_panels"]

teacher_results_120 = teacher_versions[120]["teacher_results"]
teacher_panel_all_120 = teacher_versions[120]["teacher_panel_all"]
selected_teacher_specs_for_label_clustering_120 = teacher_versions[120]["selected_teacher_specs_for_label_clustering"]
majority_vote_specs_120 = teacher_versions[120]["majority_vote_specs"]
majority_vote_target_wide_120 = teacher_versions[120]["majority_vote_target_wide"]
majority_vote_target_diagnostics_120 = teacher_versions[120]["majority_vote_target_diagnostics"]
majority_vote_current_metrics_120 = teacher_versions[120]["majority_vote_current_metrics"]
majority_vote_h1_metrics_120 = teacher_versions[120]["majority_vote_h1_metrics"]
all_teacher_majority_long_120 = teacher_versions[120]["all_teacher_majority_long"]
all_teacher_majority_diag_120 = teacher_versions[120]["all_teacher_majority_diag"]
majority_panels_120 = teacher_versions[120]["majority_panels"]

teacher_results = teacher_versions[ACTIVE_MIN_TRAIN_OBS]["teacher_results"]
teacher_panel_all = teacher_versions[ACTIVE_MIN_TRAIN_OBS]["teacher_panel_all"]
selected_teacher_specs_for_label_clustering = teacher_versions[ACTIVE_MIN_TRAIN_OBS]["selected_teacher_specs_for_label_clustering"]
majority_vote_specs = teacher_versions[ACTIVE_MIN_TRAIN_OBS]["majority_vote_specs"]
majority_vote_target_wide = teacher_versions[ACTIVE_MIN_TRAIN_OBS]["majority_vote_target_wide"]
majority_vote_target_diagnostics = teacher_versions[ACTIVE_MIN_TRAIN_OBS]["majority_vote_target_diagnostics"]
majority_vote_current_metrics = teacher_versions[ACTIVE_MIN_TRAIN_OBS]["majority_vote_current_metrics"]
majority_vote_h1_metrics = teacher_versions[ACTIVE_MIN_TRAIN_OBS]["majority_vote_h1_metrics"]
all_teacher_majority_long = teacher_versions[ACTIVE_MIN_TRAIN_OBS]["all_teacher_majority_long"]
all_teacher_majority_diag = teacher_versions[ACTIVE_MIN_TRAIN_OBS]["all_teacher_majority_diag"]
majority_panels = teacher_versions[ACTIVE_MIN_TRAIN_OBS]["majority_panels"]

active_target_wide = majority_vote_target_wide
active_selected_six_specs = majority_vote_specs
active_target_diagnostics = majority_vote_target_diagnostics
active_min_train_obs = ACTIVE_MIN_TRAIN_OBS


# =============================================================================
# 8. Majority-vote plots
# =============================================================================

FEATURE_ORDER = [
    "SHORT_TAIL_VOL",
    "EWM_RETURN_RISK_DOWNSIDE",
    "EWM_RETURN_RISK_DRAWDOWN",
]

FEATURE_LABELS = {
    "SHORT_TAIL_VOL": "Risk state",
    "EWM_RETURN_RISK_DOWNSIDE": "Return-risk / downside state",
    "EWM_RETURN_RISK_DRAWDOWN": "Drawdown-aware state",
}

STATE_COLORS = {
    1: "#b7e4bd",
    0: "#f5b5b5",
}


def _fmt_pct(x):
    return "NA" if not np.isfinite(x) else f"{100.0 * x:0.1f}%"


def _fmt_num(x):
    return "NA" if not np.isfinite(x) else f"{x:0.2f}"


def _fmt_months(x):
    return "NA" if not np.isfinite(x) else f"{x:0.1f}m"


def _date_edges(dates):
    dates = pd.to_datetime(pd.Series(dates)).sort_values().reset_index(drop=True)

    if len(dates) == 0:
        return pd.Series(dtype="datetime64[ns]"), pd.Series(dtype="datetime64[ns]")

    if len(dates) == 1:
        left = dates.iloc[0] - pd.offsets.MonthBegin(1)
        right = dates.iloc[0] + pd.offsets.MonthEnd(1)

        return pd.Series([left]), pd.Series([right])

    mids = dates.iloc[:-1] + (dates.iloc[1:].to_numpy() - dates.iloc[:-1].to_numpy()) / 2

    left_edges = [dates.iloc[0] - (mids.iloc[0] - dates.iloc[0])]
    left_edges += list(mids)

    right_edges = list(mids)
    right_edges += [dates.iloc[-1] + (dates.iloc[-1] - mids.iloc[-1])]

    return pd.Series(left_edges), pd.Series(right_edges)


def _add_state_background_runs(ax, df, date_col, state_col):
    x = df[[date_col, state_col]].dropna().copy()

    if x.empty:
        return

    x[date_col] = pd.to_datetime(x[date_col])
    x[state_col] = x[state_col].astype(int)
    x = x.sort_values(date_col).reset_index(drop=True)

    left_edges, right_edges = _date_edges(x[date_col])
    x["left"] = left_edges
    x["right"] = right_edges
    x["run_id"] = (x[state_col] != x[state_col].shift(1)).cumsum()

    ymin, ymax = ax.get_ylim()

    for _, g in x.groupby("run_id", sort=True):
        state = int(g[state_col].iloc[0])
        ax.axvspan(
            g["left"].iloc[0],
            g["right"].iloc[-1],
            facecolor=STATE_COLORS[state],
            alpha=0.60,
            edgecolor="none",
            linewidth=0,
            antialiased=False,
            zorder=0,
        )

    ax.set_ylim(ymin, ymax)


def _recompute_presentation_metrics(target_wide, specs):
    rows = []

    asset_config = {
        "equity": {
            "ret_col": "equity_excess",
            "level_col": "equity_level",
            "dd_col": "equity_drawdown_full_path",
            "title": "Equity",
        },
        "bond": {
            "ret_col": "bond_excess",
            "level_col": "bond_level",
            "dd_col": "bond_drawdown_full_path",
            "title": "Bond",
        },
    }

    for _, spec in specs.iterrows():
        asset = spec["asset"]
        feature_group = spec["feature_group"]
        state_col = spec["state_col"]

        cfg = asset_config[asset]
        ret_col = cfg["ret_col"]
        dd_col = cfg["dd_col"]

        df = (
            target_wide[["date", state_col]]
            .merge(level_df[["date", ret_col, dd_col]], on="date", how="inner")
            .dropna(subset=[state_col, ret_col, dd_col])
            .sort_values("date")
            .reset_index(drop=True)
        )

        df[state_col] = df[state_col].astype(int)

        good = df[state_col].eq(1)
        bad = df[state_col].eq(0)

        ret_good = df.loc[good, ret_col]
        ret_bad = df.loc[bad, ret_col]

        dd_good = df.loc[good, dd_col]
        dd_bad = df.loc[bad, dd_col]

        row = {
            "target": state_col,
            "asset": asset,
            "feature_group": feature_group,
            "n_obs": int(len(df)),
            "share_good": float(good.mean()) if len(df) else np.nan,
            "switches": _count_switches(df[state_col]),
            "spell": _mean_spell_length(df[state_col]),
            "ann_mean_good": _annualized_avg_ret(ret_good),
            "ann_mean_bad": _annualized_avg_ret(ret_bad),
            "ann_vol_good": _annualized_vol(ret_good),
            "ann_vol_bad": _annualized_vol(ret_bad),
            "ann_downside_good": _downside_dev_ann(ret_good),
            "ann_downside_bad": _downside_dev_ann(ret_bad),
            "ann_sharpe_good": _annualized_sharpe(ret_good),
            "ann_sharpe_bad": _annualized_sharpe(ret_bad),
            "avg_full_path_dd_good": float(dd_good.mean()) if len(dd_good) else np.nan,
            "avg_full_path_dd_bad": float(dd_bad.mean()) if len(dd_bad) else np.nan,
            "p10_full_path_dd_good": float(dd_good.quantile(0.10)) if len(dd_good) else np.nan,
            "p10_full_path_dd_bad": float(dd_bad.quantile(0.10)) if len(dd_bad) else np.nan,
            "worst_full_path_dd_good": float(dd_good.min()) if len(dd_good) else np.nan,
            "worst_full_path_dd_bad": float(dd_bad.min()) if len(dd_bad) else np.nan,
        }

        row["good_minus_bad_ann_mean"] = row["ann_mean_good"] - row["ann_mean_bad"]
        row["good_minus_bad_ann_vol"] = row["ann_vol_good"] - row["ann_vol_bad"]
        row["good_minus_bad_downside"] = row["ann_downside_good"] - row["ann_downside_bad"]
        row["good_minus_bad_avg_full_path_dd"] = row["avg_full_path_dd_good"] - row["avg_full_path_dd_bad"]
        row["good_minus_bad_sharpe"] = row["ann_sharpe_good"] - row["ann_sharpe_bad"]

        rows.append(row)

    return pd.DataFrame(rows)


def _plot_asset_majority_for_version(min_train_obs, asset):
    result = teacher_versions[int(min_train_obs)]

    target_wide = result["majority_vote_target_wide"]
    specs = result["majority_vote_specs"]
    metrics_presentation = _recompute_presentation_metrics(target_wide, specs)

    asset_config = {
        "equity": {
            "ret_col": "equity_excess",
            "level_col": "equity_level",
            "dd_col": "equity_drawdown_full_path",
            "title": "Equity",
            "filename": str(TEACHER_PLOT_DIR / f"majority_vote1_minobs{min_train_obs}.png"),
        },
        "bond": {
            "ret_col": "bond_excess",
            "level_col": "bond_level",
            "dd_col": "bond_drawdown_full_path",
            "title": "Bond",
            "filename": str(TEACHER_PLOT_DIR / f"majority_vote2_minobs{min_train_obs}.png"),
        },
    }

    state_col_lookup = {
        (row["asset"], row["feature_group"]): row["state_col"]
        for _, row in specs.iterrows()
    }

    def _metric_row(asset_inner, feature_group, state_col):
        x = metrics_presentation.copy()
        m = (
            x["asset"].eq(asset_inner)
            & x["feature_group"].eq(feature_group)
            & x["target"].eq(state_col)
        )

        if not m.any():
            raise ValueError(f"No metric row found for {asset_inner}, {feature_group}, {state_col}")

        return x.loc[m].iloc[0].to_dict()

    def _stats_text(asset_inner, feature_group, state_col):
        r = _metric_row(asset_inner, feature_group, state_col)

        line1 = (
            f"Obs: {int(r['n_obs'])}   "
            f"Good avg: {_fmt_pct(r['share_good'])}   "
            f"Switches: {int(r['switches'])}   "
            f"Spell: {_fmt_months(r['spell'])}"
        )

        line2 = (
            f"Ann. mean good/bad: {_fmt_pct(r['ann_mean_good'])} / {_fmt_pct(r['ann_mean_bad'])}   "
            f"Sharpe good/bad: {_fmt_num(r['ann_sharpe_good'])} / {_fmt_num(r['ann_sharpe_bad'])}"
        )

        if feature_group == "SHORT_TAIL_VOL":
            line3 = (
                f"Ann. vol good/bad: {_fmt_pct(r['ann_vol_good'])} / {_fmt_pct(r['ann_vol_bad'])}"
            )
        elif feature_group == "EWM_RETURN_RISK_DOWNSIDE":
            line3 = (
                f"Downside dev. good/bad: {_fmt_pct(r['ann_downside_good'])} / {_fmt_pct(r['ann_downside_bad'])}"
            )
        elif feature_group == "EWM_RETURN_RISK_DRAWDOWN":
            line3 = (
                f"Avg full-path DD good/bad: {_fmt_pct(r['avg_full_path_dd_good'])} / {_fmt_pct(r['avg_full_path_dd_bad'])}"
            )
        else:
            line3 = ""

        return line1 + "\n" + line2 + "\n" + line3

    cfg = asset_config[asset]
    level_col = cfg["level_col"]

    fig = plt.figure(figsize=(16, 10.5))
    gs = GridSpec(
        nrows=6,
        ncols=1,
        height_ratios=[3.2, 0.95, 3.2, 0.95, 3.2, 0.95],
        hspace=0.10,
        figure=fig,
    )

    plot_axes = []

    for i, feature_group in enumerate(FEATURE_ORDER):
        ax = fig.add_subplot(gs[2 * i, 0])
        ax_stats = fig.add_subplot(gs[2 * i + 1, 0])
        plot_axes.append(ax)

        state_col = state_col_lookup[(asset, feature_group)]

        df = (
            target_wide[["date", state_col]]
            .merge(level_df[["date", level_col]], on="date", how="inner")
            .dropna(subset=[state_col, level_col])
            .sort_values("date")
            .reset_index(drop=True)
        )

        ax.plot(
            df["date"],
            df[level_col],
            color="black",
            linewidth=1.20,
            zorder=3,
        )

        _add_state_background_runs(ax, df, "date", state_col)

        ax.set_title(
            f"{cfg['title']} - {FEATURE_LABELS[feature_group]} ({state_col})",
            loc="left",
            fontsize=11,
            pad=5,
        )

        ax.set_ylabel("Level")
        ax.grid(axis="y", alpha=0.25)

        if i < len(FEATURE_ORDER) - 1:
            ax.tick_params(axis="x", labelbottom=False)
        else:
            ax.set_xlabel("Date")

        ax_stats.axis("off")
        ax_stats.text(
            0.01,
            0.50,
            _stats_text(asset, feature_group, state_col),
            ha="left",
            va="center",
            fontsize=9.2,
            family="monospace",
            transform=ax_stats.transAxes,
            bbox=dict(
                facecolor="#f7f7f7",
                edgecolor="#cccccc",
                boxstyle="round,pad=0.45",
                alpha=1.0,
            ),
        )

    legend_handles = [
        Patch(facecolor=STATE_COLORS[1], edgecolor="none", alpha=0.60, label="Good state"),
        Patch(facecolor=STATE_COLORS[0], edgecolor="none", alpha=0.60, label="Bad state"),
    ]

    plot_axes[0].legend(
        handles=legend_handles,
        loc="upper right",
        frameon=True,
        fontsize=9,
    )

    fig.suptitle(
        f"Majority-vote SJM teacher labels over {cfg['title'].lower()} level "
        f"(min train obs = {min_train_obs})",
        fontsize=15,
        y=0.992,
    )

    fig.tight_layout(rect=[0, 0, 1, 0.975])

    if SAVE_TEACHER_FIGURES:
        TEACHER_PLOT_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(cfg["filename"], dpi=220, bbox_inches="tight")

    plt.show()

    return metrics_presentation


majority_vote_current_metrics_presentation_by_minobs = {}

for min_obs in MIN_TRAIN_OBS_GRID:
    _title(f"MAJORITY-VOTE PRESENTATION METRICS — MIN_TRAIN_OBS = {min_obs}")

    metrics_presentation = _recompute_presentation_metrics(
        teacher_versions[int(min_obs)]["majority_vote_target_wide"],
        teacher_versions[int(min_obs)]["majority_vote_specs"],
    )

    majority_vote_current_metrics_presentation_by_minobs[int(min_obs)] = metrics_presentation

    display_cols = [
        "target",
        "asset",
        "feature_group",
        "n_obs",
        "share_good",
        "switches",
        "spell",
        "ann_mean_good",
        "ann_mean_bad",
        "ann_vol_good",
        "ann_vol_bad",
        "ann_downside_good",
        "ann_downside_bad",
        "avg_full_path_dd_good",
        "avg_full_path_dd_bad",
        "ann_sharpe_good",
        "ann_sharpe_bad",
    ]

    print(metrics_presentation[display_cols].to_string(index=False))

    _plot_asset_majority_for_version(int(min_obs), "equity")
    _plot_asset_majority_for_version(int(min_obs), "bond")

majority_vote_current_metrics_presentation_96 = majority_vote_current_metrics_presentation_by_minobs[96]
majority_vote_current_metrics_presentation_120 = majority_vote_current_metrics_presentation_by_minobs[120]


# =============================================================================
# 9. Final report
# =============================================================================

_title("Teacher Regime Labeling COMPLETE — SJM TEACHER VERSIONS AND MAJORITY-VOTE TARGETS")

print("Teacher versions fitted:", MIN_TRAIN_OBS_GRID)
print("Active default min_train_obs:", ACTIVE_MIN_TRAIN_OBS)
print("PNG saving:", SAVE_TEACHER_FIGURES)

print("\nVersioned dictionaries:")
print("teacher_versions")
print("teacher_results_by_minobs")
print("teacher_panel_all_by_minobs")
print("selected_teacher_specs_for_label_clustering_by_minobs")
print("majority_vote_specs_by_minobs")
print("majority_vote_target_wide_by_minobs")
print("majority_vote_target_diagnostics_by_minobs")
print("majority_vote_current_metrics_by_minobs")
print("majority_vote_h1_metrics_by_minobs")
print("all_teacher_majority_long_by_minobs")
print("all_teacher_majority_diag_by_minobs")
print("majority_panels_by_minobs")

print("\nExplicit 96-month objects:")
print(f"teacher_results_96                              shape={teacher_results_96.shape}")
print(f"teacher_panel_all_96                            shape={teacher_panel_all_96.shape}")
print(f"selected_teacher_specs_for_label_clustering_96  shape={selected_teacher_specs_for_label_clustering_96.shape}")
print(f"majority_vote_specs_96                          shape={majority_vote_specs_96.shape}")
print(f"majority_vote_target_wide_96                    shape={majority_vote_target_wide_96.shape}")
print(f"majority_vote_target_diagnostics_96             shape={majority_vote_target_diagnostics_96.shape}")
print(f"majority_vote_current_metrics_96                shape={majority_vote_current_metrics_96.shape}")
print(f"majority_vote_h1_metrics_96                     shape={majority_vote_h1_metrics_96.shape}")

print("\nExplicit 120-month objects:")
print(f"teacher_results_120                              shape={teacher_results_120.shape}")
print(f"teacher_panel_all_120                            shape={teacher_panel_all_120.shape}")
print(f"selected_teacher_specs_for_label_clustering_120  shape={selected_teacher_specs_for_label_clustering_120.shape}")
print(f"majority_vote_specs_120                          shape={majority_vote_specs_120.shape}")
print(f"majority_vote_target_wide_120                    shape={majority_vote_target_wide_120.shape}")
print(f"majority_vote_target_diagnostics_120             shape={majority_vote_target_diagnostics_120.shape}")
print(f"majority_vote_current_metrics_120                shape={majority_vote_current_metrics_120.shape}")
print(f"majority_vote_h1_metrics_120                     shape={majority_vote_h1_metrics_120.shape}")

print("\nActive downstream aliases:")
print("active_target_wide")
print("active_selected_six_specs")
print("active_target_diagnostics")
print("active_min_train_obs")

print("\nFinal H1 target columns in active_target_wide:")
for c in active_target_wide.columns:
    if c.endswith("_h1") and "_prob_" not in c and not c.endswith("prob_h1"):
        print(c)


# =============================================================================
# 10. student prediction stage input files
# =============================================================================

STUDENT_PREDICTION_INPUT_DIR = BASE_DIR / "student_prediction_inputs"
STUDENT_PREDICTION_INPUT_DIR.mkdir(parents=True, exist_ok=True)

STUDENT_PREDICTION_TARGET_WIDE_96_PATH = STUDENT_PREDICTION_INPUT_DIR / "majority_vote_target_wide_96.pkl"
STUDENT_PREDICTION_TARGET_WIDE_120_PATH = STUDENT_PREDICTION_INPUT_DIR / "majority_vote_target_wide_120.pkl"
STUDENT_PREDICTION_SPECS_96_PATH = STUDENT_PREDICTION_INPUT_DIR / "majority_vote_specs_96.pkl"
STUDENT_PREDICTION_SPECS_120_PATH = STUDENT_PREDICTION_INPUT_DIR / "majority_vote_specs_120.pkl"

majority_vote_target_wide_96.to_pickle(STUDENT_PREDICTION_TARGET_WIDE_96_PATH)
majority_vote_target_wide_120.to_pickle(STUDENT_PREDICTION_TARGET_WIDE_120_PATH)
majority_vote_specs_96.to_pickle(STUDENT_PREDICTION_SPECS_96_PATH)
majority_vote_specs_120.to_pickle(STUDENT_PREDICTION_SPECS_120_PATH)

print("\n" + "=" * 160)
print("Student Prediction and Allocation INPUT FILES SAVED")
print("=" * 160)
print(STUDENT_PREDICTION_TARGET_WIDE_96_PATH)
print(STUDENT_PREDICTION_TARGET_WIDE_120_PATH)
print(STUDENT_PREDICTION_SPECS_96_PATH)
print(STUDENT_PREDICTION_SPECS_120_PATH)


# =============================================================================
# 11. Reload code for student prediction stage
# =============================================================================

print("\nReload code for student prediction stage after restart:")
print(r'''
STUDENT_PREDICTION_INPUT_DIR = BASE_DIR / "student_prediction_inputs"

majority_vote_target_wide_96 = pd.read_pickle(STUDENT_PREDICTION_INPUT_DIR / "majority_vote_target_wide_96.pkl")
majority_vote_target_wide_120 = pd.read_pickle(STUDENT_PREDICTION_INPUT_DIR / "majority_vote_target_wide_120.pkl")
majority_vote_specs_96 = pd.read_pickle(STUDENT_PREDICTION_INPUT_DIR / "majority_vote_specs_96.pkl")
majority_vote_specs_120 = pd.read_pickle(STUDENT_PREDICTION_INPUT_DIR / "majority_vote_specs_120.pkl")

majority_vote_target_wide_by_minobs = {
    96: majority_vote_target_wide_96,
    120: majority_vote_target_wide_120,
}

majority_vote_specs_by_minobs = {
    96: majority_vote_specs_96,
    120: majority_vote_specs_120,
}
''')

In [ ]:
# =============================================================================
# Student Prediction and Allocation — STUDENT PREDICTION AND MV-TC ALLOCATION
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import time

from pathlib import Path
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
    confusion_matrix,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

warnings.filterwarnings("ignore")

EPS = 1e-12

pd.set_option("display.max_columns", 420)
pd.set_option("display.width", 420)
pd.set_option("display.float_format", lambda x: f"{x:0.6f}")


# =============================================================================
# 1. Configuration
# =============================================================================

STUDENT_PREDICTION_RUN_TAG = "full_grid_teacher96_120_student_credit_wocredit_z_zsmooth_majority_vote"

RUN_TEACHER_MINOBS = [96, 120]
RUN_STUDENT_PANELS = ["student_with_credit", "student_wo_credit"]
RUN_FEATURE_TRANSFORMS = ["z", "z_smooth"]
RUN_MODEL_FAMILIES = ["LOGIT", "XGB", "RF"]

REQUIRE_XGBOOST = True

if REQUIRE_XGBOOST and not XGBOOST_AVAILABLE:
    raise ImportError("xgboost is not available, but RUN_MODEL_FAMILIES includes XGB.")

MODEL_RANDOM_SEEDS = [101, 111, 121]

MODEL_MIN_TRAIN_MONTHS = 96
MODEL_VALIDATION_MONTHS = 36
MODEL_EVAL_THRESHOLD = 0.50

TCOST_ONE_WAY = 0.0005

MV_TC_GAMMA = 3.0
MV_TC_TAU = 1.0
MV_TC_SHRINK_K = 36.0

MV_TC_WEIGHT_STEP = 0.01
MV_TC_MAX_EQUITY_WEIGHT = 1.0
MV_TC_MAX_BOND_WEIGHT = 1.0
MV_TC_MAX_TOTAL_RISKY_WEIGHT = 1.0
MV_TC_STARTING_WEALTH = 100.0

ALLOC_FEATURE_ORDER = ["drawdown", "risk", "downside"]

LOGIT_GRID = [
    {"family": "LOGIT", "name": "logit_l2_C0p03_balanced", "C": 0.03, "class_weight": "balanced"},
    {"family": "LOGIT", "name": "logit_l2_C0p05_balanced", "C": 0.05, "class_weight": "balanced"},
    {"family": "LOGIT", "name": "logit_l2_C0p10_balanced", "C": 0.10, "class_weight": "balanced"},
    {"family": "LOGIT", "name": "logit_l2_C0p25_balanced", "C": 0.25, "class_weight": "balanced"},
    {"family": "LOGIT", "name": "logit_l2_C0p50_balanced", "C": 0.50, "class_weight": "balanced"},
    {"family": "LOGIT", "name": "logit_l2_C1p00_balanced", "C": 1.00, "class_weight": "balanced"},
]

XGB_GRID = [
    {"family": "XGB", "name": "xgb_d1_lr0p03_lam10", "max_depth": 1, "learning_rate": 0.03, "n_estimators": 200, "reg_lambda": 10.0, "subsample": 0.85, "colsample_bytree": 0.85, "min_child_weight": 5.0},
    {"family": "XGB", "name": "xgb_d1_lr0p05_lam10", "max_depth": 1, "learning_rate": 0.05, "n_estimators": 200, "reg_lambda": 10.0, "subsample": 0.85, "colsample_bytree": 0.85, "min_child_weight": 5.0},
    {"family": "XGB", "name": "xgb_d2_lr0p03_lam10", "max_depth": 2, "learning_rate": 0.03, "n_estimators": 200, "reg_lambda": 10.0, "subsample": 0.85, "colsample_bytree": 0.85, "min_child_weight": 5.0},
    {"family": "XGB", "name": "xgb_d2_lr0p05_lam10", "max_depth": 2, "learning_rate": 0.05, "n_estimators": 200, "reg_lambda": 10.0, "subsample": 0.85, "colsample_bytree": 0.85, "min_child_weight": 5.0},
    {"family": "XGB", "name": "xgb_d2_lr0p03_lam30", "max_depth": 2, "learning_rate": 0.03, "n_estimators": 200, "reg_lambda": 30.0, "subsample": 0.85, "colsample_bytree": 0.85, "min_child_weight": 5.0},
    {"family": "XGB", "name": "xgb_d2_lr0p05_lam30", "max_depth": 2, "learning_rate": 0.05, "n_estimators": 200, "reg_lambda": 30.0, "subsample": 0.85, "colsample_bytree": 0.85, "min_child_weight": 5.0},
]

RF_GRID = [
    {"family": "RF", "name": "rf_d2_leaf10", "n_estimators": 500, "max_depth": 2, "min_samples_leaf": 10, "max_features": "sqrt", "class_weight": "balanced_subsample"},
    {"family": "RF", "name": "rf_d2_leaf20", "n_estimators": 500, "max_depth": 2, "min_samples_leaf": 20, "max_features": "sqrt", "class_weight": "balanced_subsample"},
    {"family": "RF", "name": "rf_d3_leaf10", "n_estimators": 500, "max_depth": 3, "min_samples_leaf": 10, "max_features": "sqrt", "class_weight": "balanced_subsample"},
    {"family": "RF", "name": "rf_d3_leaf20", "n_estimators": 500, "max_depth": 3, "min_samples_leaf": 20, "max_features": "sqrt", "class_weight": "balanced_subsample"},
    {"family": "RF", "name": "rf_d4_leaf20", "n_estimators": 500, "max_depth": 4, "min_samples_leaf": 20, "max_features": "sqrt", "class_weight": "balanced_subsample"},
]

GRID_BY_FAMILY = {
    "LOGIT": LOGIT_GRID,
    "XGB": XGB_GRID if XGBOOST_AVAILABLE else [],
    "RF": RF_GRID,
}

LINE_COLORS = {
    "60/40": "#4d4d4d",
    "LOGIT": "#1f77b4",
    "XGB": "#d62728",
    "RF": "#2ca02c",
}

STATE_COLORS = {
    "both_good": "#2ca25f",
    "equity_good_only": "#3182bd",
    "bond_good_only": "#fdae6b",
    "both_bad": "#de2d26",
    "missing": "#bdbdbd",
}


# =============================================================================
# 2. Input objects
# =============================================================================

def title(x):
    print("\n" + "=" * 160)
    print(x)
    print("=" * 160)


if "BASE_DIR" not in globals():
    raise NameError("BASE_DIR missing. Run Cell 1 first or define BASE_DIR before student prediction stage.")

BASE_DIR = Path(BASE_DIR)

STUDENT_PREDICTION_INPUT_DIR = BASE_DIR / "student_prediction_inputs"

if "majority_vote_target_wide_96" not in globals():
    p = STUDENT_PREDICTION_INPUT_DIR / "majority_vote_target_wide_96.pkl"
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}")
    majority_vote_target_wide_96 = pd.read_pickle(p)

if "majority_vote_target_wide_120" not in globals():
    p = STUDENT_PREDICTION_INPUT_DIR / "majority_vote_target_wide_120.pkl"
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}")
    majority_vote_target_wide_120 = pd.read_pickle(p)

if "majority_vote_specs_96" not in globals():
    p = STUDENT_PREDICTION_INPUT_DIR / "majority_vote_specs_96.pkl"
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}")
    majority_vote_specs_96 = pd.read_pickle(p)

if "majority_vote_specs_120" not in globals():
    p = STUDENT_PREDICTION_INPUT_DIR / "majority_vote_specs_120.pkl"
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}")
    majority_vote_specs_120 = pd.read_pickle(p)

majority_vote_target_wide_by_minobs = {
    96: majority_vote_target_wide_96.copy(),
    120: majority_vote_target_wide_120.copy(),
}

majority_vote_specs_by_minobs = {
    96: majority_vote_specs_96.copy(),
    120: majority_vote_specs_120.copy(),
}

if "student_ensemble_feature_panel" not in globals():
    p = BASE_DIR / "student_ensemble_feature_panel.csv"
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}")
    student_ensemble_feature_panel = pd.read_csv(p)

if "student_ensemble_feature_panel_wo_credit" not in globals():
    p = BASE_DIR / "student_ensemble_feature_panel_wo_credit.csv"
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}")
    student_ensemble_feature_panel_wo_credit = pd.read_csv(p)

student_panel_by_name = {
    "student_with_credit": student_ensemble_feature_panel.copy(),
    "student_wo_credit": student_ensemble_feature_panel_wo_credit.copy(),
}

for k in student_panel_by_name:
    student_panel_by_name[k]["date"] = pd.to_datetime(student_panel_by_name[k]["date"])
    student_panel_by_name[k] = (
        student_panel_by_name[k]
        .sort_values("date")
        .drop_duplicates("date", keep="last")
        .reset_index(drop=True)
    )

BASE_PANEL_PATH = BASE_DIR / "base_panel_monthly_trunc_2024_11.parquet"

if not BASE_PANEL_PATH.exists():
    raise FileNotFoundError(f"Missing base return panel: {BASE_PANEL_PATH}")

base_returns = pd.read_parquet(BASE_PANEL_PATH).copy()
base_returns["date"] = pd.to_datetime(base_returns["date"])
base_returns = base_returns.sort_values("date").drop_duplicates("date", keep="last").reset_index(drop=True)

required_return_cols = ["sprtrn_sp500", "agg_ret", "rf"]
missing_return_cols = [c for c in required_return_cols if c not in base_returns.columns]

if missing_return_cols:
    raise KeyError(f"Base panel missing return columns: {missing_return_cols}")

for c in required_return_cols:
    base_returns[c] = pd.to_numeric(base_returns[c], errors="coerce")

returns = base_returns[["date", "sprtrn_sp500", "agg_ret", "rf"]].copy()
returns = returns.dropna(subset=["sprtrn_sp500", "agg_ret", "rf"]).reset_index(drop=True)
returns["equity_excess"] = returns["sprtrn_sp500"] - returns["rf"]
returns["bond_excess"] = returns["agg_ret"] - returns["rf"]
returns["equity_ret_h1"] = returns["equity_excess"].shift(-1)
returns["bond_ret_h1"] = returns["bond_excess"].shift(-1)

title("Student Prediction and Allocation INPUT CHECK")
print("Teacher versions:", RUN_TEACHER_MINOBS)
print("Student panels:", RUN_STUDENT_PANELS)
print("Feature transforms:", RUN_FEATURE_TRANSFORMS)
print("Model families:", RUN_MODEL_FAMILIES)
print("XGBoost available:", XGBOOST_AVAILABLE)
print("Returns shape:", returns.shape)
print("Returns date range:", returns["date"].min(), "to", returns["date"].max())

for name, df in student_panel_by_name.items():
    print(f"{name}: shape={df.shape}, date range={df['date'].min()} to {df['date'].max()}")

for m in RUN_TEACHER_MINOBS:
    tw = majority_vote_target_wide_by_minobs[m]
    sp = majority_vote_specs_by_minobs[m]
    tw["date"] = pd.to_datetime(tw["date"])
    print(f"teacher_minobs{m}: target_wide={tw.shape}, specs={sp.shape}")


# =============================================================================
# 3. Metrics and model helpers
# =============================================================================

def target_stub(target_col):
    return target_col.replace("_state_h1", "").replace("_state", "")


def safe_binary_balanced_accuracy(y_true, y_pred):
    y_true = pd.Series(y_true).astype(int).to_numpy()
    y_pred = pd.Series(y_pred).astype(int).to_numpy()

    if len(y_true) == 0:
        return np.nan

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp = cm[0, 0], cm[0, 1]
    fn, tp = cm[1, 0], cm[1, 1]

    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else np.nan

    vals = [x for x in [recall_0, recall_1] if np.isfinite(x)]

    return float(np.mean(vals)) if vals else np.nan


def safe_metrics(y_true, p_hat):
    y_true = pd.Series(y_true).astype(int).to_numpy()
    p_hat = np.asarray(p_hat, dtype=float)
    p_hat = np.clip(p_hat, EPS, 1.0 - EPS)

    out = {
        "auc": np.nan,
        "accuracy": np.nan,
        "balanced_accuracy": np.nan,
        "brier": np.nan,
        "logloss": np.nan,
    }

    if len(y_true) == 0:
        return out

    y_pred = (p_hat >= 0.5).astype(int)

    out["accuracy"] = float(accuracy_score(y_true, y_pred))
    out["balanced_accuracy"] = safe_binary_balanced_accuracy(y_true, y_pred)
    out["brier"] = float(brier_score_loss(y_true, p_hat))

    if len(np.unique(y_true)) == 2:
        out["auc"] = float(roc_auc_score(y_true, p_hat))
        out["logloss"] = float(log_loss(y_true, p_hat, labels=[0, 1]))

    return out


def ann_return(x):
    x = pd.Series(x).dropna().astype(float)

    return float(12.0 * x.mean()) if len(x) else np.nan


def ann_vol(x):
    x = pd.Series(x).dropna().astype(float)

    if len(x) < 2:
        return np.nan

    sd = x.std(ddof=1)

    return float(np.sqrt(12.0) * sd) if np.isfinite(sd) else np.nan


def ann_sharpe(x):
    x = pd.Series(x).dropna().astype(float)

    if len(x) < 2:
        return np.nan

    sd = x.std(ddof=1)

    if not np.isfinite(sd) or sd <= EPS:
        return np.nan

    return float(np.sqrt(12.0) * x.mean() / sd)


def max_drawdown_from_returns(x):
    x = pd.Series(x).dropna().astype(float)

    if len(x) == 0:
        return np.nan

    wealth = (1.0 + x).cumprod()
    peak = wealth.cummax()
    dd = wealth / peak - 1.0

    return float(dd.min())


def performance_summary(ret, name):
    r = pd.Series(ret).dropna().astype(float)

    if len(r) == 0:
        return {
            "strategy": name,
            "n_months": 0,
            "ann_return": np.nan,
            "ann_vol": np.nan,
            "sharpe": np.nan,
            "max_drawdown": np.nan,
        }

    wealth = (1.0 + r).cumprod()
    dd = wealth / wealth.cummax() - 1.0

    out_ann_return = 12.0 * r.mean()
    out_ann_vol = np.sqrt(12.0) * r.std(ddof=1) if len(r) >= 2 else np.nan
    out_sharpe = out_ann_return / out_ann_vol if np.isfinite(out_ann_vol) and out_ann_vol > EPS else np.nan

    return {
        "strategy": name,
        "n_months": int(len(r)),
        "ann_return": float(out_ann_return),
        "ann_vol": float(out_ann_vol),
        "sharpe": float(out_sharpe),
        "max_drawdown": float(dd.min()),
    }


def make_good_state_signal(p_hat, threshold=0.5):
    return (np.asarray(p_hat, dtype=float) >= threshold).astype(float)


def evaluate_simple_timing_rule(asset_ret_h1, p_hat, threshold=0.5):
    x = pd.DataFrame({
        "asset_ret_h1": pd.Series(asset_ret_h1).astype(float).to_numpy(),
        "p_hat": np.asarray(p_hat, dtype=float),
    }).replace([np.inf, -np.inf], np.nan).dropna()

    out = {
        "n_econ_obs": int(len(x)),
        "signal_share": np.nan,
        "turnover": np.nan,
        "net_sharpe": np.nan,
        "buyhold_sharpe": np.nan,
        "drawdown_improvement_vs_buyhold": np.nan,
        "vol_reduction_vs_buyhold": np.nan,
    }

    if len(x) < 2:
        return out

    signal = pd.Series(make_good_state_signal(x["p_hat"], threshold=threshold), index=x.index).astype(float)

    turnover = signal.diff().abs()
    turnover.iloc[0] = signal.iloc[0]
    turnover = turnover.fillna(0.0)

    net = signal * x["asset_ret_h1"] - TCOST_ONE_WAY * turnover
    buyhold = x["asset_ret_h1"]

    net_sharpe = ann_sharpe(net)
    buyhold_sharpe = ann_sharpe(buyhold)
    net_mdd = max_drawdown_from_returns(net)
    buyhold_mdd = max_drawdown_from_returns(buyhold)
    net_vol = ann_vol(net)
    buyhold_vol = ann_vol(buyhold)

    out["signal_share"] = float(signal.mean())
    out["turnover"] = float(turnover.mean())
    out["net_sharpe"] = net_sharpe
    out["buyhold_sharpe"] = buyhold_sharpe

    if np.isfinite(net_mdd) and np.isfinite(buyhold_mdd):
        out["drawdown_improvement_vs_buyhold"] = net_mdd - buyhold_mdd

    if np.isfinite(net_vol) and np.isfinite(buyhold_vol):
        out["vol_reduction_vs_buyhold"] = buyhold_vol - net_vol

    return out


def build_model(params, seed, y_train=None):
    family = params["family"]

    if family == "LOGIT":
        model = LogisticRegression(
            penalty="l2",
            C=params["C"],
            class_weight=params["class_weight"],
            solver="lbfgs",
            max_iter=3000,
            random_state=seed,
        )

        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", model),
        ])

    if family == "XGB":
        if not XGBOOST_AVAILABLE:
            raise ImportError("xgboost is not available.")

        y_arr = pd.Series(y_train).astype(int).to_numpy()
        n_pos = int((y_arr == 1).sum())
        n_neg = int((y_arr == 0).sum())
        scale_pos_weight = n_neg / max(n_pos, 1)

        model = XGBClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            learning_rate=params["learning_rate"],
            reg_lambda=params["reg_lambda"],
            subsample=params["subsample"],
            colsample_bytree=params["colsample_bytree"],
            min_child_weight=params["min_child_weight"],
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=seed,
            n_jobs=1,
            scale_pos_weight=scale_pos_weight,
            verbosity=0,
        )

        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", model),
        ])

    if family == "RF":
        model = RandomForestClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            min_samples_leaf=params["min_samples_leaf"],
            max_features=params["max_features"],
            class_weight=params["class_weight"],
            random_state=seed,
            n_jobs=1,
            bootstrap=True,
        )

        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", model),
        ])

    raise ValueError(f"Unknown model family: {family}")


def realized_class_proba(pipe, X):
    model = pipe.named_steps["model"]
    probs = pipe.predict_proba(X)
    classes = list(model.classes_)

    if 1 in classes:
        return probs[:, classes.index(1)]

    return np.zeros(len(X), dtype=float)


def choose_best_within_family(grid_df):
    x = grid_df.copy()

    x["val_auc_rank"] = x["val_auc"].fillna(-999.0)
    x["val_balacc_rank"] = x["val_balanced_accuracy"].fillna(-999.0)
    x["val_brier_rank"] = x["val_brier"].fillna(999.0)
    x["val_net_sharpe_rank"] = x["val_net_sharpe"].fillna(-999.0)
    x["val_dd_improvement_rank"] = x["val_drawdown_improvement_vs_buyhold"].fillna(-999.0)
    x["val_vol_reduction_rank"] = x["val_vol_reduction_vs_buyhold"].fillna(-999.0)
    x["val_turnover_rank"] = x["val_turnover"].fillna(999.0)

    sort_cols = [
        "val_auc_rank",
        "val_balacc_rank",
        "val_brier_rank",
        "val_net_sharpe_rank",
        "val_dd_improvement_rank",
        "val_vol_reduction_rank",
        "val_turnover_rank",
        "model_name",
    ]

    ascending = [False, False, True, False, False, False, True, True]

    x = x.sort_values(sort_cols, ascending=ascending)

    return str(x.iloc[0]["model_name"])


def extract_feature_importance(pipe, feature_cols, family):
    model = pipe.named_steps["model"]

    if family == "LOGIT" and hasattr(model, "coef_"):
        imp = np.abs(model.coef_[0])
    elif family in {"XGB", "RF"} and hasattr(model, "feature_importances_"):
        imp = model.feature_importances_
    else:
        imp = np.zeros(len(feature_cols), dtype=float)

    imp = np.asarray(imp, dtype=float)

    if len(imp) != len(feature_cols):
        imp = np.zeros(len(feature_cols), dtype=float)

    return imp


# =============================================================================
# 4. Scenario construction
# =============================================================================

def feature_cols_for_transform(df, transform_name):
    all_cols = [c for c in df.columns if c != "date"]

    if transform_name == "z":
        cols = [
            c for c in all_cols
            if c.endswith("_z") and not c.endswith("_z_smooth")
        ]
    elif transform_name == "z_smooth":
        cols = [
            c for c in all_cols
            if c.endswith("_z_smooth")
        ]
    else:
        raise ValueError(f"Unknown transform_name: {transform_name}")

    return cols


def build_target_meta(specs):
    meta = {}

    for _, r in specs.iterrows():
        state_h1 = r["state_col"] + "_h1"
        asset = r["asset"]
        feature_group = r["feature_group"]

        if feature_group == "SHORT_TAIL_VOL":
            target_type = "risk"
        elif feature_group == "EWM_RETURN_RISK_DOWNSIDE":
            target_type = "downside"
        elif feature_group == "EWM_RETURN_RISK_DRAWDOWN":
            target_type = "drawdown"
        else:
            target_type = "other"

        meta[state_h1] = {
            "asset": asset,
            "feature_group": feature_group,
            "role": str(r.get("role", state_h1)),
            "target_type": target_type,
            "good_state": int(r["good_state"]),
            "target_design": str(r.get("target_design", "all_valid_sjm_majority_vote")),
            "state_col": r["state_col"],
            "prob_col": r["prob_col"],
        }

    return meta


def build_prediction_base(student_df, target_wide, specs, transform_name):
    student = student_df.copy()
    student["date"] = pd.to_datetime(student["date"])
    student = student.sort_values("date").drop_duplicates("date", keep="last").reset_index(drop=True)

    target = target_wide.copy()
    target["date"] = pd.to_datetime(target["date"])
    target = target.sort_values("date").drop_duplicates("date", keep="last").reset_index(drop=True)

    feature_cols = feature_cols_for_transform(student, transform_name)

    if len(feature_cols) == 0:
        raise ValueError(f"No feature columns found for transform={transform_name}")

    specs = specs.copy()

    target_state_cols = [c + "_h1" for c in specs["state_col"].tolist()]
    target_prob_cols = [c + "_h1" for c in specs["prob_col"].tolist()]

    missing_targets = [c for c in target_state_cols if c not in target.columns]

    if missing_targets:
        raise KeyError(f"Target panel missing H1 columns: {missing_targets}")

    merged = (
        student
        .merge(target, on="date", how="inner")
        .merge(
            returns[["date", "equity_excess", "bond_excess", "equity_ret_h1", "bond_ret_h1"]],
            on="date",
            how="left",
        )
        .sort_values("date")
        .reset_index(drop=True)
    )

    forbidden = set(target.columns)
    forbidden |= set(target_state_cols)
    forbidden |= set(target_prob_cols)

    feature_cols = [
        c for c in feature_cols
        if c in merged.columns
        and c not in forbidden
        and not c.endswith("_state")
        and not c.endswith("_state_h1")
        and not c.endswith("_prob")
        and not c.endswith("_prob_h1")
    ]

    if len(feature_cols) == 0:
        raise ValueError("Feature column list empty after target-exclusion filter.")

    merged = merged.dropna(subset=target_state_cols, how="all").copy()
    merged = merged.loc[merged[feature_cols].notna().any(axis=1)].copy()
    merged = merged.sort_values("date").reset_index(drop=True)

    if "year" not in merged.columns:
        merged["year"] = merged["date"].dt.year

    return merged, feature_cols, target_state_cols, target_prob_cols, build_target_meta(specs)


def build_historical_label_return_panel(prediction_base, specs):
    hist_map = {
        "risk": {"eq_y": "eq_risk_state_h1", "bd_y": "bd_risk_state_h1"},
        "downside": {"eq_y": "eq_downside_state_h1", "bd_y": "bd_downside_state_h1"},
        "drawdown": {"eq_y": "eq_drawdown_state_h1", "bd_y": "bd_drawdown_state_h1"},
    }

    required_targets = sorted({v for d in hist_map.values() for v in d.values()})
    missing = [c for c in required_targets if c not in prediction_base.columns]

    if missing:
        raise KeyError(f"Prediction base missing historical target columns: {missing}")

    out = (
        prediction_base[["date"] + required_targets]
        .copy()
        .replace([np.inf, -np.inf], np.nan)
    )

    out["date"] = pd.to_datetime(out["date"])

    out = (
        out
        .sort_values("date")
        .drop_duplicates("date", keep="last")
        .merge(
            returns[["date", "equity_ret_h1", "bond_ret_h1"]],
            on="date",
            how="left",
        )
        .sort_values("date")
        .reset_index(drop=True)
    )

    return out, hist_map


# =============================================================================
# 5. Student model estimation
# =============================================================================

def run_one_family_prediction(
    scenario_id,
    df,
    feature_cols,
    target_cols,
    target_meta,
    family,
):
    active_grid = GRID_BY_FAMILY[family]

    if len(active_grid) == 0:
        raise RuntimeError(f"No grid entries for family={family}")

    unique_dates = df["date"].sort_values().drop_duplicates().reset_index(drop=True)
    first_test_idx = MODEL_MIN_TRAIN_MONTHS + MODEL_VALIDATION_MONTHS

    if len(unique_dates) <= first_test_idx:
        raise ValueError(
            f"{scenario_id}: not enough observations for train + validation windows. "
            f"n_dates={len(unique_dates)}, required>{first_test_idx}"
        )

    test_dates = unique_dates.iloc[first_test_idx:].tolist()

    result_rows = []
    selected_rows = []
    prediction_parts = []
    importance_summary_rows = []

    title(f"{scenario_id} | FAMILY START: {family}")
    print("Rows:", len(df))
    print("Date range:", df["date"].min(), "to", df["date"].max())
    print("Features:", len(feature_cols))
    print("Targets:", len(target_cols))
    print("First OOS date:", min(test_dates))
    print("Last OOS date:", max(test_dates))
    print("Grid size:", len(active_grid))
    print("Seeds:", MODEL_RANDOM_SEEDS)

    t_family = time.time()

    for target_col in target_cols:
        meta = target_meta[target_col]
        asset = meta["asset"]
        feature_group = meta["feature_group"]
        target_type = meta["target_type"]
        ret_h1_col = "equity_ret_h1" if asset == "equity" else "bond_ret_h1"

        print(f"{family} | {target_col}")

        target_pred_parts = []
        target_importance_accumulator = []

        for test_date in test_dates:
            test_idx = int(unique_dates[unique_dates.eq(test_date)].index[0])
            train_end_idx = test_idx - MODEL_VALIDATION_MONTHS

            train_dates = unique_dates.iloc[:train_end_idx]
            val_dates = unique_dates.iloc[train_end_idx:test_idx]

            train_mask = df["date"].isin(train_dates)
            val_mask = df["date"].isin(val_dates)
            test_mask = df["date"].eq(test_date)

            work_cols = ["date", "year", ret_h1_col] + feature_cols + [target_col]

            train = df.loc[train_mask, work_cols].dropna(subset=[target_col]).copy()
            val = df.loc[val_mask, work_cols].dropna(subset=[target_col]).copy()
            test = df.loc[test_mask, work_cols].dropna(subset=[target_col]).copy()

            if len(train) < MODEL_MIN_TRAIN_MONTHS or len(val) < MODEL_VALIDATION_MONTHS or test.empty:
                continue

            y_train = train[target_col].astype(int)
            y_val = val[target_col].astype(int)
            y_test = test[target_col].astype(int)

            if y_train.nunique() < 2:
                continue

            X_train = train[feature_cols]
            X_val = val[feature_cols]
            X_test = test[feature_cols]

            grid_rows_date = []

            for params in active_grid:
                val_seed_preds = []
                fit_errors = []

                for seed in MODEL_RANDOM_SEEDS:
                    try:
                        pipe = build_model(params, seed, y_train=y_train)
                        pipe.fit(X_train, y_train)
                        val_seed_preds.append(realized_class_proba(pipe, X_val))
                    except Exception as e:
                        fit_errors.append(str(e))

                if len(val_seed_preds) == 0:
                    p_val = np.full(len(y_val), np.nan)
                    val_metrics = {
                        "auc": np.nan,
                        "accuracy": np.nan,
                        "balanced_accuracy": np.nan,
                        "brier": np.nan,
                        "logloss": np.nan,
                    }
                else:
                    p_val = np.mean(np.column_stack(val_seed_preds), axis=1)
                    val_metrics = safe_metrics(y_val, p_val)

                econ = evaluate_simple_timing_rule(
                    asset_ret_h1=val[ret_h1_col].to_numpy(dtype=float),
                    p_hat=p_val,
                    threshold=MODEL_EVAL_THRESHOLD,
                )

                grid_row = {
                    "scenario_id": scenario_id,
                    "family": family,
                    "target": target_col,
                    "asset": asset,
                    "feature_group": feature_group,
                    "role": meta["role"],
                    "target_type": target_type,
                    "target_design": meta.get("target_design", ""),
                    "test_date": test_date,
                    "model_name": params["name"],
                    "threshold": float(MODEL_EVAL_THRESHOLD),
                    "n_train": int(len(train)),
                    "n_validation": int(len(val)),
                    "n_test": int(len(test)),
                    "train_positive_share": float(y_train.mean()),
                    "validation_positive_share": float(y_val.mean()),
                    "test_positive_share": float(y_test.mean()),
                    "val_auc": val_metrics["auc"],
                    "val_accuracy": val_metrics["accuracy"],
                    "val_balanced_accuracy": val_metrics["balanced_accuracy"],
                    "val_brier": val_metrics["brier"],
                    "val_logloss": val_metrics["logloss"],
                    "val_n_econ_obs": econ["n_econ_obs"],
                    "val_signal_share": econ["signal_share"],
                    "val_turnover": econ["turnover"],
                    "val_net_sharpe": econ["net_sharpe"],
                    "val_buyhold_sharpe": econ["buyhold_sharpe"],
                    "val_drawdown_improvement_vs_buyhold": econ["drawdown_improvement_vs_buyhold"],
                    "val_vol_reduction_vs_buyhold": econ["vol_reduction_vs_buyhold"],
                    "fit_error_count": int(len(fit_errors)),
                    "fit_error_first": fit_errors[0] if fit_errors else "",
                }

                for k, v in params.items():
                    if k not in grid_row:
                        grid_row[k] = v

                grid_rows_date.append(grid_row)

            grid_df_date = pd.DataFrame(grid_rows_date)

            if grid_df_date.empty:
                continue

            best_name = choose_best_within_family(grid_df_date)
            best_params = next(p for p in active_grid if p["name"] == best_name)

            trainval = (
                pd.concat([train, val], axis=0)
                .sort_values("date")
                .reset_index(drop=True)
            )

            X_trainval = trainval[feature_cols]
            y_trainval = trainval[target_col].astype(int)

            if y_trainval.nunique() < 2:
                continue

            test_seed_preds = []
            seed_importances = []

            for seed in MODEL_RANDOM_SEEDS:
                pipe = build_model(best_params, seed, y_train=y_trainval)
                pipe.fit(X_trainval, y_trainval)
                test_seed_preds.append(realized_class_proba(pipe, X_test))
                seed_importances.append(extract_feature_importance(pipe, feature_cols, family))

            p_test = np.mean(np.column_stack(test_seed_preds), axis=1)
            test_metrics = safe_metrics(y_test, p_test)
            test_signal = make_good_state_signal(p_test, threshold=MODEL_EVAL_THRESHOLD)

            for _, r in grid_df_date.iterrows():
                row = r.to_dict()
                is_selected = row["model_name"] == best_name

                row["selected_model"] = bool(is_selected)
                row["test_auc"] = test_metrics["auc"] if is_selected else np.nan
                row["test_accuracy"] = test_metrics["accuracy"] if is_selected else np.nan
                row["test_balanced_accuracy"] = test_metrics["balanced_accuracy"] if is_selected else np.nan
                row["test_brier"] = test_metrics["brier"] if is_selected else np.nan
                row["test_logloss"] = test_metrics["logloss"] if is_selected else np.nan

                result_rows.append(row)

                if is_selected:
                    selected_rows.append(row)

            stub = target_stub(target_col)

            pred_part = test[["date", "year", target_col, ret_h1_col]].copy()
            pred_part = pred_part.rename(columns={target_col: "y"})
            pred_part["p_hat"] = p_test
            pred_part["signal"] = test_signal
            pred_part["threshold_fixed"] = MODEL_EVAL_THRESHOLD
            pred_part["scenario_id"] = scenario_id
            pred_part["family"] = family
            pred_part["selected_model"] = best_name
            pred_part["target"] = target_col
            pred_part["target_stub"] = stub
            pred_part["asset"] = asset
            pred_part["target_type"] = target_type
            pred_part["target_design"] = meta.get("target_design", "")
            pred_part["n_trainval"] = int(len(trainval))
            pred_part["train_start"] = trainval["date"].min()
            pred_part["train_end"] = trainval["date"].max()

            target_pred_parts.append(pred_part)

            imp_mean = np.mean(np.vstack(seed_importances), axis=0)
            target_importance_accumulator.append(imp_mean)

        if target_pred_parts:
            prediction_parts.append(pd.concat(target_pred_parts, axis=0, ignore_index=True))

        if target_importance_accumulator:
            imp_avg = np.mean(np.vstack(target_importance_accumulator), axis=0)
            imp_df = pd.DataFrame({
                "scenario_id": scenario_id,
                "family": family,
                "target": target_col,
                "feature": feature_cols,
                "mean_importance": imp_avg,
            }).sort_values("mean_importance", ascending=False).reset_index(drop=True)
            imp_df["rank"] = np.arange(1, len(imp_df) + 1)
            importance_summary_rows.append(imp_df.head(50))

    result_table = pd.DataFrame(result_rows)
    selected_model_table = pd.DataFrame(selected_rows)

    prediction_long = (
        pd.concat(prediction_parts, axis=0, ignore_index=True)
        .sort_values(["scenario_id", "family", "target", "date"])
        .reset_index(drop=True)
        if prediction_parts else pd.DataFrame()
    )

    feature_importance_top = (
        pd.concat(importance_summary_rows, axis=0, ignore_index=True)
        if importance_summary_rows else pd.DataFrame()
    )

    title(f"{scenario_id} | FAMILY COMPLETE: {family}")
    print("Elapsed seconds:", round(time.time() - t_family, 2))
    print("result_table shape:", result_table.shape)
    print("selected_model_table shape:", selected_model_table.shape)
    print("prediction_long shape:", prediction_long.shape)
    print("feature_importance_top shape:", feature_importance_top.shape)

    return {
        "scenario_id": scenario_id,
        "family": family,
        "result_table": result_table,
        "selected_model_table": selected_model_table,
        "prediction_long": prediction_long,
        "feature_importance_top": feature_importance_top,
    }


def summarize_family_predictions(prediction_long, df, target_meta):
    rows = []

    for (scenario_id, family, target_col), g in prediction_long.groupby(["scenario_id", "family", "target"]):
        meta = target_meta[target_col]
        asset = meta["asset"]
        ret_h1_col = "equity_ret_h1" if asset == "equity" else "bond_ret_h1"

        tmp = (
            g[["date", "y", "p_hat", "signal"]]
            .merge(df[["date", ret_h1_col]].drop_duplicates("date"), on="date", how="left")
            .dropna(subset=["y", "p_hat"])
            .copy()
        )

        metrics = safe_metrics(tmp["y"].astype(int), tmp["p_hat"].astype(float))

        signal = tmp["signal"].astype(float)
        turnover = signal.diff().abs()

        if len(turnover) > 0:
            turnover.iloc[0] = signal.iloc[0]

        turnover = turnover.fillna(0.0)

        gross = signal * tmp[ret_h1_col].astype(float)
        net = gross - TCOST_ONE_WAY * turnover
        buyhold = tmp[ret_h1_col].astype(float)

        net_sharpe = ann_sharpe(net)
        buyhold_sharpe = ann_sharpe(buyhold)
        net_mdd = max_drawdown_from_returns(net)
        buyhold_mdd = max_drawdown_from_returns(buyhold)
        buyhold_vol = ann_vol(buyhold)
        net_vol = ann_vol(net)

        rows.append({
            "scenario_id": scenario_id,
            "family": family,
            "target": target_col,
            "asset": asset,
            "target_type": meta["target_type"],
            "target_design": meta.get("target_design", ""),
            "n_oos_months": int(len(tmp)),
            "first_oos": tmp["date"].min(),
            "last_oos": tmp["date"].max(),
            "positive_share": float(tmp["y"].mean()),
            "prob_mean": float(tmp["p_hat"].mean()),
            "prob_std": float(tmp["p_hat"].std(ddof=1)),
            "oos_auc": metrics["auc"],
            "oos_accuracy": metrics["accuracy"],
            "oos_balanced_accuracy": metrics["balanced_accuracy"],
            "oos_brier": metrics["brier"],
            "oos_logloss": metrics["logloss"],
            "oos_signal_share": float(signal.mean()),
            "oos_turnover": float(turnover.mean()),
            "oos_net_ann_return": ann_return(net),
            "oos_net_ann_vol": net_vol,
            "oos_net_sharpe": net_sharpe,
            "oos_net_max_drawdown": net_mdd,
            "oos_buyhold_ann_return": ann_return(buyhold),
            "oos_buyhold_ann_vol": buyhold_vol,
            "oos_buyhold_sharpe": buyhold_sharpe,
            "oos_buyhold_max_drawdown": buyhold_mdd,
            "oos_sharpe_lift_vs_buyhold": net_sharpe - buyhold_sharpe if np.isfinite(net_sharpe) and np.isfinite(buyhold_sharpe) else np.nan,
            "oos_drawdown_improvement_vs_buyhold": net_mdd - buyhold_mdd if np.isfinite(net_mdd) and np.isfinite(buyhold_mdd) else np.nan,
            "oos_vol_reduction_vs_buyhold": buyhold_vol - net_vol if np.isfinite(buyhold_vol) and np.isfinite(net_vol) else np.nan,
        })

    return (
        pd.DataFrame(rows)
        .sort_values(["scenario_id", "target", "family"])
        .reset_index(drop=True)
    )


def build_family_wide_panel(prediction_long, target_cols):
    if prediction_long.empty:
        return pd.DataFrame()

    scenario_id = str(prediction_long["scenario_id"].iloc[0])
    family = str(prediction_long["family"].iloc[0])

    base_dates = (
        prediction_long[["date", "year"]]
        .drop_duplicates()
        .sort_values("date")
        .reset_index(drop=True)
    )

    wide = base_dates.copy()

    for target_col in target_cols:
        stub = target_stub(target_col)

        this = prediction_long[prediction_long["target"].eq(target_col)].copy()

        if this.empty:
            continue

        this = this[["date", "y", "p_hat", "signal", "selected_model", "threshold_fixed"]].copy()

        this = this.rename(columns={
            "y": "y_" + stub,
            "p_hat": "p_" + stub,
            "signal": "signal_" + stub,
            "selected_model": "model_" + stub,
            "threshold_fixed": "threshold_" + stub,
        })

        wide = wide.merge(this, on="date", how="left")

        p_col = "p_" + stub
        pred_col = "pred_" + stub

        if p_col in wide.columns:
            wide[pred_col] = (wide[p_col] >= 0.5).astype(float)
            wide.loc[wide[p_col].isna(), pred_col] = np.nan

    wide["scenario_id"] = scenario_id
    wide["family"] = family

    return wide


# =============================================================================
# 6. MV-TC allocator
# =============================================================================

def shrink_state_moments_36m(past, y_col, ret_col, initial_mu, initial_var):
    x = (
        past[[y_col, ret_col]]
        .copy()
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=[y_col, ret_col])
    )

    if x.empty:
        x[y_col] = pd.Series(dtype=int)
    else:
        x[y_col] = x[y_col].astype(int)

    out = {}

    for state in [0, 1]:
        r = x.loc[x[y_col].eq(state), ret_col].astype(float).dropna()
        n = int(len(r))

        mu_raw = float(r.mean()) if n > 0 else float(initial_mu)
        var_raw = float(r.var(ddof=1)) if n >= 2 else float(initial_var)

        w = n / (n + MV_TC_SHRINK_K) if (n + MV_TC_SHRINK_K) > 0 else 0.0

        mu = w * mu_raw + (1.0 - w) * float(initial_mu)
        var = w * var_raw + (1.0 - w) * float(initial_var)

        out[state] = {
            "n": n,
            "shrink_k": float(MV_TC_SHRINK_K),
            "weight": float(w),
            "mu_raw": float(mu_raw),
            "var_raw": float(var_raw),
            "mu": float(mu),
            "var": float(max(var, EPS)),
        }

    return out


def expanding_corr_36m(past, initial_rho):
    x = (
        past[["equity_ret_h1", "bond_ret_h1"]]
        .copy()
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    n = int(len(x))

    if n >= 3:
        rho_emp = float(x["equity_ret_h1"].corr(x["bond_ret_h1"]))
        if not np.isfinite(rho_emp):
            rho_emp = float(initial_rho)
        rho_source = "expanding_empirical"
    else:
        rho_emp = float(initial_rho)
        rho_source = "initial"

    w = n / (n + MV_TC_SHRINK_K) if (n + MV_TC_SHRINK_K) > 0 else 0.0
    rho = w * rho_emp + (1.0 - w) * float(initial_rho)
    rho = float(np.clip(rho, -0.999, 0.999))

    return rho, rho_source, n, float(w), rho_emp


def probability_mixture_moments(p1, moments):
    p1 = float(np.clip(p1, 0.0, 1.0))
    p0 = 1.0 - p1

    mu0 = moments[0]["mu"]
    mu1 = moments[1]["mu"]
    var0 = moments[0]["var"]
    var1 = moments[1]["var"]

    mu = p0 * mu0 + p1 * mu1
    second = p0 * (var0 + mu0 ** 2) + p1 * (var1 + mu1 ** 2)
    var = max(second - mu ** 2, 1e-6)

    return float(mu), float(var)


def solve_mv_tc_grid(mu_vec, cov_mat, prev_w_eq, prev_w_bd):
    best_obj = -np.inf
    best_w = np.array([0.0, 0.0], dtype=float)

    grid = np.arange(0.0, 1.0 + 0.5 * MV_TC_WEIGHT_STEP, MV_TC_WEIGHT_STEP)

    for w_eq in grid:
        if w_eq > MV_TC_MAX_EQUITY_WEIGHT + EPS:
            continue

        max_bd = min(
            MV_TC_MAX_BOND_WEIGHT,
            MV_TC_MAX_TOTAL_RISKY_WEIGHT - w_eq,
        )

        for w_bd in grid:
            if w_bd > max_bd + EPS:
                continue

            w = np.array([w_eq, w_bd], dtype=float)
            turnover = abs(w_eq - prev_w_eq) + abs(w_bd - prev_w_bd)

            ret_term = float(w @ mu_vec)
            risk_term = float(w @ cov_mat @ w)
            trade_penalty = MV_TC_TAU * TCOST_ONE_WAY * turnover

            obj = ret_term - MV_TC_GAMMA * risk_term - trade_penalty

            if obj > best_obj:
                best_obj = obj
                best_w = w.copy()

    return best_w, best_obj


def initial_moments_from_pre_oos(historical_label_return_panel, first_prediction_date):
    pre = (
        historical_label_return_panel[
            historical_label_return_panel["date"] < first_prediction_date
        ]
        .copy()
        .replace([np.inf, -np.inf], np.nan)
        .sort_values("date")
        .reset_index(drop=True)
    )

    z = pre[["equity_ret_h1", "bond_ret_h1"]].dropna().astype(float)

    if len(z) < 12:
        raise ValueError(f"Too few pre-OOS observations for initial moments: {len(z)}")

    mu_eq = float(z["equity_ret_h1"].mean())
    mu_bd = float(z["bond_ret_h1"].mean())

    var_eq = max(float(z["equity_ret_h1"].var(ddof=1)), 1e-6)
    var_bd = max(float(z["bond_ret_h1"].var(ddof=1)), 1e-6)

    cov = z[["equity_ret_h1", "bond_ret_h1"]].cov().to_numpy(dtype=float)

    if not np.all(np.isfinite(cov)):
        cov = np.array([[var_eq, 0.0], [0.0, var_bd]], dtype=float)

    cov = cov + np.eye(2) * 1e-8

    sd_eq = np.sqrt(max(cov[0, 0], 1e-8))
    sd_bd = np.sqrt(max(cov[1, 1], 1e-8))

    rho = cov[0, 1] / max(sd_eq * sd_bd, EPS)
    rho = float(np.clip(rho, -0.95, 0.95))

    return {
        "first_prediction_date": first_prediction_date,
        "sample_start": pre["date"].min(),
        "sample_end": pre["date"].max(),
        "n_obs": int(len(z)),
        "eq": {"mu": mu_eq, "var": var_eq},
        "bd": {"mu": mu_bd, "var": var_bd},
        "rho": rho,
    }


def run_allocator_for_family_feature(
    panel,
    scenario_id,
    family_name,
    feature_set,
    historical_label_return_panel,
    historical_targets,
    initial_moments,
):
    y_eq_pred = f"y_eq_{feature_set}"
    p_eq = f"p_eq_{feature_set}"

    y_bd_pred = f"y_bd_{feature_set}"
    p_bd = f"p_bd_{feature_set}"

    hist_eq_y = historical_targets[feature_set]["eq_y"]
    hist_bd_y = historical_targets[feature_set]["bd_y"]

    required_cols = [
        "date",
        "equity_ret_h1",
        "bond_ret_h1",
        y_eq_pred,
        p_eq,
        y_bd_pred,
        p_bd,
    ]

    missing_cols = [c for c in required_cols if c not in panel.columns]

    if missing_cols:
        raise KeyError(f"{scenario_id} {family_name} {feature_set}: missing columns: {missing_cols}")

    x = (
        panel[required_cols]
        .copy()
        .replace([np.inf, -np.inf], np.nan)
        .sort_values("date")
        .reset_index(drop=True)
    )

    rows = []
    prev_w_eq = 0.0
    prev_w_bd = 0.0

    for i in range(len(x)):
        row = x.iloc[i]
        date = row["date"]

        if not np.isfinite(row[p_eq]) or not np.isfinite(row[p_bd]):
            continue

        if not np.isfinite(row["equity_ret_h1"]) or not np.isfinite(row["bond_ret_h1"]):
            continue

        past_hist = (
            historical_label_return_panel[
                historical_label_return_panel["date"] < date
            ]
            .copy()
            .sort_values("date")
            .reset_index(drop=True)
        )

        past_for_eq = (
            past_hist[[hist_eq_y, "equity_ret_h1"]]
            .copy()
            .rename(columns={hist_eq_y: "eq_state"})
        )

        past_for_bd = (
            past_hist[[hist_bd_y, "bond_ret_h1"]]
            .copy()
            .rename(columns={hist_bd_y: "bd_state"})
        )

        eq_mom = shrink_state_moments_36m(
            past=past_for_eq,
            y_col="eq_state",
            ret_col="equity_ret_h1",
            initial_mu=initial_moments["eq"]["mu"],
            initial_var=initial_moments["eq"]["var"],
        )

        bd_mom = shrink_state_moments_36m(
            past=past_for_bd,
            y_col="bd_state",
            ret_col="bond_ret_h1",
            initial_mu=initial_moments["bd"]["mu"],
            initial_var=initial_moments["bd"]["var"],
        )

        mu_eq, var_eq = probability_mixture_moments(row[p_eq], eq_mom)
        mu_bd, var_bd = probability_mixture_moments(row[p_bd], bd_mom)

        rho, rho_source, rho_n, rho_w, rho_emp = expanding_corr_36m(
            past=past_hist,
            initial_rho=initial_moments["rho"],
        )

        sd_eq = np.sqrt(max(var_eq, 1e-8))
        sd_bd = np.sqrt(max(var_bd, 1e-8))

        cov = np.array(
            [
                [sd_eq ** 2, rho * sd_eq * sd_bd],
                [rho * sd_eq * sd_bd, sd_bd ** 2],
            ],
            dtype=float,
        )

        mu_vec = np.array([mu_eq, mu_bd], dtype=float)

        w, obj = solve_mv_tc_grid(
            mu_vec=mu_vec,
            cov_mat=cov,
            prev_w_eq=prev_w_eq,
            prev_w_bd=prev_w_bd,
        )

        w_eq = float(w[0])
        w_bd = float(w[1])
        w_cash = float(1.0 - w_eq - w_bd)

        turnover = abs(w_eq - prev_w_eq) + abs(w_bd - prev_w_bd)
        tcost = TCOST_ONE_WAY * turnover

        gross_ret = (
            w_eq * float(row["equity_ret_h1"])
            + w_bd * float(row["bond_ret_h1"])
        )

        net_ret = gross_ret - tcost

        rows.append({
            "scenario_id": scenario_id,
            "date": date,
            "family": family_name,
            "feature_set": feature_set,
            "strategy_name": f"{scenario_id}__{family_name}_{feature_set}",

            "p_eq": float(row[p_eq]),
            "p_bd": float(row[p_bd]),

            "mu_eq": float(mu_eq),
            "mu_bd": float(mu_bd),
            "sd_eq": float(sd_eq),
            "sd_bd": float(sd_bd),
            "rho": float(rho),

            "w_eq": w_eq,
            "w_bd": w_bd,
            "w_cash": w_cash,
            "turnover": float(turnover),
            "tcost": float(tcost),

            "equity_ret_h1": float(row["equity_ret_h1"]),
            "bond_ret_h1": float(row["bond_ret_h1"]),
            "strategy_gross_ret": float(gross_ret),
            "strategy_net_ret": float(net_ret),
            "objective": float(obj),

            "gamma": MV_TC_GAMMA,
            "tau": MV_TC_TAU,
            "tcost_one_way": TCOST_ONE_WAY,
            "shrink_k": MV_TC_SHRINK_K,

            "eq_n_state0": eq_mom[0]["n"],
            "eq_n_state1": eq_mom[1]["n"],
            "bd_n_state0": bd_mom[0]["n"],
            "bd_n_state1": bd_mom[1]["n"],

            "eq_weight_state0": eq_mom[0]["weight"],
            "eq_weight_state1": eq_mom[1]["weight"],
            "bd_weight_state0": bd_mom[0]["weight"],
            "bd_weight_state1": bd_mom[1]["weight"],

            "rho_source": rho_source,
            "rho_n_obs": rho_n,
            "rho_weight": rho_w,
            "rho_empirical": rho_emp,
        })

        prev_w_eq = w_eq
        prev_w_bd = w_bd

    return pd.DataFrame(rows)


def build_true_6040_benchmark(date_index):
    bench = (
        returns[["date", "equity_ret_h1", "bond_ret_h1"]]
        .copy()
        .dropna()
    )

    bench["date"] = pd.to_datetime(bench["date"])

    bench = bench[bench["date"].isin(date_index)].copy()
    bench = bench.sort_values("date").reset_index(drop=True)

    target_eq = 0.60
    target_bd = 0.40

    bench["ret_60_40_gross"] = (
        target_eq * bench["equity_ret_h1"]
        + target_bd * bench["bond_ret_h1"]
    )

    turnover_6040 = np.zeros(len(bench), dtype=float)

    if len(bench) > 0:
        turnover_6040[0] = abs(target_eq - 0.0) + abs(target_bd - 0.0)

    for i in range(1, len(bench)):
        prev_eq_ret = float(bench.loc[i - 1, "equity_ret_h1"])
        prev_bd_ret = float(bench.loc[i - 1, "bond_ret_h1"])

        portfolio_growth = (
            target_eq * (1.0 + prev_eq_ret)
            + target_bd * (1.0 + prev_bd_ret)
        )

        if np.isfinite(portfolio_growth) and abs(portfolio_growth) > EPS:
            w_eq_drift = target_eq * (1.0 + prev_eq_ret) / portfolio_growth
            w_bd_drift = target_bd * (1.0 + prev_bd_ret) / portfolio_growth
            turnover_6040[i] = abs(target_eq - w_eq_drift) + abs(target_bd - w_bd_drift)
        else:
            turnover_6040[i] = 0.0

    bench["turnover_60_40"] = turnover_6040

    bench["ret_60_40_net"] = (
        bench["ret_60_40_gross"]
        - TCOST_ONE_WAY * bench["turnover_60_40"]
    )

    bench["wealth_60_40"] = (
        MV_TC_STARTING_WEALTH
        * (1.0 + bench["ret_60_40_net"]).cumprod()
    )

    return bench


def summarize_allocator(allocation_results, benchmark):
    rows = []

    bench_summary = performance_summary(
        benchmark["ret_60_40_net"],
        "monthly_rebalanced_60_40",
    )

    bench_summary["scenario_id"] = allocation_results["scenario_id"].iloc[0] if len(allocation_results) else ""
    bench_summary["family"] = "60/40"
    bench_summary["feature_set"] = "benchmark"
    bench_summary["avg_w_eq"] = 0.60
    bench_summary["avg_w_bd"] = 0.40
    bench_summary["avg_w_cash"] = 0.00
    bench_summary["avg_turnover"] = float(benchmark["turnover_60_40"].mean()) if len(benchmark) else np.nan
    bench_summary["first_date"] = benchmark["date"].min() if len(benchmark) else pd.NaT
    bench_summary["last_date"] = benchmark["date"].max() if len(benchmark) else pd.NaT
    bench_summary["gamma"] = np.nan
    bench_summary["tau"] = np.nan
    bench_summary["shrink_k"] = MV_TC_SHRINK_K

    rows.append(bench_summary)

    for strategy_name, g in allocation_results.groupby("strategy_name"):
        s = performance_summary(g["strategy_net_ret"], strategy_name)

        s["scenario_id"] = str(g["scenario_id"].iloc[0])
        s["family"] = str(g["family"].iloc[0])
        s["feature_set"] = str(g["feature_set"].iloc[0])
        s["avg_w_eq"] = float(g["w_eq"].mean())
        s["avg_w_bd"] = float(g["w_bd"].mean())
        s["avg_w_cash"] = float(g["w_cash"].mean())
        s["avg_turnover"] = float(g["turnover"].mean())
        s["first_date"] = g["date"].min()
        s["last_date"] = g["date"].max()
        s["gamma"] = MV_TC_GAMMA
        s["tau"] = MV_TC_TAU
        s["shrink_k"] = MV_TC_SHRINK_K

        rows.append(s)

    return (
        pd.DataFrame(rows)
        .sort_values("sharpe", ascending=False)
        .reset_index(drop=True)
    )


def shrinkage_diagnostics(allocation_results):
    return (
        allocation_results
        .groupby(["scenario_id", "family", "feature_set"], as_index=False)
        .agg(
            n_months=("date", "count"),
            first_date=("date", "min"),
            last_date=("date", "max"),
            shrink_k=("shrink_k", "first"),
            min_eq_n_state0=("eq_n_state0", "min"),
            min_eq_n_state1=("eq_n_state1", "min"),
            min_bd_n_state0=("bd_n_state0", "min"),
            min_bd_n_state1=("bd_n_state1", "min"),
            avg_eq_weight_state0=("eq_weight_state0", "mean"),
            avg_eq_weight_state1=("eq_weight_state1", "mean"),
            avg_bd_weight_state0=("bd_weight_state0", "mean"),
            avg_bd_weight_state1=("bd_weight_state1", "mean"),
            avg_rho_weight=("rho_weight", "mean"),
            avg_turnover=("turnover", "mean"),
        )
        .sort_values(["scenario_id", "feature_set", "family"])
        .reset_index(drop=True)
    )


def run_immediate_allocator_for_family(
    scenario_id,
    family,
    family_wide_panel,
    historical_label_return_panel,
    historical_targets,
):
    panel = family_wide_panel.copy()
    panel["date"] = pd.to_datetime(panel["date"])

    panel = panel.merge(
        returns[["date", "equity_ret_h1", "bond_ret_h1"]],
        on="date",
        how="left",
    )

    first_prediction_date = panel["date"].dropna().min()

    initial_moments = initial_moments_from_pre_oos(
        historical_label_return_panel,
        first_prediction_date,
    )

    title(f"{scenario_id} | MV-TC ALLOCATION: {family}")
    print("Objective: w'mu - gamma*w'Sigma*w - tau*c*turnover")
    print("gamma:", MV_TC_GAMMA)
    print("tau:", MV_TC_TAU)
    print("transaction cost c:", TCOST_ONE_WAY)
    print("shrinkage k:", MV_TC_SHRINK_K)
    print("first prediction date:", first_prediction_date)
    print("initial moment sample:", initial_moments["sample_start"], "to", initial_moments["sample_end"])
    print("initial n_obs:", initial_moments["n_obs"])
    print("initial monthly mu_eq:", initial_moments["eq"]["mu"])
    print("initial monthly mu_bd:", initial_moments["bd"]["mu"])
    print("initial monthly sd_eq:", np.sqrt(initial_moments["eq"]["var"]))
    print("initial monthly sd_bd:", np.sqrt(initial_moments["bd"]["var"]))
    print("initial rho:", initial_moments["rho"])

    parts = []

    for feature_set in ALLOC_FEATURE_ORDER:
        print(f"Running allocator: {family}_{feature_set}")

        part = run_allocator_for_family_feature(
            panel=panel,
            scenario_id=scenario_id,
            family_name=family,
            feature_set=feature_set,
            historical_label_return_panel=historical_label_return_panel,
            historical_targets=historical_targets,
            initial_moments=initial_moments,
        )

        parts.append(part)

    allocation_results = (
        pd.concat(parts, axis=0, ignore_index=True)
        .sort_values(["feature_set", "family", "date"])
        .reset_index(drop=True)
    )

    if allocation_results.empty:
        raise RuntimeError(f"{scenario_id} {family}: no allocation results produced.")

    common_dates = (
        allocation_results
        .groupby(["family", "feature_set"])["date"]
        .apply(set)
    )

    common_date_set = set.intersection(*common_dates.tolist())
    common_date_index = sorted(common_date_set)

    allocation_results = (
        allocation_results[allocation_results["date"].isin(common_date_index)]
        .sort_values(["feature_set", "family", "date"])
        .reset_index(drop=True)
    )

    benchmark = build_true_6040_benchmark(common_date_index)

    allocation_results["wealth"] = (
        allocation_results
        .groupby("strategy_name")["strategy_net_ret"]
        .transform(lambda r: MV_TC_STARTING_WEALTH * (1.0 + r).cumprod())
    )

    summary = summarize_allocator(allocation_results, benchmark)
    diag = shrinkage_diagnostics(allocation_results)

    title(f"{scenario_id} | ALLOCATION SUMMARY: {family}")

    display_cols = [
        "strategy",
        "family",
        "feature_set",
        "n_months",
        "first_date",
        "last_date",
        "ann_return",
        "ann_vol",
        "sharpe",
        "max_drawdown",
        "avg_w_eq",
        "avg_w_bd",
        "avg_w_cash",
        "avg_turnover",
        "gamma",
        "tau",
        "shrink_k",
    ]

    display_cols = [c for c in display_cols if c in summary.columns]
    print(summary[display_cols].to_string(index=False))

    print("\nShrinkage diagnostics:")
    print(diag.to_string(index=False))

    plot_family_allocation(
        scenario_id=scenario_id,
        family=family,
        allocation_results=allocation_results,
        benchmark=benchmark,
        summary=summary,
        historical_label_return_panel=historical_label_return_panel,
        historical_targets=historical_targets,
    )

    return {
        "scenario_id": scenario_id,
        "family": family,
        "allocation_results": allocation_results,
        "benchmark": benchmark,
        "allocation_summary": summary,
        "shrinkage_diagnostics": diag,
        "initial_moments": initial_moments,
    }


# =============================================================================
# 7. Plotting
# =============================================================================

def contiguous_spans(mask_series):
    mask_series = mask_series.fillna(False).astype(bool)
    spans = []
    start = None
    prev_date = None

    for dt, val in zip(mask_series.index, mask_series.values):
        if val and start is None:
            start = dt

        if (not val) and start is not None:
            spans.append((start, prev_date))
            start = None

        prev_date = dt

    if start is not None:
        spans.append((start, prev_date))

    return spans


def state_background_panel(feature_set, date_index, historical_label_return_panel, historical_targets):
    eq_col = historical_targets[feature_set]["eq_y"]
    bd_col = historical_targets[feature_set]["bd_y"]

    bg = (
        historical_label_return_panel[["date", eq_col, bd_col]]
        .copy()
        .rename(columns={eq_col: "eq_good", bd_col: "bd_good"})
    )

    bg["date"] = pd.to_datetime(bg["date"])
    bg = bg[bg["date"].isin(date_index)].sort_values("date").reset_index(drop=True)

    bg["eq_good"] = bg["eq_good"].astype(float)
    bg["bd_good"] = bg["bd_good"].astype(float)

    bg["state_background"] = "missing"
    bg.loc[bg["eq_good"].eq(1.0) & bg["bd_good"].eq(1.0), "state_background"] = "both_good"
    bg.loc[bg["eq_good"].eq(1.0) & bg["bd_good"].eq(0.0), "state_background"] = "equity_good_only"
    bg.loc[bg["eq_good"].eq(0.0) & bg["bd_good"].eq(1.0), "state_background"] = "bond_good_only"
    bg.loc[bg["eq_good"].eq(0.0) & bg["bd_good"].eq(0.0), "state_background"] = "both_bad"

    return bg


def add_state_spans(ax, bg):
    shade_specs = [
        ("both_good", "equity good / bond good"),
        ("equity_good_only", "equity good / bond bad"),
        ("bond_good_only", "equity bad / bond good"),
        ("both_bad", "equity bad / bond bad"),
    ]

    handles = []

    for state, label in shade_specs:
        mask = bg["state_background"].eq(state)
        spans = contiguous_spans(pd.Series(mask.values, index=bg["date"]))

        for start, end in spans:
            ax.axvspan(
                start,
                end + pd.offsets.MonthEnd(1),
                color=STATE_COLORS[state],
                alpha=0.13,
                lw=0,
                zorder=0,
            )

        handles.append(
            Patch(
                facecolor=STATE_COLORS[state],
                edgecolor="none",
                alpha=0.22,
                label=label,
            )
        )

    return handles


def fmt_num(x):
    return "NA" if not np.isfinite(x) else f"{x:.3f}"


def build_stats_table(feature_set, family, allocation_summary):
    strategies = ["monthly_rebalanced_60_40", f"{allocation_summary['scenario_id'].iloc[0]}__{family}_{feature_set}"]

    rows = []

    for strat in strategies:
        r = allocation_summary.loc[allocation_summary["strategy"].eq(strat)]

        if r.empty:
            continue

        r = r.iloc[0]
        label = "60/40" if strat == "monthly_rebalanced_60_40" else str(r["family"])

        rows.append({
            "Strategy": label,
            "Ann.ret": fmt_num(float(r["ann_return"])),
            "Sharpe": fmt_num(float(r["sharpe"])),
            "MaxDD": fmt_num(float(r["max_drawdown"])),
            "Turn": fmt_num(float(r["avg_turnover"])),
            "EQ/FI/Cash": (
                f"{float(r['avg_w_eq']):.2f} / "
                f"{float(r['avg_w_bd']):.2f} / "
                f"{float(r['avg_w_cash']):.2f}"
            ),
        })

    return pd.DataFrame(rows).set_index("Strategy")


def add_stats_table(ax, feature_set, family, allocation_summary, bbox=(0.55, 0.03, 0.43, 0.18)):
    stats = build_stats_table(feature_set, family, allocation_summary)

    tbl = ax.table(
        cellText=stats.values,
        rowLabels=list(stats.index),
        colLabels=list(stats.columns),
        cellLoc="center",
        rowLoc="center",
        bbox=bbox,
    )

    tbl.auto_set_font_size(False)
    tbl.set_fontsize(7.8)

    for _, cell in tbl.get_celld().items():
        cell.set_alpha(0.94)

    return tbl


def plot_family_allocation(
    scenario_id,
    family,
    allocation_results,
    benchmark,
    summary,
    historical_label_return_panel,
    historical_targets,
):
    for feature_set in ALLOC_FEATURE_ORDER:
        fig, ax = plt.subplots(figsize=(16, 6.4))

        plot_df = benchmark[["date", "wealth_60_40"]].copy()

        ax.plot(
            plot_df["date"],
            plot_df["wealth_60_40"],
            color=LINE_COLORS["60/40"],
            linestyle="--",
            lw=2.2,
            label="60/40",
            zorder=3,
        )

        strat = f"{scenario_id}__{family}_{feature_set}"

        g = (
            allocation_results[allocation_results["strategy_name"].eq(strat)]
            .copy()
            .sort_values("date")
            .reset_index(drop=True)
        )

        if g.empty:
            plt.close(fig)
            continue

        ax.plot(
            g["date"],
            g["wealth"],
            color=LINE_COLORS[family],
            lw=2.1,
            label=family,
            zorder=4,
        )

        bg = state_background_panel(
            feature_set=feature_set,
            date_index=plot_df["date"],
            historical_label_return_panel=historical_label_return_panel,
            historical_targets=historical_targets,
        )

        state_handles = add_state_spans(ax, bg)

        line_handles, line_labels = ax.get_legend_handles_labels()

        ax.legend(
            line_handles + state_handles,
            line_labels + [h.get_label() for h in state_handles],
            loc="upper left",
            ncol=2,
            fontsize=8,
            frameon=True,
        )

        add_stats_table(ax, feature_set, family, summary)

        ax.set_title(
            f"{scenario_id} | MV-TC allocation — {family} — {feature_set} | "
            f"return - {MV_TC_GAMMA:g} risk - {MV_TC_TAU:g} transaction-cost penalty | "
            f"36-month shrinkage"
        )

        ax.set_xlabel("Date")
        ax.set_ylabel("Wealth index")
        ax.grid(True, alpha=0.25)

        plt.tight_layout()
        plt.show()


# =============================================================================
# 8. Main run
# =============================================================================

student_prediction_scenario_registry = {}
student_prediction_prediction_result_parts = []
student_prediction_selected_model_parts = []
student_prediction_prediction_long_parts = []
student_prediction_family_comparison_parts = []
student_prediction_family_wide_panels = {}
student_prediction_feature_importance_top_parts = []

student_prediction_allocation_result_parts = []
student_prediction_allocation_benchmarks = {}
student_prediction_allocation_summary_parts = []
student_prediction_shrinkage_diag_parts = []
student_prediction_initial_moments = {}

student_prediction_start_time = time.time()

title("Student Prediction and Allocation FULL RUN START")
print("Run tag:", STUDENT_PREDICTION_RUN_TAG)
print("Teacher versions:", RUN_TEACHER_MINOBS)
print("Student panels:", RUN_STUDENT_PANELS)
print("Feature transforms:", RUN_FEATURE_TRANSFORMS)
print("Model families:", RUN_MODEL_FAMILIES)
print("Model min train months:", MODEL_MIN_TRAIN_MONTHS)
print("Validation months:", MODEL_VALIDATION_MONTHS)
print("MV-TC gamma:", MV_TC_GAMMA)
print("MV-TC tau:", MV_TC_TAU)
print("MV-TC shrink k:", MV_TC_SHRINK_K)

for minobs in RUN_TEACHER_MINOBS:
    target_wide = majority_vote_target_wide_by_minobs[int(minobs)].copy()
    specs = majority_vote_specs_by_minobs[int(minobs)].copy()

    target_wide["date"] = pd.to_datetime(target_wide["date"])

    for student_name in RUN_STUDENT_PANELS:
        student_df = student_panel_by_name[student_name].copy()

        for transform_name in RUN_FEATURE_TRANSFORMS:
            scenario_id = f"teacher_minobs{minobs}__{student_name}__{transform_name}"

            title(f"SCENARIO START: {scenario_id}")

            prediction_base, feature_cols, target_cols, target_prob_cols, target_meta = build_prediction_base(
                student_df=student_df,
                target_wide=target_wide,
                specs=specs,
                transform_name=transform_name,
            )

            historical_label_return_panel, historical_targets = build_historical_label_return_panel(
                prediction_base=prediction_base,
                specs=specs,
            )

            student_prediction_scenario_registry[scenario_id] = {
                "teacher_minobs": int(minobs),
                "student_panel": student_name,
                "feature_transform": transform_name,
                "n_rows": int(len(prediction_base)),
                "date_min": prediction_base["date"].min(),
                "date_max": prediction_base["date"].max(),
                "n_features": int(len(feature_cols)),
                "target_cols": list(target_cols),
                "target_prob_cols": list(target_prob_cols),
            }

            print("Prediction base shape:", prediction_base.shape)
            print("Prediction date range:", prediction_base["date"].min(), "to", prediction_base["date"].max())
            print("Feature count:", len(feature_cols))
            print("Target columns:", target_cols)
            print("First 40 features:")
            print(feature_cols[:40])

            for family in RUN_MODEL_FAMILIES:
                if family == "XGB" and not XGBOOST_AVAILABLE:
                    raise ImportError("XGB requested but xgboost unavailable.")

                family_result = run_one_family_prediction(
                    scenario_id=scenario_id,
                    df=prediction_base,
                    feature_cols=feature_cols,
                    target_cols=target_cols,
                    target_meta=target_meta,
                    family=family,
                )

                family_prediction_long = family_result["prediction_long"]

                if family_prediction_long.empty:
                    raise RuntimeError(f"{scenario_id} {family}: empty prediction_long.")

                family_comparison = summarize_family_predictions(
                    prediction_long=family_prediction_long,
                    df=prediction_base,
                    target_meta=target_meta,
                )

                title(f"{scenario_id} | STUDENT FAMILY COMPARISON: {family}")

                display_cols = [
                    "target",
                    "family",
                    "asset",
                    "target_type",
                    "target_design",
                    "n_oos_months",
                    "first_oos",
                    "last_oos",
                    "positive_share",
                    "prob_mean",
                    "prob_std",
                    "oos_auc",
                    "oos_balanced_accuracy",
                    "oos_brier",
                    "oos_net_sharpe",
                    "oos_buyhold_sharpe",
                    "oos_sharpe_lift_vs_buyhold",
                    "oos_turnover",
                    "oos_signal_share",
                ]

                display_cols = [c for c in display_cols if c in family_comparison.columns]
                print(family_comparison[display_cols].to_string(index=False))

                family_wide_panel = build_family_wide_panel(
                    prediction_long=family_prediction_long,
                    target_cols=target_cols,
                )

                if family_wide_panel.empty:
                    raise RuntimeError(f"{scenario_id} {family}: empty family wide panel.")

                student_prediction_family_wide_panels[(scenario_id, family)] = family_wide_panel.copy()

                allocation_result = run_immediate_allocator_for_family(
                    scenario_id=scenario_id,
                    family=family,
                    family_wide_panel=family_wide_panel,
                    historical_label_return_panel=historical_label_return_panel,
                    historical_targets=historical_targets,
                )

                student_prediction_prediction_result_parts.append(family_result["result_table"])
                student_prediction_selected_model_parts.append(family_result["selected_model_table"])
                student_prediction_prediction_long_parts.append(family_result["prediction_long"])
                student_prediction_family_comparison_parts.append(family_comparison)

                if not family_result["feature_importance_top"].empty:
                    student_prediction_feature_importance_top_parts.append(family_result["feature_importance_top"])

                student_prediction_allocation_result_parts.append(allocation_result["allocation_results"])
                student_prediction_allocation_benchmarks[(scenario_id, family)] = allocation_result["benchmark"]
                student_prediction_allocation_summary_parts.append(allocation_result["allocation_summary"])
                student_prediction_shrinkage_diag_parts.append(allocation_result["shrinkage_diagnostics"])
                student_prediction_initial_moments[(scenario_id, family)] = allocation_result["initial_moments"]


# =============================================================================
# 9. Output objects
# =============================================================================

student_prediction_student_result_table = (
    pd.concat(student_prediction_prediction_result_parts, axis=0, ignore_index=True)
    .sort_values(["scenario_id", "family", "target", "test_date", "model_name"])
    .reset_index(drop=True)
)

student_prediction_student_selected_model_table = (
    pd.concat(student_prediction_selected_model_parts, axis=0, ignore_index=True)
    .sort_values(["scenario_id", "family", "target", "test_date"])
    .reset_index(drop=True)
)

student_prediction_student_oos_prediction_panel_long = (
    pd.concat(student_prediction_prediction_long_parts, axis=0, ignore_index=True)
    .sort_values(["scenario_id", "family", "target", "date"])
    .reset_index(drop=True)
)

student_prediction_student_family_comparison_table = (
    pd.concat(student_prediction_family_comparison_parts, axis=0, ignore_index=True)
    .sort_values(["scenario_id", "target", "family"])
    .reset_index(drop=True)
)

student_prediction_feature_importance_top = (
    pd.concat(student_prediction_feature_importance_top_parts, axis=0, ignore_index=True)
    .sort_values(["scenario_id", "family", "target", "rank"])
    .reset_index(drop=True)
    if student_prediction_feature_importance_top_parts else pd.DataFrame()
)

student_prediction_allocation_results_long = (
    pd.concat(student_prediction_allocation_result_parts, axis=0, ignore_index=True)
    .sort_values(["scenario_id", "family", "feature_set", "date"])
    .reset_index(drop=True)
)

student_prediction_allocation_strategy_summary = (
    pd.concat(student_prediction_allocation_summary_parts, axis=0, ignore_index=True)
    .sort_values(["scenario_id", "sharpe"], ascending=[True, False])
    .reset_index(drop=True)
)

student_prediction_shrinkage_diagnostics = (
    pd.concat(student_prediction_shrinkage_diag_parts, axis=0, ignore_index=True)
    .sort_values(["scenario_id", "feature_set", "family"])
    .reset_index(drop=True)
)

student_prediction_scenario_summary = pd.DataFrame([
    {
        "scenario_id": k,
        **v,
    }
    for k, v in student_prediction_scenario_registry.items()
]).sort_values("scenario_id").reset_index(drop=True)

student_result_table = student_prediction_student_result_table
student_selected_model_table = student_prediction_student_selected_model_table
student_oos_prediction_panel_long = student_prediction_student_oos_prediction_panel_long
student_family_comparison_table = student_prediction_student_family_comparison_table
student_oos_prediction_panels_by_scenario_family = student_prediction_family_wide_panels

mv_tc_allocation_results_long = student_prediction_allocation_results_long
mv_tc_allocation_strategy_summary = student_prediction_allocation_strategy_summary
mv_tc_shrinkage_diagnostics = student_prediction_shrinkage_diagnostics


# =============================================================================
# 10. Final report
# =============================================================================

title("Student Prediction and Allocation COMPLETE — STUDENT GRID AND MV-TC ALLOCATION")

print("Elapsed minutes:", round((time.time() - student_prediction_start_time) / 60.0, 2))

print("\nScenario summary:")
scenario_display_cols = [
    "scenario_id",
    "teacher_minobs",
    "student_panel",
    "feature_transform",
    "n_rows",
    "date_min",
    "date_max",
    "n_features",
]
print(student_prediction_scenario_summary[scenario_display_cols].to_string(index=False))

print("\nStudent result objects:")
print("student_prediction_student_result_table shape:", student_prediction_student_result_table.shape)
print("student_prediction_student_selected_model_table shape:", student_prediction_student_selected_model_table.shape)
print("student_prediction_student_oos_prediction_panel_long shape:", student_prediction_student_oos_prediction_panel_long.shape)
print("student_prediction_student_family_comparison_table shape:", student_prediction_student_family_comparison_table.shape)
print("student_prediction_feature_importance_top shape:", student_prediction_feature_importance_top.shape)

print("\nAllocation result objects:")
print("student_prediction_allocation_results_long shape:", student_prediction_allocation_results_long.shape)
print("student_prediction_allocation_strategy_summary shape:", student_prediction_allocation_strategy_summary.shape)
print("student_prediction_shrinkage_diagnostics shape:", student_prediction_shrinkage_diagnostics.shape)

print("\nTop allocation rows by scenario:")
alloc_display_cols = [
    "scenario_id",
    "strategy",
    "family",
    "feature_set",
    "n_months",
    "first_date",
    "last_date",
    "ann_return",
    "ann_vol",
    "sharpe",
    "max_drawdown",
    "avg_w_eq",
    "avg_w_bd",
    "avg_w_cash",
    "avg_turnover",
]
alloc_display_cols = [c for c in alloc_display_cols if c in student_prediction_allocation_strategy_summary.columns]

top_alloc = (
    student_prediction_allocation_strategy_summary
    .sort_values(["scenario_id", "sharpe"], ascending=[True, False])
    .groupby("scenario_id", as_index=False)
    .head(8)
    .reset_index(drop=True)
)

print(top_alloc[alloc_display_cols].to_string(index=False))

print("\nBest strategy per scenario:")
best_alloc = (
    student_prediction_allocation_strategy_summary
    .sort_values(["scenario_id", "sharpe"], ascending=[True, False])
    .groupby("scenario_id", as_index=False)
    .head(1)
    .reset_index(drop=True)
)
print(best_alloc[alloc_display_cols].to_string(index=False))

print("\nMain aliases:")
print("student_result_table")
print("student_selected_model_table")
print("student_oos_prediction_panel_long")
print("student_family_comparison_table")
print("student_oos_prediction_panels_by_scenario_family")
print("mv_tc_allocation_results_long")
print("mv_tc_allocation_strategy_summary")
print("mv_tc_shrinkage_diagnostics")